In [ ]:
#   K_A = k_B * T * <A> / Var(A)

# Where:
#   <A>    = mean total projected area = <Box_X * Box_Y>
#   Var(A) = variance of A(t) from the time series
#   NO N_leaflet in this formula

# =============================================================================
import numpy as np
# =============================================================================
# PARAMETERS - Modify these for your system
# =============================================================================
xvg_file = "box_DLPA_95.xvg"    # <-- Change this to your file path
N_LEAFLET = 144            # Number of lipids per leaflet (for APL only)
T = 298.15                 # Temperature (K)
k_B = 1.38065e-23          # Boltzmann constant (J/K)
# =============================================================================
# Read the .xvg file
# =============================================================================
time_list, box_x_list, box_y_list = [], [], []
with open(xvg_file, 'r') as f:
    for line in f:

        if line.startswith('#') or line.startswith('@') or line.strip() == '':

            continue

        vals = line.split()

        if len(vals) >= 3:

            time_list.append(float(vals[0]))

            box_x_list.append(float(vals[1]))

            box_y_list.append(float(vals[2]))
 
time = np.array(time_list)

box_x = np.array(box_x_list)

box_y = np.array(box_y_list)
 
print(f"File:    {xvg_file}")

print(f"Frames:  {len(time):,}")

print(f"Time:    {time[0]:.0f} to {time[-1]:.0f} ps ({(time[-1]-time[0])/1000:.1f} ns)")
 
# =============================================================================

# Compute area time series

# =============================================================================
 
area = box_x * box_y           # nm^2 at every frame

mean_A = np.mean(area)         # nm^2

var_A = np.var(area, ddof=1)   # nm^4 (sample variance)

std_A = np.std(area, ddof=1)   # nm^2
 
mean_X = np.mean(box_x)

rmsd_X = np.std(box_x, ddof=1)
 
print(f"\nBox-X:   mean = {mean_X:.5f} nm,  RMSD = {rmsd_X:.6f} nm")

print(f"Area:    mean = {mean_A:.2f} nm²,  std = {std_A:.4f} nm²,  Var = {var_A:.4f} nm⁴")
 
# =============================================================================

# K_A Calculation (correct formula: no N_leaflet)

# =============================================================================
 
K_A = (k_B * T * mean_A * 1e-18) / (var_A * 1e-36)   # N/m

K_A_mNm = K_A * 1000                                   # mN/m
 
# =============================================================================
# APL (uses N_leaflet)
# =============================================================================
APL = mean_A / N_LEAFLET * 100   # Angstrom^2
# =============================================================================
# Block averaging for error estimation
# =============================================================================
 
n_blocks = 5

block_size = len(area) // n_blocks

ka_blocks = []
 
for i in range(n_blocks):

    block = area[i * block_size : (i + 1) * block_size]

    bm = np.mean(block) * 1e-18       # m^2

    bv = np.var(block, ddof=1) * 1e-36  # m^4

    if bv > 0:

        ka_b = (k_B * T * bm) / bv * 1000   # mN/m

        ka_blocks.append(ka_b)
 
ka_blocks = np.array(ka_blocks)
ka_error = np.std(ka_blocks, ddof=1) / np.sqrt(len(ka_blocks))
# =============================================================================
# Print Results
# =============================================================================
 
print(f"\n{'='*65}")
print(f"  RESULTS")
print(f"{'='*65}")
print(f"  APL:             {APL:.1f} Å²")
print(f"  Mean Area:       {mean_A:.2f} nm²")
print(f"  Var(A):          {var_A:.4f} nm⁴")
print(f"  std(A):          {std_A:.4f} nm²")
print(f"  Relative fluct:  {std_A/mean_A*100:.2f}%")
print(f"")
print(f"  K_A = {K_A_mNm:.1f} ± {ka_error:.1f} mN/m")
print(f"")
print(f"  Block K_A values: {', '.join(f'{k:.1f}' for k in ka_blocks)}")
print(f"{'='*65}")
print(f"")
print(f"  Formula used: K_A = k_B * T * <A> / Var(A)")
print(f"  Reference: Feller & Pastor, JCP (1999)")
print(f"{'='*65}")

Radial Distribution Function

In [ ]:
# %% [markdown]
# # RDF - DOPC/DOPA composition series  (VS Code interactive script)
#
# Run cell-by-cell with the "Run Cell" links above each `# %%` (VS Code's Python
# Interactive window), or run the whole file. Same analysis as the notebook.
#
# Trajectory to use: a **whole + centered** xtc. Do NOT use the nojump xtc here -
# nojump is only for diffusion.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the VS Code terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
HEAD    = "P"
RMAX_NM = 5.0          # x-axis range (nm)
BIN_NM  = 0.02         # bin width (nm)
NBLOCKS = 5            # time blocks for the error band
STEP    = 1            # frame stride

# label -> (topology, trajectory[whole+centered], DOPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#2ca02c",   # green
    "72/28": "#1f77b4",   # blue
    "80/20": "#ff7f0e",   # orange
    "87/13": "#9467bd",   # purple
    "95/5" : "#d62728",   # red
}

PAIRS = [("DOPC", "DOPC"), ("DOPA", "DOPA"), ("DOPC", "DOPA")]
YLIM  = {("DOPC", "DOPC"): (0.5, 1.5),
         ("DOPA", "DOPA"): (0.5, 1.5),
         ("DOPC", "DOPA"): (0.75, 1.25)}
os.makedirs("results", exist_ok=True)

# %%  RDF core (per-leaflet, block-averaged)
def _pdist2d(a_xy, b_xy, Lx, Ly, same):
    """Minimum-image lateral distances (A) between two point sets."""
    if len(a_xy) == 0 or len(b_xy) == 0:
        return np.empty(0)
    dx = a_xy[:, None, 0] - b_xy[None, :, 0]
    dy = a_xy[:, None, 1] - b_xy[None, :, 1]
    dx -= Lx * np.round(dx / Lx)
    dy -= Ly * np.round(dy / Ly)
    r = np.sqrt(dx * dx + dy * dy)
    return r[np.triu_indices(r.shape[0], 1)] if same else r.ravel()


def rdf2d_blocks(u, resA, resB, head=HEAD, rmax_nm=RMAX_NM, bin_nm=BIN_NM,
                 nblocks=NBLOCKS, step=STEP):
    same = (resA == resB)
    rmax = rmax_nm * 10.0
    nbins = int(round(rmax / (bin_nm * 10.0)))
    edges = np.linspace(0, rmax, nbins + 1)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    shell = np.pi * (edges[1:] ** 2 - edges[:-1] ** 2)
    selA = u.select_atoms(f"name {head} and resname {resA}")
    selB = u.select_atoms(f"name {head} and resname {resB}")
    allP = u.select_atoms(f"name {head}")
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    gblocks = []
    for blk in blocks:
        hist = np.zeros(nbins)
        norm = 0.0
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            area = Lx * Ly
            zmid = allP.positions[:, 2].mean()
            Apos, Bpos = selA.positions, selB.positions
            for sign in (True, False):                 # upper then lower leaflet
                a = Apos[(Apos[:, 2] >= zmid) == sign][:, :2]
                b = Bpos[(Bpos[:, 2] >= zmid) == sign][:, :2]
                hist += np.histogram(_pdist2d(a, b, Lx, Ly, same), bins=edges)[0]
                Na, Nb = len(a), len(b)
                norm += 0.5 * Na * ((Nb - 1) / area) if same else Na * (Nb / area)
        if norm > 0:
            gblocks.append(hist / (norm * shell))
    G = np.array(gblocks)
    mean = G.mean(axis=0)
    std = G.std(axis=0) if len(G) > 1 else np.zeros_like(mean)
    return ctr / 10.0, mean, std                        # r in nm

# %%  Compute for every composition and pair
results = {(a, b): {} for a, b in PAIRS}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for a, b in PAIRS:
        r, m, s = rdf2d_blocks(u, a, b)
        results[(a, b)][label] = (r, m, s)
        np.savetxt(f"results/rdf_{a}-{b}_{label.replace('/', '-')}.dat",
                   np.column_stack([r, m, s]), header="r_nm  g(r)  std")
print("done")

# %%  Plot - one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DOPA -> low
# fall back to a qualitative colormap for any label missing from COLORS
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, len(PAIRS), figsize=(4.2 * len(PAIRS), 3.6))
if len(PAIRS) == 1:
    axes = [axes]
for ax, (a, b) in zip(axes, PAIRS):
    for L in labels_sorted:
        if L not in results[(a, b)]:
            continue
        r, m, s = results[(a, b)][L]
        ax.fill_between(r, m - s, m + s, color=colors[L], alpha=0.18, lw=0)
        ax.plot(r, m, color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DOPA)")
    ax.axhline(1.0, color="0.6", lw=0.6, ls="--")
    ax.set_xlim(0, RMAX_NM)
    ax.set_ylim(*YLIM.get((a, b), (0.5, 1.5)))
    ax.set_xlabel("Distance (nm)")
    ax.text(0.95, 0.06, f"{a}-{b}", transform=ax.transAxes, style="italic",
            ha="right", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("RDF")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=n if (n := len(labels_sorted)) else 1,
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.12))
fig.suptitle("DOPC/DOPA lateral RDF (P-P) vs composition", y=1.18)
plt.tight_layout()
plt.savefig("results/rdf_DOPC_DOPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## Optional: plot straight from `gmx rdf` .xvg files

# %%  read gmx xvg (optional)
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # r(nm), g(r)

# gmx_files = {"65/35": "rdf_DOPC-DOPA_65.xvg"}
# plt.figure(figsize=(4.5, 3.6))
# for L in labels_sorted:
#     if L in gmx_files:
#         r, g = read_xvg_xy(gmx_files[L]); plt.plot(r, g, color=colors[L], label=L)
# plt.axhline(1, ls="--", color="0.6", lw=.6); plt.xlim(0, RMAX_NM); plt.ylim(0.75, 1.25)
# plt.xlabel("Distance (nm)"); plt.ylabel("RDF"); plt.legend(); plt.show()

Water Depth

In [ ]:
# %% [markdown]
# # Lipid tail splay - DOPC/DOPA composition series  (VS Code interactive script)
#
# Intra-lipid splay = angle between a lipid's two acyl-chain vectors
# (sn-1: first->last C3x carbon; sn-2: first->last C2x carbon). We build the
# probability distribution P(theta) per composition, per lipid type, with an
# error band from block averaging - same style as the RDF figure.
#
# Trajectory to use: the **full-atom, whole + centered** xtc (e.g.
# `Prod_merged_center_*.xtc`). NOT the P-only `heads` trajectory (splay needs the
# chain carbons) and NOT the nojump one (that's diffusion only).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the VS Code terminal
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DOPA"]    # the two lipids -> one panel each
BINW_DEG = 2.0                 # splay-angle bin width (degrees)
ANG_MAX  = 180.0               # angle axis range
NBLOCKS  = 5                   # time blocks for the error band
STEP     = 1                   # frame stride

# label -> (topology, full-atom trajectory[whole+centered], DOPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#2ca02c",   # green
    "72/28": "#1f77b4",   # blue
    "80/20": "#ff7f0e",   # orange
    "87/13": "#9467bd",   # purple
    "95/5" : "#d62728",   # red
}
os.makedirs("results", exist_ok=True)

# %%  Splay core (block-averaged distribution)
def _carbons(res, prefix):
    """Acyl carbons of a chain ('C3'=sn-1, 'C2'=sn-2), skipping carbonyl idx 1."""
    pat = re.compile(r"^" + prefix + r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in res.atoms
            if (m := pat.match(at.name)) and int(m.group(1)) >= 2]
    return sorted(hits, key=lambda t: t[0])


def _splay_indices(u, resname):
    """(N,4) atom indices: sn1_first, sn1_last, sn2_first, sn2_last per lipid."""
    idx = []
    for res in u.select_atoms(f"resname {resname}").residues:
        c3 = _carbons(res, "C3")      # sn-1
        c2 = _carbons(res, "C2")      # sn-2
        if len(c3) >= 2 and len(c2) >= 2:
            idx.append((c3[0][1].index, c3[-1][1].index,
                        c2[0][1].index, c2[-1][1].index))
    return np.array(idx, dtype=int)


def splay_blocks(u, resname, binw=BINW_DEG, angmax=ANG_MAX,
                 nblocks=NBLOCKS, step=STEP):
    idx = _splay_indices(u, resname)
    edges = np.arange(0.0, angmax + binw, binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    if len(idx) == 0:
        return ctr, np.zeros_like(ctr), np.zeros_like(ctr), np.nan
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    Hs, all_ang = [], []
    for blk in blocks:
        ang = []
        for fi in blk:
            u.trajectory[int(fi)]
            pos = u.atoms.positions
            v1 = pos[idx[:, 1]] - pos[idx[:, 0]]      # sn-1 vector
            v2 = pos[idx[:, 3]] - pos[idx[:, 2]]      # sn-2 vector
            n1 = np.linalg.norm(v1, axis=1)
            n2 = np.linalg.norm(v2, axis=1)
            good = (n1 > 0) & (n2 > 0)
            cos = np.einsum("ij,ij->i", v1, v2) / (n1 * n2)
            ang.append(np.degrees(np.arccos(np.clip(cos[good], -1, 1))))
        ang = np.concatenate(ang)
        all_ang.append(ang)
        Hs.append(np.histogram(ang, bins=edges, density=True)[0])
    H = np.array(Hs)
    mean = H.mean(axis=0)
    std = H.std(axis=0) if len(H) > 1 else np.zeros_like(mean)
    mean_angle = np.concatenate(all_ang).mean()
    return ctr, mean, std, mean_angle

# %%  Compute for every composition and lipid
results = {lip: {} for lip in LIPIDS}   # lip -> {label: (theta, P, Pstd)}
means = {}                              # (lip,label) -> mean splay angle
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for lip in LIPIDS:
        theta, P, Pstd, mang = splay_blocks(u, lip)
        results[lip][label] = (theta, P, Pstd)
        means[(lip, label)] = mang
        np.savetxt(f"results/splay_{lip}_{label.replace('/', '-')}.dat",
                   np.column_stack([theta, P, Pstd]), header="angle_deg  P  std")

print("\nMean splay angle (deg):")
print(f"{'comp':<8}" + "".join(f"{lip:>10}" for lip in LIPIDS))
for label in SYSTEMS:
    row = "".join(f"{means.get((lip,label), float('nan')):10.1f}" for lip in LIPIDS)
    print(f"{label:<8}{row}")

# %%  Plot - one panel per lipid, one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DOPA -> low
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, len(LIPIDS), figsize=(4.6 * len(LIPIDS), 3.8),
                         sharex=True)
if len(LIPIDS) == 1:
    axes = [axes]
for ax, lip in zip(axes, LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        theta, P, Pstd = results[lip][L]
        ax.fill_between(theta, P - Pstd, P + Pstd, color=colors[L], alpha=0.18, lw=0)
        ax.plot(theta, P, color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DOPA)")
    ax.set_xlim(0, ANG_MAX)
    ax.set_xlabel("Tail splay angle (deg)")
    ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="right", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("P(θ)")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.10))
fig.suptitle("DOPC/DOPA intra-lipid tail splay vs composition", y=1.16)
plt.tight_layout()
plt.savefig("results/splay_DOPC_DOPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route (equivalent) - `gmx gangle`
#
# The intra-lipid splay is the angle between the sn-1 and sn-2 chain vectors.
# `gmx gangle -g1 vector -g2 vector` pairs vector *i* of group1 with vector *i*
# of group2. Each vector is a pair of atoms (chain first & last carbon), so the
# selections below give exactly one vector per lipid.
#
# DOPC/DOPA are 18:1, so the terminal carbons are C318 (sn-1) and C218 (sn-2).
# (For DLPG/DLPA use C312 / C212.)
#
# ```bash
# # one composition (repeat for 65,72,80,87,95), one lipid at a time:
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc
#
# # DOPC splay
# gmx gangle -s $TPR -f $XTC -g1 vector -g2 vector \
#     -group1 'name C32 C318 and resname DOPC' \
#     -group2 'name C22 C218 and resname DOPC' \
#     -oh splay_DOPC_65.xvg -binw 2
#
# # DOPA splay
# gmx gangle -s $TPR -f $XTC -g1 vector -g2 vector \
#     -group1 'name C32 C318 and resname DOPA' \
#     -group2 'name C22 C218 and resname DOPA' \
#     -oh splay_DOPA_65.xvg -binw 2
# ```
#
# `-oh` writes the normalized angle histogram P(theta). For the error band, run on
# time blocks (`-b/-e`) and take mean/std across the .xvg files, or use the Python
# block averaging above.

# %%  (Optional) plot straight from gmx gangle -oh .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # angle(deg), P

# gmx_files = {"DOPC": {"65/35": "splay_DOPC_65.xvg"}, "DOPA": {"65/35": "splay_DOPA_65.xvg"}}
# fig, axes = plt.subplots(1, len(LIPIDS), figsize=(4.6*len(LIPIDS), 3.8))
# for ax, lip in zip(np.atleast_1d(axes), LIPIDS):
#     for L in labels_sorted:
#         p = gmx_files.get(lip, {}).get(L)
#         if p and os.path.exists(p):
#             x, y = read_xvg_xy(p); ax.plot(x, y, color=colors[L], lw=1.8, label=L)
#     ax.set_xlabel("Tail splay angle (deg)"); ax.set_xlim(0, ANG_MAX)
#     ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic", ha="right", va="top")
# plt.legend(); plt.tight_layout(); plt.show()

Water Depth

In [ ]:
# %% [markdown]
# # Water depth / penetration - DOPC/DOPA composition series
#
# Symmetrised water density profile across the bilayer normal: water density
# (relative to bulk) vs z from the membrane centre, overlaid for the five
# compositions with a block-averaged error band. Deeper penetration = the curve
# stays high closer to z = 0.
#
# Also prints a table per composition: half-density depth (|z| where water falls
# to 0.5 of bulk), mean phosphate position (headgroup plane), and mean # waters in
# the hydrophobic core (|z| < core).
#
# Trajectory to use: the full **whole + centered** xtc that still contains water
# (`Prod_merged_center_*.xtc`). Not the water-stripped mem/heads trajectories, and
# not nojump (diffusion only). Centering matters here (profile is from the centre).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPID_SEL = "resname DOPC DOPA"          # membrane (defines the centre)
WATER_SEL = "resname TIP3 SOL and (name OH2 or name OW or name O)"  # water oxygen
HEAD      = "P"
ZMAX_NM   = 4.0     # profile half-range from centre (nm)
BINW_NM   = 0.05    # bin width (nm)
CORE_NM   = 1.0     # |z| < CORE counts as hydrophobic-core water
NBLOCKS   = 5       # time blocks for the error band
STEP      = 1

# label -> (topology, full trajectory[whole+centered, WITH water], DOPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#2ca02c",   # green
    "72/28": "#1f77b4",   # blue
    "80/20": "#ff7f0e",   # orange
    "87/13": "#9467bd",   # purple
    "95/5" : "#d62728",   # red
}
os.makedirs("results", exist_ok=True)

# %%  Water-profile core (block-averaged, symmetrised)
def water_profile_blocks(u, lipid_sel=LIPID_SEL, water_sel=WATER_SEL, head=HEAD,
                         zmax_nm=ZMAX_NM, binw_nm=BINW_NM, core_nm=CORE_NM,
                         nblocks=NBLOCKS, step=STEP):
    lip = u.select_atoms(lipid_sel)
    wat = u.select_atoms(water_sel)
    P = u.select_atoms(f"name {head}")
    zmax, binw, core = zmax_nm * 10, binw_nm * 10, core_nm * 10
    edges = np.arange(-zmax, zmax + binw, binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    nb_bins = len(ctr)
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]

    prof_blocks = []
    pzs, core_counts = [], []
    for blk in blocks:
        hw = np.zeros(nb_bins)
        areas = []
        nf = 0
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            zc = lip.center_of_mass()[2]
            zw = wat.positions[:, 2] - zc
            hw += np.histogram(zw, bins=edges)[0]
            areas.append(Lx * Ly)
            nf += 1
            pzs.append(np.abs(P.positions[:, 2] - zc).mean())
            core_counts.append(int(np.sum(np.abs(zw) < core)))
        vol = np.mean(areas) * binw                 # A^3 per bin
        dens = hw / nf / vol                          # atoms / A^3
        dens = 0.5 * (dens + dens[::-1])              # symmetrise about centre
        bulk = dens[np.abs(ctr) > (zmax - 10)].mean() # outer 1 nm = bulk
        prof_blocks.append(dens / bulk if bulk > 0 else dens)

    G = np.array(prof_blocks)
    mean = G.mean(axis=0)
    std = G.std(axis=0) if len(G) > 1 else np.zeros_like(mean)

    # half-density depth: |z| where the mean profile drops to 0.5 (going inward)
    zpos = ctr[ctr >= 0]
    mpos = mean[ctr >= 0]
    order = np.argsort(zpos)
    zpos, mpos = zpos[order], mpos[order]
    z_half = np.nan
    for i in range(len(zpos) - 1, 0, -1):
        if mpos[i] < 0.5 <= mpos[i - 1]:
            z_half = np.interp(0.5, [mpos[i], mpos[i - 1]], [zpos[i], zpos[i - 1]])
            break
    return (ctr / 10.0, mean, std, np.mean(pzs) / 10.0,
            (z_half / 10.0 if z_half == z_half else np.nan), np.mean(core_counts))

# %%  Compute for every composition
results = {}   # label -> (z, dens, std)
summary = {}   # label -> (phosphate_nm, half_depth_nm, core_waters)
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    z, d, s, pz, zhalf, ncore = water_profile_blocks(u)
    results[label] = (z, d, s)
    summary[label] = (pz, zhalf, ncore)
    np.savetxt(f"results/water_{label.replace('/', '-')}.dat",
               np.column_stack([z, d, s]), header="z_nm  water_rel_density  std")

print(f"\n{'comp':<8}{'P plane(nm)':>13}{'half-depth(nm)':>16}{'core waters':>13}")
for label in SYSTEMS:
    if label in summary:
        pz, zh, nc = summary[label]
        print(f"{label:<8}{pz:13.2f}{zh:16.2f}{nc:13.1f}")

# %%  Plot - water penetration, one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, ax = plt.subplots(figsize=(6.2, 4.2))
for L in labels_sorted:
    if L not in results:
        continue
    z, d, s = results[L]
    ax.fill_between(z, d - s, d + s, color=colors[L], alpha=0.18, lw=0)
    ax.plot(z, d, color=colors[L], lw=1.8, label=f"{L}  ({SYSTEMS[L][2]}% DOPA)")
# mark the mean phosphate plane
if summary:
    pmean = np.nanmean([summary[L][0] for L in summary])
    for xline in (pmean, -pmean):
        ax.axvline(xline, color="0.5", lw=0.8, ls=":")
    ax.text(pmean, 1.02, "P", color="0.4", ha="center", va="bottom", fontsize=10)
ax.axhline(0.5, color="0.8", lw=0.6, ls="--")
ax.set_xlim(-ZMAX_NM, ZMAX_NM)
ax.set_ylim(0, 1.15)
ax.set_xlabel("z from bilayer centre (nm)")
ax.set_ylabel("Water density (rel. to bulk)")
ax.text(0.03, 0.92, "Water", transform=ax.transAxes, style="italic",
        ha="left", va="top", fontsize=13, fontweight="bold")
ax.tick_params(direction="in", top=True, right=True)
ax.legend(loc="upper center", ncol=3, frameon=True, fontsize=8,
          bbox_to_anchor=(0.5, 1.18))
fig.suptitle("DOPC/DOPA water penetration vs composition", y=1.06)
plt.tight_layout()
plt.savefig("results/water_depth_DOPC_DOPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route (equivalent) - `gmx density`
#
# ```bash
# # one composition (repeat for 65,72,80,87,95):
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc ; NDX=index.ndx
#
# # centering group = Membrane , output group = Water
# printf "Membrane\nWater\n" | gmx density -s $TPR -f $XTC -n $NDX \
#     -d Z -sl 400 -center -symm -dens number -o water_density_65.xvg
#
# # phosphate plane for reference (output group = the P atoms):
# printf "Membrane\nPhosphate\n" | gmx density -s $TPR -f $XTC -n $NDX \
#     -d Z -sl 400 -center -symm -dens number -o phos_density_65.xvg
# ```
#
# `-center -symm` centres on the membrane and symmetrises; `-dens number` gives a
# number density. For error bands, run on time blocks (`-b/-e`) and take mean/std
# across the .xvg files, or use the Python block averaging above.

# %%  (Optional) plot straight from gmx density .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # z(nm), density

# gmx_files = {"65/35": "water_density_65.xvg"}
# plt.figure(figsize=(6.2, 4.2))
# for L in labels_sorted:
#     if L in gmx_files and os.path.exists(gmx_files[L]):
#         z, dw = read_xvg_xy(gmx_files[L])
#         plt.plot(z, dw / np.nanmax(dw), color=colors[L], lw=1.8, label=L)
# plt.xlabel("z (nm)"); plt.ylabel("water density (norm.)"); plt.legend(); plt.show()

Lipid Order Tail Parameter

In [ ]:
# %% [markdown]
# # Lipid tail order parameter (-S_CD) - DOPC/DOPA composition series
#
# Deuterium order parameter per acyl-chain carbon:
#   S_CD = < (3 cos^2(theta_CH) - 1) / 2 >    (theta_CH = C-H vs bilayer normal z)
# reported as -S_CD (the usual positive quantity). For each CH2 the two C-H
# values are averaged; C-H bonds are found automatically from the topology, so
# the cis double bond at C9-C10 in the oleoyl chains shows its dip on its own.
#
# One panel per (lipid x chain): DOPC/DOPA x sn-1/sn-2, each overlaying the five
# compositions with a block-averaged error band - same style as the RDF/splay
# figures.
#
# Trajectory to use: the full-atom **whole (+centered)** xtc WITH hydrogens
# (`Prod_merged_center_*.xtc`), plus the `.tpr` (needed for C-H bonds). Not the
# P-only `heads` trajectory, and not nojump (that is diffusion only).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import re
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS  = ["DOPC", "DOPA"]              # one row of panels each
CHAINS  = [("C3", "sn-1"), ("C2", "sn-2")]   # CHARMM36: sn-1=C3x, sn-2=C2x
NBLOCKS = 5                            # time blocks for the error band
STEP    = 1                            # frame stride

# label -> (topology[.tpr], full-atom trajectory[whole+centered], DOPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#2ca02c",   # green
    "72/28": "#1f77b4",   # blue
    "80/20": "#ff7f0e",   # orange
    "87/13": "#9467bd",   # purple
    "95/5" : "#d62728",   # red
}
os.makedirs("results", exist_ok=True)

# %%  Order-parameter core (block-averaged, auto C-H detection)
def _carbons(res, prefix):
    # acyl carbons of a chain ('C3'=sn-1, 'C2'=sn-2), skipping carbonyl idx 1
    pat = re.compile(r"^" + prefix + r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in res.atoms
            if (m := pat.match(at.name)) and int(m.group(1)) >= 2]
    return sorted(hits, key=lambda t: t[0])

def _build_pairs(u, resname, prefix, has_bonds):
    # returns (sorted carbon numbers, {k: (C_index_array, H_index_array)})
    C, H = defaultdict(list), defaultdict(list)
    for res in u.select_atoms(f"resname {resname}").residues:
        for k, catom in _carbons(res, prefix):
            if has_bonds:
                hs = [a for a in catom.bonded_atoms if a.name.startswith("H")]
            else:
                hs = list(u.select_atoms("name H* and around 1.3 group c",
                                         c=u.atoms[[catom.index]]))
            for h in hs:
                C[k].append(catom.index); H[k].append(h.index)
    ks = sorted(C)
    return ks, {k: (np.array(C[k]), np.array(H[k])) for k in ks}

def scd_blocks(u, resname, prefix, has_bonds, nblocks=NBLOCKS, step=STEP):
    ks, pairs = _build_pairs(u, resname, prefix, has_bonds)
    if not ks:
        return np.array([]), np.array([]), np.array([])
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    block_means = []
    for blk in blocks:
        acc = {k: [] for k in ks}
        for fi in blk:
            u.trajectory[int(fi)]
            pos = u.atoms.positions
            for k in ks:
                Ci, Hi = pairs[k]
                v = pos[Hi] - pos[Ci]
                n = np.linalg.norm(v, axis=1)
                cosz = v[:, 2] / n
                acc[k].append(np.mean(0.5 * (3 * cosz ** 2 - 1)))
        block_means.append([-np.mean(acc[k]) for k in ks])   # -S_CD
    B = np.array(block_means)
    return np.array(ks), B.mean(axis=0), (B.std(axis=0) if len(B) > 1 else np.zeros(len(ks)))

# %%  Compute for every composition, lipid and chain
results = {(lip, prefix): {} for lip in LIPIDS for prefix, _ in CHAINS}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    has_bonds = len(getattr(u, "bonds", [])) > 0
    print(f"{label}: {len(u.trajectory)} frames, bonds={has_bonds}")
    for lip in LIPIDS:
        for prefix, tag in CHAINS:
            ks, scd, err = scd_blocks(u, lip, prefix, has_bonds)
            if ks.size == 0:
                continue
            results[(lip, prefix)][label] = (ks, scd, err)
            np.savetxt(f"results/scd_{lip}_{tag}_{label.replace('/', '-')}.dat",
                       np.column_stack([ks, scd, err]), header="carbon  -SCD  std")
print("done")

# %%  Plot - 2x2 grid (lipid x chain), one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(len(LIPIDS), len(CHAINS),
                         figsize=(4.4 * len(CHAINS), 3.4 * len(LIPIDS)),
                         sharex=True, sharey=True)
axes = np.atleast_2d(axes)
for r, lip in enumerate(LIPIDS):
    for c, (prefix, tag) in enumerate(CHAINS):
        ax = axes[r, c]
        for L in labels_sorted:
            if L not in results[(lip, prefix)]:
                continue
            ks, scd, err = results[(lip, prefix)][L]
            ax.fill_between(ks, scd - err, scd + err, color=colors[L], alpha=0.18, lw=0)
            ax.plot(ks, scd, color=colors[L], lw=1.8, marker="o", ms=3,
                    label=f"{L}  ({SYSTEMS[L][2]}% DOPA)")
        ax.text(0.95, 0.92, f"{lip} {tag}", transform=ax.transAxes, style="italic",
                ha="right", va="top", fontsize=12, fontweight="bold")
        #ax.grid(alpha=0.3)
        ax.tick_params(direction="in", top=True, right=True)
        if r == len(LIPIDS) - 1:
            ax.set_xlabel("Carbon number")
        if c == 0:
            ax.set_ylabel(r"$-S_{CD}$")
handles, labs = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.05))
fig.suptitle("DOPC/DOPA acyl-chain order parameters vs composition", y=1.10)
plt.tight_layout()
plt.savefig("results/order_parameter_DOPC_DOPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route - `gmx order`
#
# For all-atom CHARMM36 the rigorous S_CD comes from the explicit C-H vectors
# (the Python above). `gmx order` was designed for united-atom chains and works
# on the carbon skeleton, so it gives the closely-related carbon order parameter
# rather than the exact explicit-H S_CD. Use it as a cross-check.
#
# It needs an index file listing the chain carbons **in order** (one group per
# chain). Build it per lipid with `gmx select`, then run `gmx order`:
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc
#
# # sn-1 carbons of DOPC, in order (C32..C318):
# gmx select -s $TPR -on sn1_DOPC.ndx \
#     -select 'resname DOPC and name C32 C33 C34 C35 C36 C37 C38 C39 C310 \
#              C311 C312 C313 C314 C315 C316 C317 C318'
#
# gmx order -s $TPR -f $XTC -n sn1_DOPC.ndx -d z \
#     -od deuter_DOPC_sn1_65.xvg -os order_DOPC_sn1_65.xvg
# ```
#
# Repeat with `name C22..C218` for sn-2, and for DOPA. `-od` writes the
# deuterium order parameter vs carbon. For error bands, run on time blocks
# (`-b/-e`) and take mean/std across the .xvg files, or use the Python blocks.

# %%  (Optional) plot straight from gmx order -od .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # carbon, -SCD

# gmx_files = {("DOPC","C3"): {"65/35": "deuter_DOPC_sn1_65.xvg"}}
# for (lip, prefix), fdict in gmx_files.items():
#     plt.figure(figsize=(5, 3.6))
#     for L in labels_sorted:
#         p = fdict.get(L)
#         if p and os.path.exists(p):
#             x, y = read_xvg_xy(p); plt.plot(x, y, color=colors[L], marker="o", ms=3, label=L)
#     plt.xlabel("Carbon number"); plt.ylabel(r"$-S_{CD}$"); plt.legend(); plt.title(f"{lip} {prefix}")
#     plt.show()

Lipid lateral diffusion coefficient

In [ ]:
# %% [markdown]
# # Lipid lateral diffusion coefficient - DOPC/DOPA composition series
#
# Lateral mean-squared displacement of each lipid's centre of mass, with
# per-leaflet COM drift removed, fitted as  MSD_xy(t) = 4 D t  ->  D = slope/4.
# Reported per lipid (DOPC, DOPA) for each composition.
#
# Plots (overlay style, one distinct colour per composition):
#   panel 1: DOPC MSD vs lag      panel 2: DOPA MSD vs lag
#   panel 3: fitted D vs DOPA %
# Error bands / error bars come from block averaging (NBLOCKS sub-trajectories).
#
# *** Trajectory to use: the NO-JUMP trajectory ***  (this is the ONLY property
# that uses nojump). Make it first, stripping water to keep it small:
#
#   gmx trjconv -s Prod_65.tpr -f <full_traj>.xtc -n mem.ndx \
#       -pbc nojump -o Prod_nojump_65.xtc            # choose the Membrane group
#
# Do NOT use the centered/whole xtc here - individual lipids still wrap around the
# box in those and the MSD will be wrong.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DOPA"]
HEAD     = "P"
NBLOCKS  = 5           # sub-trajectories for the error estimate
STEP     = 1           # frame stride
FITMIN   = 0.2         # fit window (fraction of each block's lag range)
FITMAX   = 0.6

# label -> (topology, NO-JUMP trajectory, DOPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_nojump_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_nojump_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_nojump_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_nojump_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_nojump_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#2ca02c",   # green
    "72/28": "#1f77b4",   # blue
    "80/20": "#ff7f0e",   # orange
    "87/13": "#9467bd",   # purple
    "95/5" : "#d62728",   # red
}
os.makedirs("results", exist_ok=True)

# %%  Diffusion core (lateral MSD, per-leaflet drift removed, block-averaged)
def _msd_curve(seg):
    # seg: (Ls, n, 2) lateral COM positions -> MSD over lags 0..Ls-1
    Ls = seg.shape[0]
    msd = np.zeros(Ls)
    for lag in range(1, Ls):
        disp = seg[lag:] - seg[:-lag]
        msd[lag] = (disp ** 2).sum(axis=2).mean()
    return msd

def diffusion_blocks(u, resname, head=HEAD, nblocks=NBLOCKS, step=STEP,
                     fitmin=FITMIN, fitmax=FITMAX):
    residues = u.select_atoms(f"resname {resname}").residues
    if len(residues) == 0:
        return None
    P = u.select_atoms(f"name {head} and resname {resname}")
    # leaflet assignment from frame 0
    u.trajectory[0]
    zc0 = P.positions[:, 2].mean()
    pz = {a.resid: a.position[2] for a in P}
    leaf = np.array([0 if pz.get(r.resid, zc0) >= zc0 else 1 for r in residues])

    # gather lateral COM per frame with per-leaflet drift removed
    times, coms = [], []
    for ts in u.trajectory[::step]:
        times.append(ts.time)
        com = np.array([r.atoms.center_of_mass()[:2] for r in residues])
        for lf in (0, 1):
            m = leaf == lf
            if m.any():
                com[m] -= com[m].mean(axis=0)
        coms.append(com)
    times = np.array(times)
    coms = np.array(coms)                      # (T, n, 2)
    T = len(times)
    dt = np.mean(np.diff(times)) if T > 1 else 1.0

    # split into contiguous blocks; MSD per block on a common lag range
    segs = [s for s in np.array_split(np.arange(T), max(1, min(nblocks, T))) if len(s) > 2]
    Lmin = min(len(s) for s in segs)
    lags = np.arange(Lmin) * dt
    msds, Ds = [], []
    i0, i1 = max(1, int(fitmin * Lmin)), int(fitmax * Lmin)
    if i1 - i0 < 2:
        i0, i1 = 1, Lmin
    for s in segs:
        m = _msd_curve(coms[s][:Lmin])
        msds.append(m)
        Ds.append(np.polyfit(lags[i0:i1], m[i0:i1], 1)[0] / 4.0 * 1e-4)  # cm^2/s
    M = np.array(msds)
    Ds = np.array(Ds)
    return {"lags": lags, "msd": M.mean(0), "msd_std": M.std(0) if len(M) > 1 else np.zeros(Lmin),
            "D": Ds.mean(), "D_err": Ds.std() if len(Ds) > 1 else 0.0}

# %%  Compute for every composition and lipid
results = {lip: {} for lip in LIPIDS}
Dtab = {}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for lip in LIPIDS:
        out = diffusion_blocks(u, lip)
        if out is None:
            continue
        results[lip][label] = out
        Dtab[(lip, label)] = (out["D"], out["D_err"])
        np.savetxt(f"results/msd_{lip}_{label.replace('/', '-')}.dat",
                   np.column_stack([out["lags"], out["msd"], out["msd_std"]]),
                   header="lag_ps  msd_A2  std")

print(f"\n{'comp':<8}" + "".join(f"{lip+' D(1e-7)':>18}" for lip in LIPIDS))
for label in SYSTEMS:
    row = ""
    for lip in LIPIDS:
        if (lip, label) in Dtab:
            D, e = Dtab[(lip, label)]
            row += f"{D*1e7:11.3f}+/-{e*1e7:5.3f}"
        else:
            row += f"{'-':>18}"
    print(f"{label:<8}{row}")
print("\n(D in cm^2/s; column shows D x 1e7)")

# %%  Plot - MSD overlays + D vs composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))
# panels 1 & 2: MSD per lipid
for ax, lip in zip(axes[:2], LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        o = results[lip][L]
        lag_k = o["lags"] / 1000.0          # ps -> thousands of ps
        ax.fill_between(lag_k, o["msd"] - o["msd_std"], o["msd"] + o["msd_std"],
                        color=colors[L], alpha=0.18, lw=0)
        ax.plot(lag_k, o["msd"], color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DOPA)")
    ax.set_xlabel(r"lag time ($\times10^{3}$ ps)")
    ax.set_ylabel(r"MSD$_{xy}$ ($\AA^2$)")
    ax.text(0.05, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="left", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)

# panel 3: D vs DOPA %
ax = axes[2]
markers = {LIPIDS[0]: "o", LIPIDS[-1]: "s"}
for lip in LIPIDS:
    xs = [SYSTEMS[L][2] for L in labels_sorted if L in results[lip]]
    ys = [Dtab[(lip, L)][0] * 1e7 for L in labels_sorted if L in results[lip]]
    es = [Dtab[(lip, L)][1] * 1e7 for L in labels_sorted if L in results[lip]]
    cs = [colors[L] for L in labels_sorted if L in results[lip]]
    ax.plot(xs, ys, color="0.6", lw=1.0, zorder=1)
    ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor="0.5", capsize=3, zorder=2)
    ax.scatter(xs, ys, c=cs, marker=markers.get(lip, "o"), s=55,
               edgecolor="k", linewidth=0.5, zorder=3, label=lip)
ax.set_xlabel("DOPA %")
ax.set_ylabel(r"$D$ (10$^{-7}$ cm$^2$/s)")
ax.text(0.05, 0.92, "D vs composition", transform=ax.transAxes,
        ha="left", va="top", fontsize=11)
ax.tick_params(direction="in", top=True, right=True)
ax.legend(title="lipid (marker)", fontsize=9)

handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("DOPC/DOPA lateral diffusion vs composition", y=1.15)
plt.tight_layout()
plt.savefig("results/diffusion_DOPC_DOPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route - `gmx msd`
#
# `gmx msd -lateral z` gives the in-plane MSD and prints D. Run on the NO-JUMP
# trajectory. Fit only the linear (diffusive) regime with `-beginfit/-endfit`.
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_nojump_65.xtc ; NDX=index.ndx
#
# # DOPC (choose the DOPC group at the prompt):
# gmx msd -s $TPR -f $XTC -n $NDX -lateral z -rmcomm \
#     -beginfit 20000 -endfit 80000 -o msd_DOPC_65.xvg
#
# # DOPA:
# gmx msd -s $TPR -f $XTC -n $NDX -lateral z -rmcomm \
#     -beginfit 20000 -endfit 80000 -o msd_DOPA_65.xvg
# ```
#
# `-lateral z` = in-plane MSD, D = slope/4 (gmx prints it). `-rmcomm` removes the
# overall COM motion; the Python version additionally removes per-leaflet drift,
# which is the more careful choice. Report the finite-size (Yeh-Hummer) caveat:
# absolute D depends on box size - compare compositions at matched box sizes.

# %%  (Optional) read D straight from gmx msd terminal output / .xvg
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # time(ps), MSD

# gmx prints "D[ DOPC] = ... (1e-5 cm^2/s)" to the terminal; you can also refit the
# .xvg yourself:
# t, msd = read_xvg_xy("msd_DOPC_65.xvg")
# i0, i1 = int(0.2*len(t)), int(0.6*len(t))
# slope = np.polyfit(t[i0:i1], msd[i0:i1], 1)[0]
# print("D =", slope/4*1e-4, "cm^2/s")

In [ ]:
# %% [markdown]
# # Voronoi analysis (area per lipid) - DOPC/DOPA composition series
#
# Periodic 2D Voronoi tessellation of the phosphate (P) atoms, done per leaflet.
# From it, per lipid type: area per lipid (APL), mean number of Voronoi
# neighbours, and the like-neighbour fraction (fraction of a lipid's neighbours
# that are the SAME type - a mixing / clustering signal).
#
# Plots (overlay style, one distinct colour per composition):
#   panel 1: DOPC APL distribution P(APL)   panel 2: DOPA APL distribution
#   panel 3: mean APL vs DOPA %
# Bands / error bars from block averaging (NBLOCKS sub-trajectories).
#
# Trajectory to use: a **whole + centered** xtc. Voronoi only needs the P atoms,
# so the small P-only `heads` trajectory is ideal (or the full one). Not nojump.
#
# Requires scipy:  pip install scipy

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy scipy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
from scipy.spatial import Voronoi
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DOPA"]
HEAD     = "P"
APL_MIN  = 30.0        # APL histogram range (A^2)
APL_MAX  = 110.0
APL_BINW = 1.5
NBLOCKS  = 5           # time blocks for the error estimate
STEP     = 1           # frame stride (raise to subsample heavy trajectories)

# label -> (topology, whole+centered trajectory, DOPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#2ca02c",   # green
    "72/28": "#1f77b4",   # blue
    "80/20": "#ff7f0e",   # orange
    "87/13": "#9467bd",   # purple
    "95/5" : "#d62728",   # red
}
os.makedirs("results", exist_ok=True)

# %%  Voronoi core (periodic, per leaflet)
def _poly_area(p):
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def _tile(pts, Lx, Ly):
    sh = [(i, j) for i in (-1, 0, 1) for j in (-1, 0, 1)]
    tp = np.vstack([pts + np.array([a * Lx, b * Ly]) for a, b in sh])
    tmap = np.concatenate([np.arange(len(pts)) for _ in sh])
    return tp, tmap

def _leaflet_voronoi(pts, rtypes, Lx, Ly):
    # returns per-point: area, neighbour count, like-neighbour fraction
    N = len(pts)
    tp, tmap = _tile(pts, Lx, Ly)
    vor = Voronoi(tp)
    central = np.arange(4 * N, 5 * N)          # the (0,0) image block
    areas = np.full(N, np.nan)
    for k, pi in enumerate(central):
        reg = vor.regions[vor.point_region[pi]]
        if reg and -1 not in reg:
            areas[k] = _poly_area(vor.vertices[reg])
    neigh = [set() for _ in range(N)]
    for i, j in vor.ridge_points:
        if 4 * N <= i < 5 * N:
            neigh[i - 4 * N].add(tmap[j])
        if 4 * N <= j < 5 * N:
            neigh[j - 4 * N].add(tmap[i])
    for k in range(N):
        neigh[k].discard(k)
    nn = np.array([len(neigh[k]) for k in range(N)], float)
    like = np.array([np.mean([rtypes[m] == rtypes[k] for m in neigh[k]])
                     if neigh[k] else np.nan for k in range(N)])
    return areas, nn, like

def voronoi_blocks(u, lipids=LIPIDS, head=HEAD, apl_min=APL_MIN, apl_max=APL_MAX,
                   apl_binw=APL_BINW, nblocks=NBLOCKS, step=STEP):
    P = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    rt_all = np.array([a.resname for a in P])
    edges = np.arange(apl_min, apl_max + apl_binw, apl_binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]

    hist = {lip: [] for lip in lipids}     # per-block density histogram
    aplm = {lip: [] for lip in lipids}     # per-block mean APL
    nnm = {lip: [] for lip in lipids}
    likem = {lip: [] for lip in lipids}
    for blk in blocks:
        bucket = {lip: {"apl": [], "nn": [], "like": []} for lip in lipids}
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            pos = P.positions.copy()
            pos[:, 0] %= Lx
            pos[:, 1] %= Ly
            zmid = pos[:, 2].mean()
            for mask in (pos[:, 2] >= zmid, pos[:, 2] < zmid):
                sub = pos[mask][:, :2]
                rs = rt_all[mask]
                if len(sub) < 4:
                    continue
                ar, nn, lk = _leaflet_voronoi(sub, rs, Lx, Ly)
                for lip in lipids:
                    m = rs == lip
                    bucket[lip]["apl"] += list(ar[m][~np.isnan(ar[m])])
                    bucket[lip]["nn"] += list(nn[m])
                    bucket[lip]["like"] += list(lk[m][~np.isnan(lk[m])])
        for lip in lipids:
            a = np.array(bucket[lip]["apl"])
            if a.size:
                hist[lip].append(np.histogram(a, bins=edges, density=True)[0])
                aplm[lip].append(a.mean())
                nnm[lip].append(np.mean(bucket[lip]["nn"]))
                likem[lip].append(np.nanmean(bucket[lip]["like"]))

    out = {}
    for lip in lipids:
        if not hist[lip]:
            continue
        H = np.array(hist[lip])
        out[lip] = {
            "apl": ctr, "P": H.mean(0), "Pstd": H.std(0) if len(H) > 1 else np.zeros(len(ctr)),
            "mean_apl": np.mean(aplm[lip]), "apl_err": np.std(aplm[lip]) if len(aplm[lip]) > 1 else 0.0,
            "nn": np.mean(nnm[lip]), "like": np.nanmean(likem[lip]),
        }
    return out

# %%  Compute for every composition
results = {lip: {} for lip in LIPIDS}
summary = {}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    out = voronoi_blocks(u)
    for lip in LIPIDS:
        if lip in out:
            results[lip][label] = out[lip]
            summary[(lip, label)] = out[lip]
            np.savetxt(f"results/voronoi_{lip}_{label.replace('/', '-')}.dat",
                       np.column_stack([out[lip]["apl"], out[lip]["P"], out[lip]["Pstd"]]),
                       header="APL_A2  P  std")

print(f"\n{'comp':<8}{'lipid':<7}{'APL(A^2)':>12}{'err':>7}{'<neigh>':>9}{'like':>7}")
for label in SYSTEMS:
    for lip in LIPIDS:
        if (lip, label) in summary:
            s = summary[(lip, label)]
            print(f"{label:<8}{lip:<7}{s['mean_apl']:12.2f}{s['apl_err']:7.2f}"
                  f"{s['nn']:9.2f}{s['like']:7.3f}")

# %%  Plot - APL distributions + mean APL vs composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))
for ax, lip in zip(axes[:2], LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        o = results[lip][L]
        ax.fill_between(o["apl"], o["P"] - o["Pstd"], o["P"] + o["Pstd"],
                        color=colors[L], alpha=0.18, lw=0)
        ax.plot(o["apl"], o["P"], color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DOPA)")
    ax.set_xlabel(r"Area per lipid ($\AA^2$)")
    ax.set_ylabel("P(APL)")
    ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="right", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)

ax = axes[2]
markers = {LIPIDS[0]: "o", LIPIDS[-1]: "s"}
for lip in LIPIDS:
    xs = [SYSTEMS[L][2] for L in labels_sorted if L in results[lip]]
    ys = [summary[(lip, L)]["mean_apl"] for L in labels_sorted if L in results[lip]]
    es = [summary[(lip, L)]["apl_err"] for L in labels_sorted if L in results[lip]]
    cs = [colors[L] for L in labels_sorted if L in results[lip]]
    ax.plot(xs, ys, color="0.6", lw=1.0, zorder=1)
    ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor="0.5", capsize=3, zorder=2)
    ax.scatter(xs, ys, c=cs, marker=markers.get(lip, "o"), s=55,
               edgecolor="k", linewidth=0.5, zorder=3, label=lip)
ax.set_xlabel("DOPA %")
ax.set_ylabel(r"mean APL ($\AA^2$)")
ax.text(0.05, 0.92, "APL vs composition", transform=ax.transAxes,
        ha="left", va="top", fontsize=11)
ax.tick_params(direction="in", top=True, right=True)
ax.legend(title="lipid (marker)", fontsize=9)

handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("DOPC/DOPA area per lipid vs composition", y=1.15)
plt.tight_layout()
plt.savefig("results/voronoi_DOPC_DOPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS cross-check - overall mean APL from the box
#
# GROMACS has no built-in per-lipid Voronoi tool, but the **overall** mean APL is
# just the box cross-section divided by lipids per leaflet:
#     APL_mean = 2 * Box-X * Box-Y / N_lipids
# Get the box vectors from the energy file:
#
# ```bash
# gmx energy -f Prod_65.edr -o box_65.xvg     # choose Box-X and Box-Y
# ```
#
# Then in Python:  APL = 2 * <Box-X * Box-Y> / N_lipids  (should match the
# mixture-averaged mean of the two Voronoi APLs). For per-lipid-type APL you need
# the Voronoi tessellation above, or a dedicated tool (FATSLiM, APL@Voro,
# GridMAT-MD).

# %%  (Optional) box-based mean APL from a gmx energy .xvg
def read_xvg(path):
    leg, rows = [], []
    for line in open(path):
        if line.startswith("@") and " legend " in line:
            leg.append(line.split('"')[1])
        elif not line.startswith(("#", "@")):
            rows.append([float(x) for x in line.split()])
    return leg, np.array(rows)

# leg, d = read_xvg("box_65.xvg")   # columns: time, Box-X, Box-Y (nm)
# N_lipids = 200                     # lipids per bilayer for that system
# bx, by = d[:, 1], d[:, 2]
# apl_nm2 = 2 * (bx * by).mean() / N_lipids
# print("overall mean APL = %.2f A^2" % (apl_nm2 * 100))   # nm^2 -> A^2

In [ ]:
# %% [markdown]
# # Lipid spatial distribution - DOPC/DOPA composition series
#
# Time-averaged 2D lateral density of each lipid's phosphate in the membrane
# plane, per leaflet, shown as an ENRICHMENT map: each map is divided by its own
# mean, so 1.0 = uniform, >1 = locally enriched (clustering), <1 = depleted.
#
# Unlike the other properties this one is a 2D map, so it is shown as a grid of
# heatmaps (rows = composition, columns = DOPC / DOPA) rather than overlaid
# curves. The radial (curve) version of the same information is the cross-RDF
# (DOPC-DOPA), done in the RDF script.
#
# Trajectory to use: a **whole + centered** xtc. Only the P atoms are needed, so
# the small P-only `heads` trajectory works well (or the full one). Not nojump.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DOPA"]
HEAD     = "P"
BINS     = 40           # grid resolution (BINS x BINS)
LEAFLET  = "upper"      # "upper", "lower", or "both"
SMOOTH   = 1.0          # Gaussian-ish smoothing passes (0 = none); simple box blur
STEP     = 1
VMIN, VMAX = 0.5, 1.5   # enrichment colour range (1.0 = uniform)

# label -> (topology, whole+centered trajectory, DOPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}
os.makedirs("results", exist_ok=True)

# %%  Density-map core (per leaflet, enrichment = map / mean)
def _leaf_mask(z, zmid, leaflet):
    if leaflet == "upper":
        return z >= zmid
    if leaflet == "lower":
        return z < zmid
    return np.ones(len(z), dtype=bool)          # both

def _box_blur(M, passes):
    for _ in range(int(passes)):
        M = (M
             + np.roll(M, 1, 0) + np.roll(M, -1, 0)
             + np.roll(M, 1, 1) + np.roll(M, -1, 1)) / 5.0
    return M

def density_maps(u, lipids=LIPIDS, head=HEAD, bins=BINS, leaflet=LEAFLET,
                 smooth=SMOOTH, step=STEP):
    allP = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    sel = {lip: u.select_atoms(f"name {head} and resname {lip}") for lip in lipids}
    H = {lip: np.zeros((bins, bins)) for lip in lipids}
    nframes = 0
    for ts in u.trajectory[::step]:
        Lx, Ly = ts.dimensions[0], ts.dimensions[1]
        zmid = allP.positions[:, 2].mean()
        ex = np.linspace(0, Lx, bins + 1)
        ey = np.linspace(0, Ly, bins + 1)
        for lip in lipids:
            p = sel[lip].positions
            m = _leaf_mask(p[:, 2], zmid, leaflet)
            q = p[m]
            if len(q) == 0:
                continue
            H[lip] += np.histogram2d(np.mod(q[:, 0], Lx), np.mod(q[:, 1], Ly),
                                     bins=[ex, ey])[0]
        nframes += 1
    maps = {}
    for lip in lipids:
        M = H[lip] / max(nframes, 1)
        if smooth:
            M = _box_blur(M, smooth)
        mean = M.mean()
        maps[lip] = M / mean if mean > 0 else M       # enrichment (uniform = 1)
    return maps

# %%  Compute for every composition
results = {}     # label -> {lip: enrichment_map}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    maps = density_maps(u)
    results[label] = maps
    for lip in LIPIDS:
        np.savetxt(f"results/spatial_{lip}_{label.replace('/', '-')}.dat", maps[lip])
print("done")

# %%  Plot - grid of enrichment maps (rows = composition, cols = lipid)
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DOPA -> low
present = [L for L in labels_sorted if L in results]
nrow, ncol = len(present), len(LIPIDS)
fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.0 * nrow), squeeze=False)
norm = TwoSlopeNorm(vmin=VMIN, vcenter=1.0, vmax=VMAX)
im = None
for r, L in enumerate(present):
    for c, lip in enumerate(LIPIDS):
        ax = axes[r, c]
        im = ax.imshow(results[L][lip].T, origin="lower", cmap="RdBu_r",
                       norm=norm, interpolation="bilinear", aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(lip, fontsize=12, style="italic", fontweight="bold")
        if c == 0:
            ax.set_ylabel(f"{L}\n({SYSTEMS[L][2]}% DOPA)", fontsize=10)
fig.suptitle(f"DOPC/DOPA lateral enrichment ({LEAFLET} leaflet)  -  "
             r"P density / mean", y=1.005, fontsize=12)
fig.subplots_adjust(right=0.88)
cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cb = fig.colorbar(im, cax=cax)
cb.set_label("enrichment (density / mean)")
plt.savefig("results/spatial_distribution_DOPC_DOPA_composition.png",
            dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## Notes
#
# * For a well-mixed FLUID bilayer the time-averaged map is close to uniform
#   (~1 everywhere) because lipids diffuse; strong red patches that persist =
#   real clustering/domains. If maps look flat, that is a physical result (good
#   mixing), and the cross-RDF quantifies any short-range preference.
# * `LEAFLET` switches between the two monolayers; run both to check symmetry.
# * `VMIN/VMAX` set the colour range around 1.0 (uniform). Tighten them
#   (e.g. 0.8-1.2) to make weak enrichment visible.

# %%  (Optional) representative snapshot - top view of P atoms coloured by lipid
def snapshot(u, lipids=LIPIDS, head=HEAD, leaflet=LEAFLET, frame=-1):
    u.trajectory[frame]
    allP = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    zmid = allP.positions[:, 2].mean()
    cols = {"DOPC": "#1f77b4", "DOPA": "#d62728"}
    plt.figure(figsize=(4.2, 4.0))
    for lip in lipids:
        p = u.select_atoms(f"name {head} and resname {lip}").positions
        m = _leaf_mask(p[:, 2], zmid, leaflet)
        plt.scatter(p[m, 0], p[m, 1], s=25, color=cols.get(lip, "k"),
                    edgecolor="k", linewidth=0.3, label=lip)
    plt.gca().set_aspect("equal"); plt.xlabel("x (Å)"); plt.ylabel("y (Å)")
    plt.legend(); plt.title(f"snapshot ({leaflet} leaflet)"); plt.tight_layout(); plt.show()

# example:
# u = mda.Universe(*SYSTEMS["65/35"][:2]); snapshot(u)

# %% [markdown]
# ## GROMACS route - `gmx densmap`
#
# `gmx densmap` writes the time-averaged 2D density of a group in the xy plane.
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc ; NDX=index.ndx
#
# # DOPC phosphate 2D map (choose the "DOPC_P" group, or make one with make_ndx):
# gmx densmap -s $TPR -f $XTC -n $NDX -bin 0.1 -od densmap_DOPC_65.dat
# # repeat with the DOPA P group -> densmap_DOPA_65.dat
# ```
#
# `-bin 0.1` = 0.1 nm grid. The .dat is a plain matrix you can load with
# `np.loadtxt` and imshow exactly like the maps above (divide by its mean for the
# enrichment view). `gmx densmap` averages over both leaflets and all z by
# default; for a single leaflet, restrict the input group to that leaflet's P
# atoms (leaflet assignment is dynamic, so the Python route above is easier).

# %%  (Optional) load a gmx densmap .dat and show it
# M = np.loadtxt("densmap_DOPC_65.dat")
# plt.imshow((M/M.mean()).T, origin="lower", cmap="RdBu_r",
#            norm=TwoSlopeNorm(0.5, 1.0, 1.5)); plt.colorbar(); plt.show()

In [ ]:
# %% [markdown]
# # Lipid-lipid interaction energies - DOPC/DOPA composition series
#
# Short-range pair interaction energies read straight from the `.edr` files
# (no trajectory needed). For each pair we report:
#   Coul-SR  = electrostatic (partial)   LJ-SR = van der Waals (partial)
#   total    = Coul-SR + LJ-SR           (the "lipid-lipid" interaction)
#
# **Requirement:** the per-pair terms (`Coul-SR:DOPC-DOPA`, `LJ-SR:DOPC-DOPA`,
# `Coul-SR:DOPC-DOPC`, ...) only exist in the .edr if `energygrps = DOPC DOPA`
# was set in the mdp of the run that made it. The first cell lists what each .edr
# actually contains; if the terms are missing, regenerate them with a rerun
# (see the GROMACS cell at the bottom).
#
# Plot: 3 pair panels (DOPC-DOPC, DOPA-DOPA, DOPC-DOPA), energy vs DOPA %, with
# Coul-SR / LJ-SR / total as separate series. Error bars = block-averaged SEM.
#
# Caveat: with PME the reciprocal-space Coulomb is NOT group-decomposable, so
# Coul-SR is the real-space part only - the standard, widely reported choice.
# Keep cutoffs identical across compositions.

# %%  Setup
# pip install panedr pandas numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
try:
    import panedr
    HAVE_PANEDR = True
except Exception:
    HAVE_PANEDR = False
print("panedr available:", HAVE_PANEDR)

# %%  Config  -- EDIT THE PATHS
NBLOCKS = 5           # time blocks for the SEM
PAIRS = [("DOPC", "DOPC"), ("DOPA", "DOPA"), ("DOPC", "DOPA")]

# label -> (energy file .edr, DOPA_percent)
SYSTEMS = {
    "65/35": ("Prod_merged_65.edr", 35),
    "72/28": ("Prod_merged_72.edr", 28),
    "80/20": ("Prod_merged_80.edr", 20),
    "87/13": ("Prod_merged_87.edr", 13),
    "95/5" : ("Prod_merged_95.edr", 5),
}

# colours for the three energy components
CC = {"Coul-SR": "#1f77b4", "LJ-SR": "#2ca02c", "total": "#d62728"}
os.makedirs("results", exist_ok=True)

# %%  List the energy terms present in each .edr  (run this first!)
def load_edr(path):
    if not HAVE_PANEDR:
        raise RuntimeError("panedr not installed; pip install panedr")
    return panedr.edr_to_df(path)

for label, (edr, pct) in SYSTEMS.items():
    if not os.path.exists(edr):
        print(f"{label}: MISSING {edr}"); continue
    df = load_edr(edr)
    pair_terms = [c for c in df.columns if c.startswith(("Coul-SR:", "LJ-SR:"))]
    print(f"\n{label} ({edr}) - {df.shape[0]} frames; pair terms found:")
    for c in pair_terms:
        print("   ", c)
    if not pair_terms:
        print("    (none - energygrps was not set for this run)")

# %%  Stats helper (block-averaged mean + SEM)
def block_stats(x, nblocks=NBLOCKS):
    x = np.asarray(x, float)
    blocks = [b for b in np.array_split(x, max(1, min(nblocks, len(x)))) if len(b)]
    bm = np.array([b.mean() for b in blocks])
    mean = bm.mean()
    sem = bm.std() / np.sqrt(len(bm)) if len(bm) > 1 else 0.0
    return mean, sem

def term(df, a, b, kind):
    for nm in (f"{kind}:{a}-{b}", f"{kind}:{b}-{a}"):
        if nm in df.columns:
            return df[nm].values
    return None

# %%  Compute for every composition and pair
results = {p: {} for p in PAIRS}     # pair -> {label: dict(coul, lj, tot + SEMs)}
for label, (edr, pct) in SYSTEMS.items():
    if not os.path.exists(edr):
        print(f"  skip {label}: missing {edr}"); continue
    df = load_edr(edr)
    for (a, b) in PAIRS:
        coul = term(df, a, b, "Coul-SR")
        lj = term(df, a, b, "LJ-SR")
        if coul is None and lj is None:
            continue
        coul = coul if coul is not None else np.zeros(len(df))
        lj = lj if lj is not None else np.zeros(len(df))
        cm, ce = block_stats(coul)
        lm, le = block_stats(lj)
        tm, te = block_stats(coul + lj)
        results[(a, b)][label] = dict(coul=cm, coul_e=ce, lj=lm, lj_e=le, tot=tm, tot_e=te)

# table
print(f"\n{'pair':<12}{'comp':<7}{'Coul-SR':>12}{'LJ-SR':>12}{'total':>12}  (kJ/mol)")
for (a, b) in PAIRS:
    for label in SYSTEMS:
        if label in results[(a, b)]:
            r = results[(a, b)][label]
            print(f"{a+'-'+b:<12}{label:<7}{r['coul']:12.1f}{r['lj']:12.1f}{r['tot']:12.1f}")
            np.savetxt(f"results/energy_{a}-{b}_{label.replace('/', '-')}.dat",
                       np.array([[r['coul'], r['coul_e'], r['lj'], r['lj_e'], r['tot'], r['tot_e']]]),
                       header="Coul SEM  LJ SEM  total SEM (kJ/mol)")

# %%  Plot - 3 pair panels, energy vs DOPA %
labels_sorted = sorted(SYSTEMS, key=lambda L: SYSTEMS[L][1])   # low -> high DOPA
fig, axes = plt.subplots(1, len(PAIRS), figsize=(4.6 * len(PAIRS), 4.0), sharex=True)
if len(PAIRS) == 1:
    axes = [axes]
for ax, (a, b) in zip(axes, PAIRS):
    labs = [L for L in labels_sorted if L in results[(a, b)]]
    xs = [SYSTEMS[L][1] for L in labs]
    for key, comp in (("coul", "Coul-SR"), ("lj", "LJ-SR"), ("tot", "total")):
        ys = [results[(a, b)][L][key] for L in labs]
        es = [results[(a, b)][L][key + "_e"] for L in labs]
        ax.errorbar(xs, ys, yerr=es, marker="o", ms=5, lw=1.6, capsize=3,
                    color=CC[comp], label=comp)
    ax.axhline(0, color="0.7", lw=0.6)
    ax.set_xlabel("DOPA %")
    ax.text(0.95, 0.06, f"{a}-{b}", transform=ax.transAxes, style="italic",
            ha="right", va="bottom", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("Interaction energy (kJ/mol)")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=3, frameon=True,
           fontsize=10, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("DOPC/DOPA short-range interaction energies vs composition", y=1.15)
plt.tight_layout()
plt.savefig("results/interaction_energy_DOPC_DOPA_composition.png",
            dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## Note on comparing across compositions
#
# Raw pair energies scale with the NUMBER of pairs, so e.g. DOPA-DOPA grows more
# negative simply because there is more DOPA at 65/35 than at 95/5. To compare the
# intrinsic interaction strength, normalise - e.g. divide the pair total by the
# number of that lipid (energy per DOPA), or per contact. Uncomment to add a
# per-lipid version if you have the lipid counts:
#
# ```python
# NLIP = {"65/35": {"DOPC": 130, "DOPA": 70}, ...}   # per bilayer
# # then plot results[pair][L]['tot'] / NLIP[L][b]  etc.
# ```

# %% [markdown]
# ## GROMACS route
#
# **If your .edr already lists the pair terms** (checked in cell 3), just pull
# them with gmx energy:
# ```bash
# gmx energy -f Prod_65.edr -o inter_DOPC-DOPA_65.xvg   # pick Coul-SR:DOPC-DOPA  LJ-SR:DOPC-DOPA
# ```
#
# **If the pair terms are missing**, regenerate them with a rerun that defines the
# energy groups (needs the trajectory; run on CPU - GPU nonbonded can't decompose
# energy groups):
# ```bash
# # add to a copy of the mdp:   energygrps = DOPC DOPA
# gmx grompp -f rerun.mdp -c Prod_65.gro -p topol.top -n index.ndx -o rerun_65.tpr -maxwarn 2
# gmx mdrun  -s rerun_65.tpr -rerun Prod_merged_center_65.xtc -e ener_65.edr -nb cpu
# gmx energy -f ener_65.edr -o inter_DOPC-DOPA_65.xvg
# ```

# %%  (Optional) read a gmx energy .xvg instead of panedr
def read_xvg(path):
    leg, rows = [], []
    for line in open(path):
        if line.startswith("@") and " legend " in line:
            leg.append(line.split('"')[1])
        elif not line.startswith(("#", "@")):
            rows.append([float(x) for x in line.split()])
    return leg, np.array(rows)

# leg, d = read_xvg("inter_DOPC-DOPA_65.xvg")
# for i, name in enumerate(leg):
#     m, e = block_stats(d[:, i + 1])
#     print(f"{name:<22}{m:10.1f} +/- {e:.1f} kJ/mol")

In [ ]:
import sys
!{sys.executable} -m pip install panedr

In [ ]:
# %% [markdown]
# # RDF - DOPC/DOPG composition series  (VS Code interactive script)
#
# Run cell-by-cell with the "Run Cell" links above each `# %%` (VS Code's Python
# Interactive window), or run the whole file.
#
# Trajectory to use: a **whole + centered** xtc. Do NOT use the nojump xtc here -
# nojump is only for diffusion.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the VS Code terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
HEAD    = "P"
RMAX_NM = 5.0          # x-axis range (nm)
BIN_NM  = 0.02         # bin width (nm)
NBLOCKS = 5            # time blocks for the error band
STEP    = 1            # frame stride

# label -> (topology, trajectory[whole+centered], DOPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (different palette from the DOPA plot).
COLORS = {
    "65/35": "#e377c2",   # pink
    "72/28": "#17becf",   # cyan
    "80/20": "#bcbd22",   # olive
    "87/13": "#7f7f7f",   # gray
    "95/5" : "#8c564b",   # brown
}

PAIRS = [("DOPC", "DOPC"), ("DOPG", "DOPG"), ("DOPC", "DOPG")]
YLIM  = {("DOPC", "DOPC"): (0.5, 1.5),
         ("DOPG", "DOPG"): (0.5, 1.5),
         ("DOPC", "DOPG"): (0.75, 1.25)}
os.makedirs("results", exist_ok=True)

# %%  RDF core (per-leaflet, block-averaged)
def _pdist2d(a_xy, b_xy, Lx, Ly, same):
    """Minimum-image lateral distances (A) between two point sets."""
    if len(a_xy) == 0 or len(b_xy) == 0:
        return np.empty(0)
    dx = a_xy[:, None, 0] - b_xy[None, :, 0]
    dy = a_xy[:, None, 1] - b_xy[None, :, 1]
    dx -= Lx * np.round(dx / Lx)
    dy -= Ly * np.round(dy / Ly)
    r = np.sqrt(dx * dx + dy * dy)
    return r[np.triu_indices(r.shape[0], 1)] if same else r.ravel()

def rdf2d_blocks(u, resA, resB, head=HEAD, rmax_nm=RMAX_NM, bin_nm=BIN_NM,
                 nblocks=NBLOCKS, step=STEP):
    same = (resA == resB)
    rmax = rmax_nm * 10.0
    nbins = int(round(rmax / (bin_nm * 10.0)))
    edges = np.linspace(0, rmax, nbins + 1)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    shell = np.pi * (edges[1:] ** 2 - edges[:-1] ** 2)
    selA = u.select_atoms(f"name {head} and resname {resA}")
    selB = u.select_atoms(f"name {head} and resname {resB}")
    allP = u.select_atoms(f"name {head}")
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    gblocks = []
    for blk in blocks:
        hist = np.zeros(nbins)
        norm = 0.0
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            area = Lx * Ly
            zmid = allP.positions[:, 2].mean()
            Apos, Bpos = selA.positions, selB.positions
            for sign in (True, False):                 # upper then lower leaflet
                a = Apos[(Apos[:, 2] >= zmid) == sign][:, :2]
                b = Bpos[(Bpos[:, 2] >= zmid) == sign][:, :2]
                hist += np.histogram(_pdist2d(a, b, Lx, Ly, same), bins=edges)[0]
                Na, Nb = len(a), len(b)
                norm += 0.5 * Na * ((Nb - 1) / area) if same else Na * (Nb / area)
        if norm > 0:
            gblocks.append(hist / (norm * shell))
    G = np.array(gblocks)
    mean = G.mean(axis=0)
    std = G.std(axis=0) if len(G) > 1 else np.zeros_like(mean)
    return ctr / 10.0, mean, std                        # r in nm

# %%  Compute for every composition and pair
results = {(a, b): {} for a, b in PAIRS}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for a, b in PAIRS:
        r, m, s = rdf2d_blocks(u, a, b)
        results[(a, b)][label] = (r, m, s)
        np.savetxt(f"results/rdf_{a}-{b}_{label.replace('/', '-')}.dat",
                   np.column_stack([r, m, s]), header="r_nm  g(r)  std")
print("done")

# %%  Plot - one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DOPG -> low
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, len(PAIRS), figsize=(4.2 * len(PAIRS), 3.6))
if len(PAIRS) == 1:
    axes = [axes]
for ax, (a, b) in zip(axes, PAIRS):
    for L in labels_sorted:
        if L not in results[(a, b)]:
            continue
        r, m, s = results[(a, b)][L]
        ax.fill_between(r, m - s, m + s, color=colors[L], alpha=0.18, lw=0)
        ax.plot(r, m, color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DOPG)")
    ax.axhline(1.0, color="0.6", lw=0.6, ls="--")
    ax.set_xlim(0, RMAX_NM)
    ax.set_ylim(*YLIM.get((a, b), (0.5, 1.5)))
    ax.set_xlabel("Distance (nm)")
    ax.text(0.95, 0.06, f"{a}-{b}", transform=ax.transAxes, style="italic",
            ha="right", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("RDF")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=n if (n := len(labels_sorted)) else 1,
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.12))
fig.suptitle("DOPC/DOPG lateral RDF (P-P) vs composition", y=1.18)
plt.tight_layout()
plt.savefig("results/rdf_DOPC_DOPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## Optional: plot straight from `gmx rdf` .xvg files
# ```bash
# D=DOPC_DOPG_65_35
# gmx rdf -s $D/md.tpr -f $D/md.xtc -n $D/index.ndx \
#     -ref "name P and resname DOPC" -sel "name P and resname DOPG" \
#     -xy -bin 0.02 -rmax 5 -o $D/rdf_DOPC-DOPG.xvg
# ```

# %%  read gmx xvg (optional)
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # r(nm), g(r)

# gmx_files = {"65/35": "rdf_DOPC-DOPG_65.xvg"}
# plt.figure(figsize=(4.5, 3.6))
# for L in labels_sorted:
#     if L in gmx_files:
#         r, g = read_xvg_xy(gmx_files[L]); plt.plot(r, g, color=colors[L], label=L)
# plt.axhline(1, ls="--", color="0.6", lw=.6); plt.xlim(0, RMAX_NM); plt.ylim(0.75, 1.25)
# plt.xlabel("Distance (nm)"); plt.ylabel("RDF"); plt.legend(); plt.show()

In [ ]:
# %% [markdown]
# # Lipid tail splay - DOPC/DOPG composition series  (VS Code interactive script)
#
# Intra-lipid splay = angle between a lipid's two acyl-chain vectors
# (sn-1: first->last C3x carbon; sn-2: first->last C2x carbon). We build the
# probability distribution P(theta) per composition, per lipid type, with an
# error band from block averaging.
#
# Trajectory to use: the **full-atom, whole + centered** xtc (e.g.
# `Prod_merged_center_*.xtc`). NOT the P-only `heads` trajectory (splay needs the
# chain carbons) and NOT the nojump one (that's diffusion only).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the VS Code terminal
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DOPG"]    # the two lipids -> one panel each
BINW_DEG = 2.0                 # splay-angle bin width (degrees)
ANG_MAX  = 180.0               # angle axis range
NBLOCKS  = 5                   # time blocks for the error band
STEP     = 1                   # frame stride

# label -> (topology, full-atom trajectory[whole+centered], DOPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (different palette from the DOPA plots).
COLORS = {
    "65/35": "#e377c2",   # pink
    "72/28": "#17becf",   # cyan
    "80/20": "#bcbd22",   # olive
    "87/13": "#7f7f7f",   # gray
    "95/5" : "#8c564b",   # brown
}
os.makedirs("results", exist_ok=True)

# %%  Splay core (block-averaged distribution)
def _carbons(res, prefix):
    """Acyl carbons of a chain ('C3'=sn-1, 'C2'=sn-2), skipping carbonyl idx 1."""
    pat = re.compile(r"^" + prefix + r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in res.atoms
            if (m := pat.match(at.name)) and int(m.group(1)) >= 2]
    return sorted(hits, key=lambda t: t[0])

def _splay_indices(u, resname):
    """(N,4) atom indices: sn1_first, sn1_last, sn2_first, sn2_last per lipid."""
    idx = []
    for res in u.select_atoms(f"resname {resname}").residues:
        c3 = _carbons(res, "C3")      # sn-1
        c2 = _carbons(res, "C2")      # sn-2
        if len(c3) >= 2 and len(c2) >= 2:
            idx.append((c3[0][1].index, c3[-1][1].index,
                        c2[0][1].index, c2[-1][1].index))
    return np.array(idx, dtype=int)

def splay_blocks(u, resname, binw=BINW_DEG, angmax=ANG_MAX,
                 nblocks=NBLOCKS, step=STEP):
    idx = _splay_indices(u, resname)
    edges = np.arange(0.0, angmax + binw, binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    if len(idx) == 0:
        return ctr, np.zeros_like(ctr), np.zeros_like(ctr), np.nan
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    Hs, all_ang = [], []
    for blk in blocks:
        ang = []
        for fi in blk:
            u.trajectory[int(fi)]
            pos = u.atoms.positions
            v1 = pos[idx[:, 1]] - pos[idx[:, 0]]      # sn-1 vector
            v2 = pos[idx[:, 3]] - pos[idx[:, 2]]      # sn-2 vector
            n1 = np.linalg.norm(v1, axis=1)
            n2 = np.linalg.norm(v2, axis=1)
            good = (n1 > 0) & (n2 > 0)
            cos = np.einsum("ij,ij->i", v1, v2) / (n1 * n2)
            ang.append(np.degrees(np.arccos(np.clip(cos[good], -1, 1))))
        ang = np.concatenate(ang)
        all_ang.append(ang)
        Hs.append(np.histogram(ang, bins=edges, density=True)[0])
    H = np.array(Hs)
    mean = H.mean(axis=0)
    std = H.std(axis=0) if len(H) > 1 else np.zeros_like(mean)
    mean_angle = np.concatenate(all_ang).mean()
    return ctr, mean, std, mean_angle

# %%  Compute for every composition and lipid
results = {lip: {} for lip in LIPIDS}   # lip -> {label: (theta, P, Pstd)}
means = {}                              # (lip,label) -> mean splay angle
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for lip in LIPIDS:
        theta, P, Pstd, mang = splay_blocks(u, lip)
        results[lip][label] = (theta, P, Pstd)
        means[(lip, label)] = mang
        np.savetxt(f"results/splay_{lip}_{label.replace('/', '-')}.dat",
                   np.column_stack([theta, P, Pstd]), header="angle_deg  P  std")

print("\nMean splay angle (deg):")
print(f"{'comp':<8}" + "".join(f"{lip:>10}" for lip in LIPIDS))
for label in SYSTEMS:
    row = "".join(f"{means.get((lip,label), float('nan')):10.1f}" for lip in LIPIDS)
    print(f"{label:<8}{row}")

# %%  Plot - one panel per lipid, one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DOPG -> low
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, len(LIPIDS), figsize=(4.6 * len(LIPIDS), 3.8),
                         sharex=True)
if len(LIPIDS) == 1:
    axes = [axes]
for ax, lip in zip(axes, LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        theta, P, Pstd = results[lip][L]
        ax.fill_between(theta, P - Pstd, P + Pstd, color=colors[L], alpha=0.18, lw=0)
        ax.plot(theta, P, color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DOPG)")
    ax.set_xlim(0, ANG_MAX)
    ax.set_xlabel("Tail splay angle (deg)")
    ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="right", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("P(θ)")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.10))
fig.suptitle("DOPC/DOPG intra-lipid tail splay vs composition", y=1.16)
plt.tight_layout()
plt.savefig("results/splay_DOPC_DOPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route (equivalent) - `gmx gangle`
#
# Splay = angle between the sn-1 and sn-2 chain vectors. `gmx gangle -g1 vector
# -g2 vector` pairs vector *i* of group1 with vector *i* of group2. Each vector is
# a pair of atoms (chain first & last carbon), so the selections below give one
# vector per lipid. DOPC/DOPG are 18:1 -> terminal carbons C318 (sn-1), C218 (sn-2).
#
# ```bash
# # one composition (repeat for 65,72,80,87,95), one lipid at a time:
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc
#
# # DOPC splay
# gmx gangle -s $TPR -f $XTC -g1 vector -g2 vector \
#     -group1 'name C32 C318 and resname DOPC' \
#     -group2 'name C22 C218 and resname DOPC' \
#     -oh splay_DOPC_65.xvg -binw 2
#
# # DOPG splay
# gmx gangle -s $TPR -f $XTC -g1 vector -g2 vector \
#     -group1 'name C32 C318 and resname DOPG' \
#     -group2 'name C22 C218 and resname DOPG' \
#     -oh splay_DOPG_65.xvg -binw 2
# ```
#
# `-oh` writes the normalized angle histogram P(theta). For the error band, run on
# time blocks (`-b/-e`) and take mean/std across the .xvg files, or use the Python
# block averaging above.

# %%  (Optional) plot straight from gmx gangle -oh .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # angle(deg), P

# gmx_files = {"DOPC": {"65/35": "splay_DOPC_65.xvg"}, "DOPG": {"65/35": "splay_DOPG_65.xvg"}}
# fig, axes = plt.subplots(1, len(LIPIDS), figsize=(4.6*len(LIPIDS), 3.8))
# for ax, lip in zip(np.atleast_1d(axes), LIPIDS):
#     for L in labels_sorted:
#         p = gmx_files.get(lip, {}).get(L)
#         if p and os.path.exists(p):
#             x, y = read_xvg_xy(p); ax.plot(x, y, color=colors[L], lw=1.8, label=L)
#     ax.set_xlabel("Tail splay angle (deg)"); ax.set_xlim(0, ANG_MAX)
#     ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic", ha="right", va="top")
# plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# %% [markdown]
# # Water depth / penetration - DOPC/DOPG composition series
#
# Symmetrised water density profile across the bilayer normal: water density
# (relative to bulk) vs z from the membrane centre, overlaid for the five
# compositions with a block-averaged error band. Deeper penetration = the curve
# stays high closer to z = 0.
#
# Also prints a table per composition: half-density depth (|z| where water falls
# to 0.5 of bulk), mean phosphate position (headgroup plane), and mean # waters in
# the hydrophobic core (|z| < core).
#
# Trajectory to use: the full **whole + centered** xtc that still contains water
# (`Prod_merged_center_*.xtc`). Not the water-stripped mem/heads trajectories, and
# not nojump (diffusion only). Centering matters here (profile is from the centre).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPID_SEL = "resname DOPC DOPG"          # membrane (defines the centre)
WATER_SEL = "resname TIP3 SOL and (name OH2 or name OW or name O)"  # water oxygen
HEAD      = "P"
ZMAX_NM   = 4.0     # profile half-range from centre (nm)
BINW_NM   = 0.05    # bin width (nm)
CORE_NM   = 1.0     # |z| < CORE counts as hydrophobic-core water
NBLOCKS   = 5       # time blocks for the error band
STEP      = 1

# label -> (topology, full trajectory[whole+centered, WITH water], DOPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#e377c2",   # pink
    "72/28": "#17becf",   # cyan
    "80/20": "#bcbd22",   # olive
    "87/13": "#7f7f7f",   # gray
    "95/5" : "#8c564b",   # brown
}
os.makedirs("results", exist_ok=True)

# %%  Water-profile core (block-averaged, symmetrised)
def water_profile_blocks(u, lipid_sel=LIPID_SEL, water_sel=WATER_SEL, head=HEAD,
                         zmax_nm=ZMAX_NM, binw_nm=BINW_NM, core_nm=CORE_NM,
                         nblocks=NBLOCKS, step=STEP):
    lip = u.select_atoms(lipid_sel)
    wat = u.select_atoms(water_sel)
    P = u.select_atoms(f"name {head}")
    zmax, binw, core = zmax_nm * 10, binw_nm * 10, core_nm * 10
    edges = np.arange(-zmax, zmax + binw, binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    nb_bins = len(ctr)
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]

    prof_blocks = []
    pzs, core_counts = [], []
    for blk in blocks:
        hw = np.zeros(nb_bins)
        areas = []
        nf = 0
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            zc = lip.center_of_mass()[2]
            zw = wat.positions[:, 2] - zc
            hw += np.histogram(zw, bins=edges)[0]
            areas.append(Lx * Ly)
            nf += 1
            pzs.append(np.abs(P.positions[:, 2] - zc).mean())
            core_counts.append(int(np.sum(np.abs(zw) < core)))
        vol = np.mean(areas) * binw                 # A^3 per bin
        dens = hw / nf / vol                          # atoms / A^3
        dens = 0.5 * (dens + dens[::-1])              # symmetrise about centre
        bulk = dens[np.abs(ctr) > (zmax - 10)].mean() # outer 1 nm = bulk
        prof_blocks.append(dens / bulk if bulk > 0 else dens)

    G = np.array(prof_blocks)
    mean = G.mean(axis=0)
    std = G.std(axis=0) if len(G) > 1 else np.zeros_like(mean)

    # half-density depth: |z| where the mean profile drops to 0.5 (going inward)
    zpos = ctr[ctr >= 0]
    mpos = mean[ctr >= 0]
    order = np.argsort(zpos)
    zpos, mpos = zpos[order], mpos[order]
    z_half = np.nan
    for i in range(len(zpos) - 1, 0, -1):
        if mpos[i] < 0.5 <= mpos[i - 1]:
            z_half = np.interp(0.5, [mpos[i], mpos[i - 1]], [zpos[i], zpos[i - 1]])
            break
    return (ctr / 10.0, mean, std, np.mean(pzs) / 10.0,
            (z_half / 10.0 if z_half == z_half else np.nan), np.mean(core_counts))

# %%  Compute for every composition
results = {}   # label -> (z, dens, std)
summary = {}   # label -> (phosphate_nm, half_depth_nm, core_waters)
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    z, d, s, pz, zhalf, ncore = water_profile_blocks(u)
    results[label] = (z, d, s)
    summary[label] = (pz, zhalf, ncore)
    np.savetxt(f"results/water_{label.replace('/', '-')}.dat",
               np.column_stack([z, d, s]), header="z_nm  water_rel_density  std")

print(f"\n{'comp':<8}{'P plane(nm)':>13}{'half-depth(nm)':>16}{'core waters':>13}")
for label in SYSTEMS:
    if label in summary:
        pz, zh, nc = summary[label]
        print(f"{label:<8}{pz:13.2f}{zh:16.2f}{nc:13.1f}")

# %%  Plot - water penetration, one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, ax = plt.subplots(figsize=(6.2, 4.2))
for L in labels_sorted:
    if L not in results:
        continue
    z, d, s = results[L]
    ax.fill_between(z, d - s, d + s, color=colors[L], alpha=0.18, lw=0)
    ax.plot(z, d, color=colors[L], lw=1.8, label=f"{L}  ({SYSTEMS[L][2]}% DOPG)")
# mark the mean phosphate plane
if summary:
    pmean = np.nanmean([summary[L][0] for L in summary])
    for xline in (pmean, -pmean):
        ax.axvline(xline, color="0.5", lw=0.8, ls=":")
    ax.text(pmean, 1.02, "P", color="0.4", ha="center", va="bottom", fontsize=10)
ax.axhline(0.5, color="0.8", lw=0.6, ls="--")
ax.set_xlim(-ZMAX_NM, ZMAX_NM)
ax.set_ylim(0, 1.15)
ax.set_xlabel("z from bilayer centre (nm)")
ax.set_ylabel("Water density (rel. to bulk)")
ax.text(0.03, 0.92, "Water", transform=ax.transAxes, style="italic",
        ha="left", va="top", fontsize=13, fontweight="bold")
ax.tick_params(direction="in", top=True, right=True)
ax.legend(loc="upper center", ncol=3, frameon=True, fontsize=8,
          bbox_to_anchor=(0.5, 1.18))
fig.suptitle("DOPC/DOPG water penetration vs composition", y=1.06)
plt.tight_layout()
plt.savefig("results/water_depth_DOPC_DOPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route (equivalent) - `gmx density`
#
# ```bash
# # one composition (repeat for 65,72,80,87,95):
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc ; NDX=index.ndx
#
# # centering group = Membrane , output group = Water
# printf "Membrane\nWater\n" | gmx density -s $TPR -f $XTC -n $NDX \
#     -d Z -sl 400 -center -symm -dens number -o water_density_65.xvg
#
# # phosphate plane for reference (output group = the P atoms):
# printf "Membrane\nPhosphate\n" | gmx density -s $TPR -f $XTC -n $NDX \
#     -d Z -sl 400 -center -symm -dens number -o phos_density_65.xvg
# ```
#
# `-center -symm` centres on the membrane and symmetrises; `-dens number` gives a
# number density. For error bands, run on time blocks (`-b/-e`) and take mean/std
# across the .xvg files, or use the Python block averaging above.

# %%  (Optional) plot straight from gmx density .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # z(nm), density

# gmx_files = {"65/35": "water_density_65.xvg"}
# plt.figure(figsize=(6.2, 4.2))
# for L in labels_sorted:
#     if L in gmx_files and os.path.exists(gmx_files[L]):
#         z, dw = read_xvg_xy(gmx_files[L])
#         plt.plot(z, dw / np.nanmax(dw), color=colors[L], lw=1.8, label=L)
# plt.xlabel("z (nm)"); plt.ylabel("water density (norm.)"); plt.legend(); plt.show()

In [ ]:
# %% [markdown]
# # Lipid tail order parameter (-S_CD) - DOPC/DOPG composition series
#
# Deuterium order parameter per acyl-chain carbon:
#   S_CD = < (3 cos^2(theta_CH) - 1) / 2 >    (theta_CH = C-H vs bilayer normal z)
# reported as -S_CD (the usual positive quantity). For each CH2 the two C-H
# values are averaged; C-H bonds are found automatically from the topology, so
# the cis double bond at C9-C10 in the oleoyl chains shows its dip on its own.
#
# One panel per (lipid x chain): DOPC/DOPG x sn-1/sn-2, each overlaying the five
# compositions with a block-averaged error band - same style as the RDF/splay
# figures.
#
# Trajectory to use: the full-atom **whole (+centered)** xtc WITH hydrogens
# (`Prod_merged_center_*.xtc`), plus the `.tpr` (needed for C-H bonds). Not the
# P-only `heads` trajectory, and not nojump (that is diffusion only).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import re
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS  = ["DOPC", "DOPG"]              # one row of panels each
CHAINS  = [("C3", "sn-1"), ("C2", "sn-2")]   # CHARMM36: sn-1=C3x, sn-2=C2x
NBLOCKS = 5                            # time blocks for the error band
STEP    = 1                            # frame stride

# label -> (topology[.tpr], full-atom trajectory[whole+centered], DOPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#e377c2",   # pink
    "72/28": "#17becf",   # cyan
    "80/20": "#bcbd22",   # olive
    "87/13": "#7f7f7f",   # gray
    "95/5" : "#8c564b",   # brown
}
os.makedirs("results", exist_ok=True)

# %%  Order-parameter core (block-averaged, auto C-H detection)
def _carbons(res, prefix):
    # acyl carbons of a chain ('C3'=sn-1, 'C2'=sn-2), skipping carbonyl idx 1
    pat = re.compile(r"^" + prefix + r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in res.atoms
            if (m := pat.match(at.name)) and int(m.group(1)) >= 2]
    return sorted(hits, key=lambda t: t[0])

def _build_pairs(u, resname, prefix, has_bonds):
    # returns (sorted carbon numbers, {k: (C_index_array, H_index_array)})
    C, H = defaultdict(list), defaultdict(list)
    for res in u.select_atoms(f"resname {resname}").residues:
        for k, catom in _carbons(res, prefix):
            if has_bonds:
                hs = [a for a in catom.bonded_atoms if a.name.startswith("H")]
            else:
                hs = list(u.select_atoms("name H* and around 1.3 group c",
                                         c=u.atoms[[catom.index]]))
            for h in hs:
                C[k].append(catom.index); H[k].append(h.index)
    ks = sorted(C)
    return ks, {k: (np.array(C[k]), np.array(H[k])) for k in ks}

def scd_blocks(u, resname, prefix, has_bonds, nblocks=NBLOCKS, step=STEP):
    ks, pairs = _build_pairs(u, resname, prefix, has_bonds)
    if not ks:
        return np.array([]), np.array([]), np.array([])
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    block_means = []
    for blk in blocks:
        acc = {k: [] for k in ks}
        for fi in blk:
            u.trajectory[int(fi)]
            pos = u.atoms.positions
            for k in ks:
                Ci, Hi = pairs[k]
                v = pos[Hi] - pos[Ci]
                n = np.linalg.norm(v, axis=1)
                cosz = v[:, 2] / n
                acc[k].append(np.mean(0.5 * (3 * cosz ** 2 - 1)))
        block_means.append([-np.mean(acc[k]) for k in ks])   # -S_CD
    B = np.array(block_means)
    return np.array(ks), B.mean(axis=0), (B.std(axis=0) if len(B) > 1 else np.zeros(len(ks)))

# %%  Compute for every composition, lipid and chain
results = {(lip, prefix): {} for lip in LIPIDS for prefix, _ in CHAINS}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    has_bonds = len(getattr(u, "bonds", [])) > 0
    print(f"{label}: {len(u.trajectory)} frames, bonds={has_bonds}")
    for lip in LIPIDS:
        for prefix, tag in CHAINS:
            ks, scd, err = scd_blocks(u, lip, prefix, has_bonds)
            if ks.size == 0:
                continue
            results[(lip, prefix)][label] = (ks, scd, err)
            np.savetxt(f"results/scd_{lip}_{tag}_{label.replace('/', '-')}.dat",
                       np.column_stack([ks, scd, err]), header="carbon  -SCD  std")
print("done")

# %%  Plot - 2x2 grid (lipid x chain), one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(len(LIPIDS), len(CHAINS),
                         figsize=(4.4 * len(CHAINS), 3.4 * len(LIPIDS)),
                         sharex=True, sharey=True)
axes = np.atleast_2d(axes)
for r, lip in enumerate(LIPIDS):
    for c, (prefix, tag) in enumerate(CHAINS):
        ax = axes[r, c]
        for L in labels_sorted:
            if L not in results[(lip, prefix)]:
                continue
            ks, scd, err = results[(lip, prefix)][L]
            ax.fill_between(ks, scd - err, scd + err, color=colors[L], alpha=0.18, lw=0)
            ax.plot(ks, scd, color=colors[L], lw=1.8, marker="o", ms=3,
                    label=f"{L}  ({SYSTEMS[L][2]}% DOPG)")
        ax.text(0.95, 0.92, f"{lip} {tag}", transform=ax.transAxes, style="italic",
                ha="right", va="top", fontsize=12, fontweight="bold")
        #ax.grid(alpha=0.3)
        ax.tick_params(direction="in", top=True, right=True)
        if r == len(LIPIDS) - 1:
            ax.set_xlabel("Carbon number")
        if c == 0:
            ax.set_ylabel(r"$-S_{CD}$")
handles, labs = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.05))
fig.suptitle("DOPC/DOPG acyl-chain order parameters vs composition", y=1.10)
plt.tight_layout()
plt.savefig("results/order_parameter_DOPC_DOPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route - `gmx order`
#
# For all-atom CHARMM36 the rigorous S_CD comes from the explicit C-H vectors
# (the Python above). `gmx order` was designed for united-atom chains and works
# on the carbon skeleton, so it gives the closely-related carbon order parameter
# rather than the exact explicit-H S_CD. Use it as a cross-check.
#
# It needs an index file listing the chain carbons **in order** (one group per
# chain). Build it per lipid with `gmx select`, then run `gmx order`:
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc
#
# # sn-1 carbons of DOPC, in order (C32..C318):
# gmx select -s $TPR -on sn1_DOPC.ndx \
#     -select 'resname DOPC and name C32 C33 C34 C35 C36 C37 C38 C39 C310 \
#              C311 C312 C313 C314 C315 C316 C317 C318'
#
# gmx order -s $TPR -f $XTC -n sn1_DOPC.ndx -d z \
#     -od deuter_DOPC_sn1_65.xvg -os order_DOPC_sn1_65.xvg
# ```
#
# Repeat with `name C22..C218` for sn-2, and for DOPG. `-od` writes the
# deuterium order parameter vs carbon. For error bands, run on time blocks
# (`-b/-e`) and take mean/std across the .xvg files, or use the Python blocks.

# %%  (Optional) plot straight from gmx order -od .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # carbon, -SCD

# gmx_files = {("DOPC","C3"): {"65/35": "deuter_DOPC_sn1_65.xvg"}}
# for (lip, prefix), fdict in gmx_files.items():
#     plt.figure(figsize=(5, 3.6))
#     for L in labels_sorted:
#         p = fdict.get(L)
#         if p and os.path.exists(p):
#             x, y = read_xvg_xy(p); plt.plot(x, y, color=colors[L], marker="o", ms=3, label=L)
#     plt.xlabel("Carbon number"); plt.ylabel(r"$-S_{CD}$"); plt.legend(); plt.title(f"{lip} {prefix}")
#     plt.show()

In [ ]:
# %% [markdown]
# # Lipid lateral diffusion coefficient - DOPC/DOPG composition series
#
# Lateral mean-squared displacement of each lipid's centre of mass, with
# per-leaflet COM drift removed, fitted as  MSD_xy(t) = 4 D t  ->  D = slope/4.
# Reported per lipid (DOPC, DOPG) for each composition.
#
# Plots (overlay style, one distinct colour per composition):
#   panel 1: DOPC MSD vs lag      panel 2: DOPG MSD vs lag
#   panel 3: fitted D vs DOPG %
# Error bands / error bars come from block averaging (NBLOCKS sub-trajectories).
#
# *** Trajectory to use: the NO-JUMP trajectory ***  (this is the ONLY property
# that uses nojump). Make it first, stripping water to keep it small:
#
#   gmx trjconv -s Prod_65.tpr -f <full_traj>.xtc -n mem.ndx \
#       -pbc nojump -o Prod_nojump_65.xtc            # choose the Membrane group
#
# Do NOT use the centered/whole xtc here - individual lipids still wrap around the
# box in those and the MSD will be wrong.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DOPG"]
HEAD     = "P"
NBLOCKS  = 5           # sub-trajectories for the error estimate
STEP     = 1           # frame stride
FITMIN   = 0.2         # fit window (fraction of each block's lag range)
FITMAX   = 0.6

# label -> (topology, NO-JUMP trajectory, DOPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_nojump_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_nojump_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_nojump_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_nojump_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_nojump_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#e377c2",   # pink
    "72/28": "#17becf",   # cyan
    "80/20": "#bcbd22",   # olive
    "87/13": "#7f7f7f",   # gray
    "95/5" : "#8c564b",   # brown
}
os.makedirs("results", exist_ok=True)

# %%  Diffusion core (lateral MSD, per-leaflet drift removed, block-averaged)
def _msd_curve(seg):
    # seg: (Ls, n, 2) lateral COM positions -> MSD over lags 0..Ls-1
    Ls = seg.shape[0]
    msd = np.zeros(Ls)
    for lag in range(1, Ls):
        disp = seg[lag:] - seg[:-lag]
        msd[lag] = (disp ** 2).sum(axis=2).mean()
    return msd

def diffusion_blocks(u, resname, head=HEAD, nblocks=NBLOCKS, step=STEP,
                     fitmin=FITMIN, fitmax=FITMAX):
    residues = u.select_atoms(f"resname {resname}").residues
    if len(residues) == 0:
        return None
    P = u.select_atoms(f"name {head} and resname {resname}")
    # leaflet assignment from frame 0
    u.trajectory[0]
    zc0 = P.positions[:, 2].mean()
    pz = {a.resid: a.position[2] for a in P}
    leaf = np.array([0 if pz.get(r.resid, zc0) >= zc0 else 1 for r in residues])

    # gather lateral COM per frame with per-leaflet drift removed
    times, coms = [], []
    for ts in u.trajectory[::step]:
        times.append(ts.time)
        com = np.array([r.atoms.center_of_mass()[:2] for r in residues])
        for lf in (0, 1):
            m = leaf == lf
            if m.any():
                com[m] -= com[m].mean(axis=0)
        coms.append(com)
    times = np.array(times)
    coms = np.array(coms)                      # (T, n, 2)
    T = len(times)
    dt = np.mean(np.diff(times)) if T > 1 else 1.0

    # split into contiguous blocks; MSD per block on a common lag range
    segs = [s for s in np.array_split(np.arange(T), max(1, min(nblocks, T))) if len(s) > 2]
    Lmin = min(len(s) for s in segs)
    lags = np.arange(Lmin) * dt
    msds, Ds = [], []
    i0, i1 = max(1, int(fitmin * Lmin)), int(fitmax * Lmin)
    if i1 - i0 < 2:
        i0, i1 = 1, Lmin
    for s in segs:
        m = _msd_curve(coms[s][:Lmin])
        msds.append(m)
        Ds.append(np.polyfit(lags[i0:i1], m[i0:i1], 1)[0] / 4.0 * 1e-4)  # cm^2/s
    M = np.array(msds)
    Ds = np.array(Ds)
    return {"lags": lags, "msd": M.mean(0), "msd_std": M.std(0) if len(M) > 1 else np.zeros(Lmin),
            "D": Ds.mean(), "D_err": Ds.std() if len(Ds) > 1 else 0.0}

# %%  Compute for every composition and lipid
results = {lip: {} for lip in LIPIDS}
Dtab = {}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for lip in LIPIDS:
        out = diffusion_blocks(u, lip)
        if out is None:
            continue
        results[lip][label] = out
        Dtab[(lip, label)] = (out["D"], out["D_err"])
        np.savetxt(f"results/msd_{lip}_{label.replace('/', '-')}.dat",
                   np.column_stack([out["lags"], out["msd"], out["msd_std"]]),
                   header="lag_ps  msd_A2  std")

print(f"\n{'comp':<8}" + "".join(f"{lip+' D(1e-7)':>18}" for lip in LIPIDS))
for label in SYSTEMS:
    row = ""
    for lip in LIPIDS:
        if (lip, label) in Dtab:
            D, e = Dtab[(lip, label)]
            row += f"{D*1e7:11.3f}+/-{e*1e7:5.3f}"
        else:
            row += f"{'-':>18}"
    print(f"{label:<8}{row}")
print("\n(D in cm^2/s; column shows D x 1e7)")

# %%  Plot - MSD overlays + D vs composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))
# panels 1 & 2: MSD per lipid
for ax, lip in zip(axes[:2], LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        o = results[lip][L]
        lag_k = o["lags"] / 1000.0          # ps -> thousands of ps
        ax.fill_between(lag_k, o["msd"] - o["msd_std"], o["msd"] + o["msd_std"],
                        color=colors[L], alpha=0.18, lw=0)
        ax.plot(lag_k, o["msd"], color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DOPG)")
    ax.set_xlabel(r"lag time ($\times10^{3}$ ps)")
    ax.set_ylabel(r"MSD$_{xy}$ ($\AA^2$)")
    ax.text(0.05, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="left", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)

# panel 3: D vs DOPG %
ax = axes[2]
markers = {LIPIDS[0]: "o", LIPIDS[-1]: "s"}
for lip in LIPIDS:
    xs = [SYSTEMS[L][2] for L in labels_sorted if L in results[lip]]
    ys = [Dtab[(lip, L)][0] * 1e7 for L in labels_sorted if L in results[lip]]
    es = [Dtab[(lip, L)][1] * 1e7 for L in labels_sorted if L in results[lip]]
    cs = [colors[L] for L in labels_sorted if L in results[lip]]
    ax.plot(xs, ys, color="0.6", lw=1.0, zorder=1)
    ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor="0.5", capsize=3, zorder=2)
    ax.scatter(xs, ys, c=cs, marker=markers.get(lip, "o"), s=55,
               edgecolor="k", linewidth=0.5, zorder=3, label=lip)
ax.set_xlabel("DOPG %")
ax.set_ylabel(r"$D$ (10$^{-7}$ cm$^2$/s)")
ax.text(0.05, 0.92, "D vs composition", transform=ax.transAxes,
        ha="left", va="top", fontsize=11)
ax.tick_params(direction="in", top=True, right=True)
ax.legend(title="lipid (marker)", fontsize=9)

handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("DOPC/DOPG lateral diffusion vs composition", y=1.15)
plt.tight_layout()
plt.savefig("results/diffusion_DOPC_DOPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route - `gmx msd`
#
# `gmx msd -lateral z` gives the in-plane MSD and prints D. Run on the NO-JUMP
# trajectory. Fit only the linear (diffusive) regime with `-beginfit/-endfit`.
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_nojump_65.xtc ; NDX=index.ndx
#
# # DOPC (choose the DOPC group at the prompt):
# gmx msd -s $TPR -f $XTC -n $NDX -lateral z -rmcomm \
#     -beginfit 20000 -endfit 80000 -o msd_DOPC_65.xvg
#
# # DOPG:
# gmx msd -s $TPR -f $XTC -n $NDX -lateral z -rmcomm \
#     -beginfit 20000 -endfit 80000 -o msd_DOPG_65.xvg
# ```
#
# `-lateral z` = in-plane MSD, D = slope/4 (gmx prints it). `-rmcomm` removes the
# overall COM motion; the Python version additionally removes per-leaflet drift,
# which is the more careful choice. Report the finite-size (Yeh-Hummer) caveat:
# absolute D depends on box size - compare compositions at matched box sizes.

# %%  (Optional) read D straight from gmx msd terminal output / .xvg
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # time(ps), MSD

# gmx prints "D[ DOPC] = ... (1e-5 cm^2/s)" to the terminal; you can also refit the
# .xvg yourself:
# t, msd = read_xvg_xy("msd_DOPC_65.xvg")
# i0, i1 = int(0.2*len(t)), int(0.6*len(t))
# slope = np.polyfit(t[i0:i1], msd[i0:i1], 1)[0]
# print("D =", slope/4*1e-4, "cm^2/s")

In [ ]:
# %% [markdown]
# # Voronoi analysis (area per lipid) - DOPC/DOPG composition series
#
# Periodic 2D Voronoi tessellation of the phosphate (P) atoms, done per leaflet.
# From it, per lipid type: area per lipid (APL), mean number of Voronoi
# neighbours, and the like-neighbour fraction (fraction of a lipid's neighbours
# that are the SAME type - a mixing / clustering signal).
#
# Plots (overlay style, one distinct colour per composition):
#   panel 1: DOPC APL distribution P(APL)   panel 2: DOPG APL distribution
#   panel 3: mean APL vs DOPG %
# Bands / error bars from block averaging (NBLOCKS sub-trajectories).
#
# Trajectory to use: a **whole + centered** xtc. Voronoi only needs the P atoms,
# so the small P-only `heads` trajectory is ideal (or the full one). Not nojump.
#
# Requires scipy:  pip install scipy

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy scipy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
from scipy.spatial import Voronoi
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DOPG"]
HEAD     = "P"
APL_MIN  = 30.0        # APL histogram range (A^2)
APL_MAX  = 110.0
APL_BINW = 1.5
NBLOCKS  = 5           # time blocks for the error estimate
STEP     = 1           # frame stride (raise to subsample heavy trajectories)

# label -> (topology, whole+centered trajectory, DOPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#e377c2",   # pink
    "72/28": "#17becf",   # cyan
    "80/20": "#bcbd22",   # olive
    "87/13": "#7f7f7f",   # gray
    "95/5" : "#8c564b",   # brown
}
os.makedirs("results", exist_ok=True)

# %%  Voronoi core (periodic, per leaflet)
def _poly_area(p):
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def _tile(pts, Lx, Ly):
    sh = [(i, j) for i in (-1, 0, 1) for j in (-1, 0, 1)]
    tp = np.vstack([pts + np.array([a * Lx, b * Ly]) for a, b in sh])
    tmap = np.concatenate([np.arange(len(pts)) for _ in sh])
    return tp, tmap

def _leaflet_voronoi(pts, rtypes, Lx, Ly):
    # returns per-point: area, neighbour count, like-neighbour fraction
    N = len(pts)
    tp, tmap = _tile(pts, Lx, Ly)
    vor = Voronoi(tp)
    central = np.arange(4 * N, 5 * N)          # the (0,0) image block
    areas = np.full(N, np.nan)
    for k, pi in enumerate(central):
        reg = vor.regions[vor.point_region[pi]]
        if reg and -1 not in reg:
            areas[k] = _poly_area(vor.vertices[reg])
    neigh = [set() for _ in range(N)]
    for i, j in vor.ridge_points:
        if 4 * N <= i < 5 * N:
            neigh[i - 4 * N].add(tmap[j])
        if 4 * N <= j < 5 * N:
            neigh[j - 4 * N].add(tmap[i])
    for k in range(N):
        neigh[k].discard(k)
    nn = np.array([len(neigh[k]) for k in range(N)], float)
    like = np.array([np.mean([rtypes[m] == rtypes[k] for m in neigh[k]])
                     if neigh[k] else np.nan for k in range(N)])
    return areas, nn, like

def voronoi_blocks(u, lipids=LIPIDS, head=HEAD, apl_min=APL_MIN, apl_max=APL_MAX,
                   apl_binw=APL_BINW, nblocks=NBLOCKS, step=STEP):
    P = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    rt_all = np.array([a.resname for a in P])
    edges = np.arange(apl_min, apl_max + apl_binw, apl_binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]

    hist = {lip: [] for lip in lipids}     # per-block density histogram
    aplm = {lip: [] for lip in lipids}     # per-block mean APL
    nnm = {lip: [] for lip in lipids}
    likem = {lip: [] for lip in lipids}
    for blk in blocks:
        bucket = {lip: {"apl": [], "nn": [], "like": []} for lip in lipids}
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            pos = P.positions.copy()
            pos[:, 0] %= Lx
            pos[:, 1] %= Ly
            zmid = pos[:, 2].mean()
            for mask in (pos[:, 2] >= zmid, pos[:, 2] < zmid):
                sub = pos[mask][:, :2]
                rs = rt_all[mask]
                if len(sub) < 4:
                    continue
                ar, nn, lk = _leaflet_voronoi(sub, rs, Lx, Ly)
                for lip in lipids:
                    m = rs == lip
                    bucket[lip]["apl"] += list(ar[m][~np.isnan(ar[m])])
                    bucket[lip]["nn"] += list(nn[m])
                    bucket[lip]["like"] += list(lk[m][~np.isnan(lk[m])])
        for lip in lipids:
            a = np.array(bucket[lip]["apl"])
            if a.size:
                hist[lip].append(np.histogram(a, bins=edges, density=True)[0])
                aplm[lip].append(a.mean())
                nnm[lip].append(np.mean(bucket[lip]["nn"]))
                likem[lip].append(np.nanmean(bucket[lip]["like"]))

    out = {}
    for lip in lipids:
        if not hist[lip]:
            continue
        H = np.array(hist[lip])
        out[lip] = {
            "apl": ctr, "P": H.mean(0), "Pstd": H.std(0) if len(H) > 1 else np.zeros(len(ctr)),
            "mean_apl": np.mean(aplm[lip]), "apl_err": np.std(aplm[lip]) if len(aplm[lip]) > 1 else 0.0,
            "nn": np.mean(nnm[lip]), "like": np.nanmean(likem[lip]),
        }
    return out

# %%  Compute for every composition
results = {lip: {} for lip in LIPIDS}
summary = {}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    out = voronoi_blocks(u)
    for lip in LIPIDS:
        if lip in out:
            results[lip][label] = out[lip]
            summary[(lip, label)] = out[lip]
            np.savetxt(f"results/voronoi_{lip}_{label.replace('/', '-')}.dat",
                       np.column_stack([out[lip]["apl"], out[lip]["P"], out[lip]["Pstd"]]),
                       header="APL_A2  P  std")

print(f"\n{'comp':<8}{'lipid':<7}{'APL(A^2)':>12}{'err':>7}{'<neigh>':>9}{'like':>7}")
for label in SYSTEMS:
    for lip in LIPIDS:
        if (lip, label) in summary:
            s = summary[(lip, label)]
            print(f"{label:<8}{lip:<7}{s['mean_apl']:12.2f}{s['apl_err']:7.2f}"
                  f"{s['nn']:9.2f}{s['like']:7.3f}")

# %%  Plot - APL distributions + mean APL vs composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))
for ax, lip in zip(axes[:2], LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        o = results[lip][L]
        ax.fill_between(o["apl"], o["P"] - o["Pstd"], o["P"] + o["Pstd"],
                        color=colors[L], alpha=0.18, lw=0)
        ax.plot(o["apl"], o["P"], color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DOPG)")
    ax.set_xlabel(r"Area per lipid ($\AA^2$)")
    ax.set_ylabel("P(APL)")
    ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="right", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)

ax = axes[2]
markers = {LIPIDS[0]: "o", LIPIDS[-1]: "s"}
for lip in LIPIDS:
    xs = [SYSTEMS[L][2] for L in labels_sorted if L in results[lip]]
    ys = [summary[(lip, L)]["mean_apl"] for L in labels_sorted if L in results[lip]]
    es = [summary[(lip, L)]["apl_err"] for L in labels_sorted if L in results[lip]]
    cs = [colors[L] for L in labels_sorted if L in results[lip]]
    ax.plot(xs, ys, color="0.6", lw=1.0, zorder=1)
    ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor="0.5", capsize=3, zorder=2)
    ax.scatter(xs, ys, c=cs, marker=markers.get(lip, "o"), s=55,
               edgecolor="k", linewidth=0.5, zorder=3, label=lip)
ax.set_xlabel("DOPG %")
ax.set_ylabel(r"mean APL ($\AA^2$)")
ax.text(0.05, 0.92, "APL vs composition", transform=ax.transAxes,
        ha="left", va="top", fontsize=11)
ax.tick_params(direction="in", top=True, right=True)
ax.legend(title="lipid (marker)", fontsize=9)

handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("DOPC/DOPG area per lipid vs composition", y=1.15)
plt.tight_layout()
plt.savefig("results/voronoi_DOPC_DOPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS cross-check - overall mean APL from the box
#
# GROMACS has no built-in per-lipid Voronoi tool, but the **overall** mean APL is
# just the box cross-section divided by lipids per leaflet:
#     APL_mean = 2 * Box-X * Box-Y / N_lipids
# Get the box vectors from the energy file:
#
# ```bash
# gmx energy -f Prod_65.edr -o box_65.xvg     # choose Box-X and Box-Y
# ```
#
# Then in Python:  APL = 2 * <Box-X * Box-Y> / N_lipids  (should match the
# mixture-averaged mean of the two Voronoi APLs). For per-lipid-type APL you need
# the Voronoi tessellation above, or a dedicated tool (FATSLiM, APL@Voro,
# GridMAT-MD).

# %%  (Optional) box-based mean APL from a gmx energy .xvg
def read_xvg(path):
    leg, rows = [], []
    for line in open(path):
        if line.startswith("@") and " legend " in line:
            leg.append(line.split('"')[1])
        elif not line.startswith(("#", "@")):
            rows.append([float(x) for x in line.split()])
    return leg, np.array(rows)

# leg, d = read_xvg("box_65.xvg")   # columns: time, Box-X, Box-Y (nm)
# N_lipids = 200                     # lipids per bilayer for that system
# bx, by = d[:, 1], d[:, 2]
# apl_nm2 = 2 * (bx * by).mean() / N_lipids
# print("overall mean APL = %.2f A^2" % (apl_nm2 * 100))   # nm^2 -> A^2

In [ ]:
# %% [markdown]
# # Lipid spatial distribution - DOPC/DOPG composition series
#
# Time-averaged 2D lateral density of each lipid's phosphate in the membrane
# plane, per leaflet, shown as an ENRICHMENT map: each map is divided by its own
# mean, so 1.0 = uniform, >1 = locally enriched (clustering), <1 = depleted.
#
# Unlike the other properties this one is a 2D map, so it is shown as a grid of
# heatmaps (rows = composition, columns = DOPC / DOPG) rather than overlaid
# curves. The radial (curve) version of the same information is the cross-RDF
# (DOPC-DOPG), done in the RDF script.
#
# Trajectory to use: a **whole + centered** xtc. Only the P atoms are needed, so
# the small P-only `heads` trajectory works well (or the full one). Not nojump.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DOPG"]
HEAD     = "P"
BINS     = 40           # grid resolution (BINS x BINS)
LEAFLET  = "upper"      # "upper", "lower", or "both"
SMOOTH   = 1.0          # Gaussian-ish smoothing passes (0 = none); simple box blur
STEP     = 1
VMIN, VMAX = 0.5, 1.5   # enrichment colour range (1.0 = uniform)

# label -> (topology, whole+centered trajectory, DOPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}
os.makedirs("results", exist_ok=True)

# %%  Density-map core (per leaflet, enrichment = map / mean)
def _leaf_mask(z, zmid, leaflet):
    if leaflet == "upper":
        return z >= zmid
    if leaflet == "lower":
        return z < zmid
    return np.ones(len(z), dtype=bool)          # both

def _box_blur(M, passes):
    for _ in range(int(passes)):
        M = (M
             + np.roll(M, 1, 0) + np.roll(M, -1, 0)
             + np.roll(M, 1, 1) + np.roll(M, -1, 1)) / 5.0
    return M

def density_maps(u, lipids=LIPIDS, head=HEAD, bins=BINS, leaflet=LEAFLET,
                 smooth=SMOOTH, step=STEP):
    allP = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    sel = {lip: u.select_atoms(f"name {head} and resname {lip}") for lip in lipids}
    H = {lip: np.zeros((bins, bins)) for lip in lipids}
    nframes = 0
    for ts in u.trajectory[::step]:
        Lx, Ly = ts.dimensions[0], ts.dimensions[1]
        zmid = allP.positions[:, 2].mean()
        ex = np.linspace(0, Lx, bins + 1)
        ey = np.linspace(0, Ly, bins + 1)
        for lip in lipids:
            p = sel[lip].positions
            m = _leaf_mask(p[:, 2], zmid, leaflet)
            q = p[m]
            if len(q) == 0:
                continue
            H[lip] += np.histogram2d(np.mod(q[:, 0], Lx), np.mod(q[:, 1], Ly),
                                     bins=[ex, ey])[0]
        nframes += 1
    maps = {}
    for lip in lipids:
        M = H[lip] / max(nframes, 1)
        if smooth:
            M = _box_blur(M, smooth)
        mean = M.mean()
        maps[lip] = M / mean if mean > 0 else M       # enrichment (uniform = 1)
    return maps

# %%  Compute for every composition
results = {}     # label -> {lip: enrichment_map}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    maps = density_maps(u)
    results[label] = maps
    for lip in LIPIDS:
        np.savetxt(f"results/spatial_{lip}_{label.replace('/', '-')}.dat", maps[lip])
print("done")

# %%  Plot - grid of enrichment maps (rows = composition, cols = lipid)
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DOPG -> low
present = [L for L in labels_sorted if L in results]
nrow, ncol = len(present), len(LIPIDS)
fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.0 * nrow), squeeze=False)
norm = TwoSlopeNorm(vmin=VMIN, vcenter=1.0, vmax=VMAX)
im = None
for r, L in enumerate(present):
    for c, lip in enumerate(LIPIDS):
        ax = axes[r, c]
        im = ax.imshow(results[L][lip].T, origin="lower", cmap="RdBu_r",
                       norm=norm, interpolation="bilinear", aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(lip, fontsize=12, style="italic", fontweight="bold")
        if c == 0:
            ax.set_ylabel(f"{L}\n({SYSTEMS[L][2]}% DOPG)", fontsize=10)
fig.suptitle(f"DOPC/DOPG lateral enrichment ({LEAFLET} leaflet)  -  "
             r"P density / mean", y=1.005, fontsize=12)
fig.subplots_adjust(right=0.88)
cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cb = fig.colorbar(im, cax=cax)
cb.set_label("enrichment (density / mean)")
plt.savefig("results/spatial_distribution_DOPC_DOPG_composition.png",
            dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## Notes
#
# * For a well-mixed FLUID bilayer the time-averaged map is close to uniform
#   (~1 everywhere) because lipids diffuse; strong red patches that persist =
#   real clustering/domains. If maps look flat, that is a physical result (good
#   mixing), and the cross-RDF quantifies any short-range preference.
# * `LEAFLET` switches between the two monolayers; run both to check symmetry.
# * `VMIN/VMAX` set the colour range around 1.0 (uniform). Tighten them
#   (e.g. 0.8-1.2) to make weak enrichment visible.

# %%  (Optional) representative snapshot - top view of P atoms coloured by lipid
def snapshot(u, lipids=LIPIDS, head=HEAD, leaflet=LEAFLET, frame=-1):
    u.trajectory[frame]
    allP = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    zmid = allP.positions[:, 2].mean()
    cols = {"DOPC": "#1f77b4", "DOPG": "#d62728"}
    plt.figure(figsize=(4.2, 4.0))
    for lip in lipids:
        p = u.select_atoms(f"name {head} and resname {lip}").positions
        m = _leaf_mask(p[:, 2], zmid, leaflet)
        plt.scatter(p[m, 0], p[m, 1], s=25, color=cols.get(lip, "k"),
                    edgecolor="k", linewidth=0.3, label=lip)
    plt.gca().set_aspect("equal"); plt.xlabel("x (Å)"); plt.ylabel("y (Å)")
    plt.legend(); plt.title(f"snapshot ({leaflet} leaflet)"); plt.tight_layout(); plt.show()

# example:
# u = mda.Universe(*SYSTEMS["65/35"][:2]); snapshot(u)

# %% [markdown]
# ## GROMACS route - `gmx densmap`
#
# `gmx densmap` writes the time-averaged 2D density of a group in the xy plane.
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc ; NDX=index.ndx
#
# # DOPC phosphate 2D map (choose the "DOPC_P" group, or make one with make_ndx):
# gmx densmap -s $TPR -f $XTC -n $NDX -bin 0.1 -od densmap_DOPC_65.dat
# # repeat with the DOPG P group -> densmap_DOPG_65.dat
# ```
#
# `-bin 0.1` = 0.1 nm grid. The .dat is a plain matrix you can load with
# `np.loadtxt` and imshow exactly like the maps above (divide by its mean for the
# enrichment view). `gmx densmap` averages over both leaflets and all z by
# default; for a single leaflet, restrict the input group to that leaflet's P
# atoms (leaflet assignment is dynamic, so the Python route above is easier).

# %%  (Optional) load a gmx densmap .dat and show it
# M = np.loadtxt("densmap_DOPC_65.dat")
# plt.imshow((M/M.mean()).T, origin="lower", cmap="RdBu_r",
#            norm=TwoSlopeNorm(0.5, 1.0, 1.5)); plt.colorbar(); plt.show()

In [ ]:
# %% [markdown]
# # RDF - DOPC/DLPG composition series  (VS Code interactive script)
#
# Run cell-by-cell with the "Run Cell" links above each `# %%` (VS Code's Python
# Interactive window), or run the whole file.
#
# Trajectory to use: a **whole + centered** xtc. Do NOT use the nojump xtc here -
# nojump is only for diffusion.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the VS Code terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
HEAD    = "P"
RMAX_NM = 5.0          # x-axis range (nm)
BIN_NM  = 0.02         # bin width (nm)
NBLOCKS = 5            # time blocks for the error band
STEP    = 1            # frame stride

# label -> (topology, trajectory[whole+centered], DLPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (different palette from the DOPA plot).
COLORS = {
    "65/35": "#1b9e77",   # pink
    "72/28": "#d95f02",   # cyan
    "80/20": "#7570b3",   # olive
    "87/13": "#e7298a",   # gray
    "95/5" : "#66a61e",   # brown
}

PAIRS = [("DOPC", "DOPC"), ("DLPG", "DLPG"), ("DOPC", "DLPG")]
YLIM  = {("DOPC", "DOPC"): (0.5, 1.5),
         ("DLPG", "DLPG"): (0.5, 1.5),
         ("DOPC", "DLPG"): (0.75, 1.25)}
os.makedirs("results", exist_ok=True)

# %%  RDF core (per-leaflet, block-averaged)
def _pdist2d(a_xy, b_xy, Lx, Ly, same):
    """Minimum-image lateral distances (A) between two point sets."""
    if len(a_xy) == 0 or len(b_xy) == 0:
        return np.empty(0)
    dx = a_xy[:, None, 0] - b_xy[None, :, 0]
    dy = a_xy[:, None, 1] - b_xy[None, :, 1]
    dx -= Lx * np.round(dx / Lx)
    dy -= Ly * np.round(dy / Ly)
    r = np.sqrt(dx * dx + dy * dy)
    return r[np.triu_indices(r.shape[0], 1)] if same else r.ravel()

def rdf2d_blocks(u, resA, resB, head=HEAD, rmax_nm=RMAX_NM, bin_nm=BIN_NM,
                 nblocks=NBLOCKS, step=STEP):
    same = (resA == resB)
    rmax = rmax_nm * 10.0
    nbins = int(round(rmax / (bin_nm * 10.0)))
    edges = np.linspace(0, rmax, nbins + 1)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    shell = np.pi * (edges[1:] ** 2 - edges[:-1] ** 2)
    selA = u.select_atoms(f"name {head} and resname {resA}")
    selB = u.select_atoms(f"name {head} and resname {resB}")
    allP = u.select_atoms(f"name {head}")
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    gblocks = []
    for blk in blocks:
        hist = np.zeros(nbins)
        norm = 0.0
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            area = Lx * Ly
            zmid = allP.positions[:, 2].mean()
            Apos, Bpos = selA.positions, selB.positions
            for sign in (True, False):                 # upper then lower leaflet
                a = Apos[(Apos[:, 2] >= zmid) == sign][:, :2]
                b = Bpos[(Bpos[:, 2] >= zmid) == sign][:, :2]
                hist += np.histogram(_pdist2d(a, b, Lx, Ly, same), bins=edges)[0]
                Na, Nb = len(a), len(b)
                norm += 0.5 * Na * ((Nb - 1) / area) if same else Na * (Nb / area)
        if norm > 0:
            gblocks.append(hist / (norm * shell))
    G = np.array(gblocks)
    mean = G.mean(axis=0)
    std = G.std(axis=0) if len(G) > 1 else np.zeros_like(mean)
    return ctr / 10.0, mean, std                        # r in nm

# %%  Compute for every composition and pair
results = {(a, b): {} for a, b in PAIRS}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for a, b in PAIRS:
        r, m, s = rdf2d_blocks(u, a, b)
        results[(a, b)][label] = (r, m, s)
        np.savetxt(f"results/rdf_{a}-{b}_{label.replace('/', '-')}.dat",
                   np.column_stack([r, m, s]), header="r_nm  g(r)  std")
print("done")

# %%  Plot - one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DLPG -> low
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, len(PAIRS), figsize=(4.2 * len(PAIRS), 3.6))
if len(PAIRS) == 1:
    axes = [axes]
for ax, (a, b) in zip(axes, PAIRS):
    for L in labels_sorted:
        if L not in results[(a, b)]:
            continue
        r, m, s = results[(a, b)][L]
        ax.fill_between(r, m - s, m + s, color=colors[L], alpha=0.18, lw=0)
        ax.plot(r, m, color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DLPG)")
    ax.axhline(1.0, color="0.6", lw=0.6, ls="--")
    ax.set_xlim(0, RMAX_NM)
    ax.set_ylim(*YLIM.get((a, b), (0.5, 1.5)))
    ax.set_xlabel("Distance (nm)")
    ax.text(0.95, 0.06, f"{a}-{b}", transform=ax.transAxes, style="italic",
            ha="right", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("RDF")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=n if (n := len(labels_sorted)) else 1,
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.12))
fig.suptitle("DOPC/DLPG lateral RDF (P-P) vs composition", y=1.18)
plt.tight_layout()
plt.savefig("results/rdf_DOPC_DLPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## Optional: plot straight from `gmx rdf` .xvg files
# ```bash
# D=DOPC_DLPG_65_35
# gmx rdf -s $D/md.tpr -f $D/md.xtc -n $D/index.ndx \
#     -ref "name P and resname DOPC" -sel "name P and resname DLPG" \
#     -xy -bin 0.02 -rmax 5 -o $D/rdf_DOPC-DLPG.xvg
# ```

# %%  read gmx xvg (optional)
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # r(nm), g(r)

# gmx_files = {"65/35": "rdf_DOPC-DLPG_65.xvg"}
# plt.figure(figsize=(4.5, 3.6))
# for L in labels_sorted:
#     if L in gmx_files:
#         r, g = read_xvg_xy(gmx_files[L]); plt.plot(r, g, color=colors[L], label=L)
# plt.axhline(1, ls="--", color="0.6", lw=.6); plt.xlim(0, RMAX_NM); plt.ylim(0.75, 1.25)
# plt.xlabel("Distance (nm)"); plt.ylabel("RDF"); plt.legend(); plt.show()

In [ ]:
# %% [markdown]
# # Lipid tail splay - DOPC/DLPG composition series  (VS Code interactive script)
#
# Intra-lipid splay = angle between a lipid's two acyl-chain vectors
# (sn-1: first->last C3x carbon; sn-2: first->last C2x carbon). We build the
# probability distribution P(theta) per composition, per lipid type, with an
# error band from block averaging.
#
# Trajectory to use: the **full-atom, whole + centered** xtc (e.g.
# `Prod_merged_center_*.xtc`). NOT the P-only `heads` trajectory (splay needs the
# chain carbons) and NOT the nojump one (that's diffusion only).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the VS Code terminal
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DLPG"]    # the two lipids -> one panel each
BINW_DEG = 2.0                 # splay-angle bin width (degrees)
ANG_MAX  = 180.0               # angle axis range
NBLOCKS  = 5                   # time blocks for the error band
STEP     = 1                   # frame stride

# label -> (topology, full-atom trajectory[whole+centered], DLPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (different palette from the DOPA plots).
COLORS = {
    "65/35": "#1b9e77",   # pink
    "72/28": "#d95f02",   # cyan
    "80/20": "#7570b3",   # olive
    "87/13": "#e7298a",   # gray
    "95/5" : "#66a61e",   # brown
}
os.makedirs("results", exist_ok=True)

# %%  Splay core (block-averaged distribution)
def _carbons(res, prefix):
    """Acyl carbons of a chain ('C3'=sn-1, 'C2'=sn-2), skipping carbonyl idx 1."""
    pat = re.compile(r"^" + prefix + r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in res.atoms
            if (m := pat.match(at.name)) and int(m.group(1)) >= 2]
    return sorted(hits, key=lambda t: t[0])

def _splay_indices(u, resname):
    """(N,4) atom indices: sn1_first, sn1_last, sn2_first, sn2_last per lipid."""
    idx = []
    for res in u.select_atoms(f"resname {resname}").residues:
        c3 = _carbons(res, "C3")      # sn-1
        c2 = _carbons(res, "C2")      # sn-2
        if len(c3) >= 2 and len(c2) >= 2:
            idx.append((c3[0][1].index, c3[-1][1].index,
                        c2[0][1].index, c2[-1][1].index))
    return np.array(idx, dtype=int)

def splay_blocks(u, resname, binw=BINW_DEG, angmax=ANG_MAX,
                 nblocks=NBLOCKS, step=STEP):
    idx = _splay_indices(u, resname)
    edges = np.arange(0.0, angmax + binw, binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    if len(idx) == 0:
        return ctr, np.zeros_like(ctr), np.zeros_like(ctr), np.nan
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    Hs, all_ang = [], []
    for blk in blocks:
        ang = []
        for fi in blk:
            u.trajectory[int(fi)]
            pos = u.atoms.positions
            v1 = pos[idx[:, 1]] - pos[idx[:, 0]]      # sn-1 vector
            v2 = pos[idx[:, 3]] - pos[idx[:, 2]]      # sn-2 vector
            n1 = np.linalg.norm(v1, axis=1)
            n2 = np.linalg.norm(v2, axis=1)
            good = (n1 > 0) & (n2 > 0)
            cos = np.einsum("ij,ij->i", v1, v2) / (n1 * n2)
            ang.append(np.degrees(np.arccos(np.clip(cos[good], -1, 1))))
        ang = np.concatenate(ang)
        all_ang.append(ang)
        Hs.append(np.histogram(ang, bins=edges, density=True)[0])
    H = np.array(Hs)
    mean = H.mean(axis=0)
    std = H.std(axis=0) if len(H) > 1 else np.zeros_like(mean)
    mean_angle = np.concatenate(all_ang).mean()
    return ctr, mean, std, mean_angle

# %%  Compute for every composition and lipid
results = {lip: {} for lip in LIPIDS}   # lip -> {label: (theta, P, Pstd)}
means = {}                              # (lip,label) -> mean splay angle
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for lip in LIPIDS:
        theta, P, Pstd, mang = splay_blocks(u, lip)
        results[lip][label] = (theta, P, Pstd)
        means[(lip, label)] = mang
        np.savetxt(f"results/splay_{lip}_{label.replace('/', '-')}.dat",
                   np.column_stack([theta, P, Pstd]), header="angle_deg  P  std")

print("\nMean splay angle (deg):")
print(f"{'comp':<8}" + "".join(f"{lip:>10}" for lip in LIPIDS))
for label in SYSTEMS:
    row = "".join(f"{means.get((lip,label), float('nan')):10.1f}" for lip in LIPIDS)
    print(f"{label:<8}{row}")

# %%  Plot - one panel per lipid, one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DLPG -> low
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, len(LIPIDS), figsize=(4.6 * len(LIPIDS), 3.8),
                         sharex=True)
if len(LIPIDS) == 1:
    axes = [axes]
for ax, lip in zip(axes, LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        theta, P, Pstd = results[lip][L]
        ax.fill_between(theta, P - Pstd, P + Pstd, color=colors[L], alpha=0.18, lw=0)
        ax.plot(theta, P, color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DLPG)")
    ax.set_xlim(0, ANG_MAX)
    ax.set_xlabel("Tail splay angle (deg)")
    ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="right", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("P(θ)")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.10))
fig.suptitle("DOPC/DLPG intra-lipid tail splay vs composition", y=1.16)
plt.tight_layout()
plt.savefig("results/splay_DOPC_DLPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route (equivalent) - `gmx gangle`
#
# Splay = angle between the sn-1 and sn-2 chain vectors. `gmx gangle -g1 vector
# -g2 vector` pairs vector *i* of group1 with vector *i* of group2. Each vector is
# a pair of atoms (chain first & last carbon), so the selections below give one
# vector per lipid. DOPC is 18:1 (C318/C218); DLPG is 12:0 (C312/C212).
#
# ```bash
# # one composition (repeat for 65,72,80,87,95), one lipid at a time:
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc
#
# # DOPC splay
# gmx gangle -s $TPR -f $XTC -g1 vector -g2 vector \
#     -group1 'name C32 C318 and resname DOPC' \
#     -group2 'name C22 C218 and resname DOPC' \
#     -oh splay_DOPC_65.xvg -binw 2
#
# # DLPG splay
# gmx gangle -s $TPR -f $XTC -g1 vector -g2 vector \
#     -group1 'name C32 C312 and resname DLPG' \
#     -group2 'name C22 C212 and resname DLPG' \
#     -oh splay_DLPG_65.xvg -binw 2
# ```
#
# `-oh` writes the normalized angle histogram P(theta). For the error band, run on
# time blocks (`-b/-e`) and take mean/std across the .xvg files, or use the Python
# block averaging above.

# %%  (Optional) plot straight from gmx gangle -oh .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # angle(deg), P

# gmx_files = {"DOPC": {"65/35": "splay_DOPC_65.xvg"}, "DLPG": {"65/35": "splay_DLPG_65.xvg"}}
# fig, axes = plt.subplots(1, len(LIPIDS), figsize=(4.6*len(LIPIDS), 3.8))
# for ax, lip in zip(np.atleast_1d(axes), LIPIDS):
#     for L in labels_sorted:
#         p = gmx_files.get(lip, {}).get(L)
#         if p and os.path.exists(p):
#             x, y = read_xvg_xy(p); ax.plot(x, y, color=colors[L], lw=1.8, label=L)
#     ax.set_xlabel("Tail splay angle (deg)"); ax.set_xlim(0, ANG_MAX)
#     ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic", ha="right", va="top")
# plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# %% [markdown]
# # Water depth / penetration - DOPC/DLPG composition series
#
# Symmetrised water density profile across the bilayer normal: water density
# (relative to bulk) vs z from the membrane centre, overlaid for the five
# compositions with a block-averaged error band. Deeper penetration = the curve
# stays high closer to z = 0.
#
# Also prints a table per composition: half-density depth (|z| where water falls
# to 0.5 of bulk), mean phosphate position (headgroup plane), and mean # waters in
# the hydrophobic core (|z| < core).
#
# Trajectory to use: the full **whole + centered** xtc that still contains water
# (`Prod_merged_center_*.xtc`). Not the water-stripped mem/heads trajectories, and
# not nojump (diffusion only). Centering matters here (profile is from the centre).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPID_SEL = "resname DOPC DLPG"          # membrane (defines the centre)
WATER_SEL = "resname TIP3 SOL and (name OH2 or name OW or name O)"  # water oxygen
HEAD      = "P"
ZMAX_NM   = 4.0     # profile half-range from centre (nm)
BINW_NM   = 0.05    # bin width (nm)
CORE_NM   = 1.0     # |z| < CORE counts as hydrophobic-core water
NBLOCKS   = 5       # time blocks for the error band
STEP      = 1

# label -> (topology, full trajectory[whole+centered, WITH water], DLPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#1b9e77",   # pink
    "72/28": "#d95f02",   # cyan
    "80/20": "#7570b3",   # olive
    "87/13": "#e7298a",   # gray
    "95/5" : "#66a61e",   # brown
}
os.makedirs("results", exist_ok=True)

# %%  Water-profile core (block-averaged, symmetrised)
def water_profile_blocks(u, lipid_sel=LIPID_SEL, water_sel=WATER_SEL, head=HEAD,
                         zmax_nm=ZMAX_NM, binw_nm=BINW_NM, core_nm=CORE_NM,
                         nblocks=NBLOCKS, step=STEP):
    lip = u.select_atoms(lipid_sel)
    wat = u.select_atoms(water_sel)
    P = u.select_atoms(f"name {head}")
    zmax, binw, core = zmax_nm * 10, binw_nm * 10, core_nm * 10
    edges = np.arange(-zmax, zmax + binw, binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    nb_bins = len(ctr)
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]

    prof_blocks = []
    pzs, core_counts = [], []
    for blk in blocks:
        hw = np.zeros(nb_bins)
        areas = []
        nf = 0
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            zc = lip.center_of_mass()[2]
            zw = wat.positions[:, 2] - zc
            hw += np.histogram(zw, bins=edges)[0]
            areas.append(Lx * Ly)
            nf += 1
            pzs.append(np.abs(P.positions[:, 2] - zc).mean())
            core_counts.append(int(np.sum(np.abs(zw) < core)))
        vol = np.mean(areas) * binw                 # A^3 per bin
        dens = hw / nf / vol                          # atoms / A^3
        dens = 0.5 * (dens + dens[::-1])              # symmetrise about centre
        bulk = dens[np.abs(ctr) > (zmax - 10)].mean() # outer 1 nm = bulk
        prof_blocks.append(dens / bulk if bulk > 0 else dens)

    G = np.array(prof_blocks)
    mean = G.mean(axis=0)
    std = G.std(axis=0) if len(G) > 1 else np.zeros_like(mean)

    # half-density depth: |z| where the mean profile drops to 0.5 (going inward)
    zpos = ctr[ctr >= 0]
    mpos = mean[ctr >= 0]
    order = np.argsort(zpos)
    zpos, mpos = zpos[order], mpos[order]
    z_half = np.nan
    for i in range(len(zpos) - 1, 0, -1):
        if mpos[i] < 0.5 <= mpos[i - 1]:
            z_half = np.interp(0.5, [mpos[i], mpos[i - 1]], [zpos[i], zpos[i - 1]])
            break
    return (ctr / 10.0, mean, std, np.mean(pzs) / 10.0,
            (z_half / 10.0 if z_half == z_half else np.nan), np.mean(core_counts))

# %%  Compute for every composition
results = {}   # label -> (z, dens, std)
summary = {}   # label -> (phosphate_nm, half_depth_nm, core_waters)
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    z, d, s, pz, zhalf, ncore = water_profile_blocks(u)
    results[label] = (z, d, s)
    summary[label] = (pz, zhalf, ncore)
    np.savetxt(f"results/water_{label.replace('/', '-')}.dat",
               np.column_stack([z, d, s]), header="z_nm  water_rel_density  std")

print(f"\n{'comp':<8}{'P plane(nm)':>13}{'half-depth(nm)':>16}{'core waters':>13}")
for label in SYSTEMS:
    if label in summary:
        pz, zh, nc = summary[label]
        print(f"{label:<8}{pz:13.2f}{zh:16.2f}{nc:13.1f}")

# %%  Plot - water penetration, one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, ax = plt.subplots(figsize=(6.2, 4.2))
for L in labels_sorted:
    if L not in results:
        continue
    z, d, s = results[L]
    ax.fill_between(z, d - s, d + s, color=colors[L], alpha=0.18, lw=0)
    ax.plot(z, d, color=colors[L], lw=1.8, label=f"{L}  ({SYSTEMS[L][2]}% DLPG)")
# mark the mean phosphate plane
if summary:
    pmean = np.nanmean([summary[L][0] for L in summary])
    for xline in (pmean, -pmean):
        ax.axvline(xline, color="0.5", lw=0.8, ls=":")
    ax.text(pmean, 1.02, "P", color="0.4", ha="center", va="bottom", fontsize=10)
ax.axhline(0.5, color="0.8", lw=0.6, ls="--")
ax.set_xlim(-ZMAX_NM, ZMAX_NM)
ax.set_ylim(0, 1.15)
ax.set_xlabel("z from bilayer centre (nm)")
ax.set_ylabel("Water density (rel. to bulk)")
ax.text(0.03, 0.92, "Water", transform=ax.transAxes, style="italic",
        ha="left", va="top", fontsize=13, fontweight="bold")
ax.tick_params(direction="in", top=True, right=True)
ax.legend(loc="upper center", ncol=3, frameon=True, fontsize=8,
          bbox_to_anchor=(0.5, 1.18))
fig.suptitle("DOPC/DLPG water penetration vs composition", y=1.06)
plt.tight_layout()
plt.savefig("results/water_depth_DOPC_DLPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route (equivalent) - `gmx density`
#
# ```bash
# # one composition (repeat for 65,72,80,87,95):
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc ; NDX=index.ndx
#
# # centering group = Membrane , output group = Water
# printf "Membrane\nWater\n" | gmx density -s $TPR -f $XTC -n $NDX \
#     -d Z -sl 400 -center -symm -dens number -o water_density_65.xvg
#
# # phosphate plane for reference (output group = the P atoms):
# printf "Membrane\nPhosphate\n" | gmx density -s $TPR -f $XTC -n $NDX \
#     -d Z -sl 400 -center -symm -dens number -o phos_density_65.xvg
# ```
#
# `-center -symm` centres on the membrane and symmetrises; `-dens number` gives a
# number density. For error bands, run on time blocks (`-b/-e`) and take mean/std
# across the .xvg files, or use the Python block averaging above.

# %%  (Optional) plot straight from gmx density .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # z(nm), density

# gmx_files = {"65/35": "water_density_65.xvg"}
# plt.figure(figsize=(6.2, 4.2))
# for L in labels_sorted:
#     if L in gmx_files and os.path.exists(gmx_files[L]):
#         z, dw = read_xvg_xy(gmx_files[L])
#         plt.plot(z, dw / np.nanmax(dw), color=colors[L], lw=1.8, label=L)
# plt.xlabel("z (nm)"); plt.ylabel("water density (norm.)"); plt.legend(); plt.show()

In [ ]:
# %% [markdown]
# # Lipid tail order parameter (-S_CD) - DOPC/DLPG composition series
#
# Deuterium order parameter per acyl-chain carbon:
#   S_CD = < (3 cos^2(theta_CH) - 1) / 2 >    (theta_CH = C-H vs bilayer normal z)
# reported as -S_CD (the usual positive quantity). For each CH2 the two C-H
# values are averaged; C-H bonds are found automatically from the topology, so
# the cis double bond at C9-C10 in the oleoyl chains shows its dip on its own.
#
# One panel per (lipid x chain): DOPC/DLPG x sn-1/sn-2, each overlaying the five
# compositions with a block-averaged error band - same style as the RDF/splay
# figures.
#
# Trajectory to use: the full-atom **whole (+centered)** xtc WITH hydrogens
# (`Prod_merged_center_*.xtc`), plus the `.tpr` (needed for C-H bonds). Not the
# P-only `heads` trajectory, and not nojump (that is diffusion only).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import re
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS  = ["DOPC", "DLPG"]              # one row of panels each
CHAINS  = [("C3", "sn-1"), ("C2", "sn-2")]   # CHARMM36: sn-1=C3x, sn-2=C2x
NBLOCKS = 5                            # time blocks for the error band
STEP    = 1                            # frame stride

# label -> (topology[.tpr], full-atom trajectory[whole+centered], DLPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#1b9e77",   # pink
    "72/28": "#d95f02",   # cyan
    "80/20": "#7570b3",   # olive
    "87/13": "#e7298a",   # gray
    "95/5" : "#66a61e",   # brown
}
os.makedirs("results", exist_ok=True)

# %%  Order-parameter core (block-averaged, auto C-H detection)
def _carbons(res, prefix):
    # acyl carbons of a chain ('C3'=sn-1, 'C2'=sn-2), skipping carbonyl idx 1
    pat = re.compile(r"^" + prefix + r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in res.atoms
            if (m := pat.match(at.name)) and int(m.group(1)) >= 2]
    return sorted(hits, key=lambda t: t[0])

def _build_pairs(u, resname, prefix, has_bonds):
    # returns (sorted carbon numbers, {k: (C_index_array, H_index_array)})
    C, H = defaultdict(list), defaultdict(list)
    for res in u.select_atoms(f"resname {resname}").residues:
        for k, catom in _carbons(res, prefix):
            if has_bonds:
                hs = [a for a in catom.bonded_atoms if a.name.startswith("H")]
            else:
                hs = list(u.select_atoms("name H* and around 1.3 group c",
                                         c=u.atoms[[catom.index]]))
            for h in hs:
                C[k].append(catom.index); H[k].append(h.index)
    ks = sorted(C)
    return ks, {k: (np.array(C[k]), np.array(H[k])) for k in ks}

def scd_blocks(u, resname, prefix, has_bonds, nblocks=NBLOCKS, step=STEP):
    ks, pairs = _build_pairs(u, resname, prefix, has_bonds)
    if not ks:
        return np.array([]), np.array([]), np.array([])
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    block_means = []
    for blk in blocks:
        acc = {k: [] for k in ks}
        for fi in blk:
            u.trajectory[int(fi)]
            pos = u.atoms.positions
            for k in ks:
                Ci, Hi = pairs[k]
                v = pos[Hi] - pos[Ci]
                n = np.linalg.norm(v, axis=1)
                cosz = v[:, 2] / n
                acc[k].append(np.mean(0.5 * (3 * cosz ** 2 - 1)))
        block_means.append([-np.mean(acc[k]) for k in ks])   # -S_CD
    B = np.array(block_means)
    return np.array(ks), B.mean(axis=0), (B.std(axis=0) if len(B) > 1 else np.zeros(len(ks)))

# %%  Compute for every composition, lipid and chain
results = {(lip, prefix): {} for lip in LIPIDS for prefix, _ in CHAINS}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    has_bonds = len(getattr(u, "bonds", [])) > 0
    print(f"{label}: {len(u.trajectory)} frames, bonds={has_bonds}")
    for lip in LIPIDS:
        for prefix, tag in CHAINS:
            ks, scd, err = scd_blocks(u, lip, prefix, has_bonds)
            if ks.size == 0:
                continue
            results[(lip, prefix)][label] = (ks, scd, err)
            np.savetxt(f"results/scd_{lip}_{tag}_{label.replace('/', '-')}.dat",
                       np.column_stack([ks, scd, err]), header="carbon  -SCD  std")
print("done")

# %%  Plot - 2x2 grid (lipid x chain), one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(len(LIPIDS), len(CHAINS),
                         figsize=(4.4 * len(CHAINS), 3.4 * len(LIPIDS)),
                         sharex=True, sharey=True)
axes = np.atleast_2d(axes)
for r, lip in enumerate(LIPIDS):
    for c, (prefix, tag) in enumerate(CHAINS):
        ax = axes[r, c]
        for L in labels_sorted:
            if L not in results[(lip, prefix)]:
                continue
            ks, scd, err = results[(lip, prefix)][L]
            ax.fill_between(ks, scd - err, scd + err, color=colors[L], alpha=0.18, lw=0)
            ax.plot(ks, scd, color=colors[L], lw=1.8, marker="o", ms=3,
                    label=f"{L}  ({SYSTEMS[L][2]}% DLPG)")
        ax.text(0.95, 0.92, f"{lip} {tag}", transform=ax.transAxes, style="italic",
                ha="right", va="top", fontsize=12, fontweight="bold")
        ax.grid(alpha=0.3)
        ax.tick_params(direction="in", top=True, right=True)
        if r == len(LIPIDS) - 1:
            ax.set_xlabel("Carbon number")
        if c == 0:
            ax.set_ylabel(r"$-S_{CD}$")
handles, labs = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.05))
fig.suptitle("DOPC/DLPG acyl-chain order parameters vs composition", y=1.10)
plt.tight_layout()
plt.savefig("results/order_parameter_DOPC_DLPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route - `gmx order`
#
# For all-atom CHARMM36 the rigorous S_CD comes from the explicit C-H vectors
# (the Python above). `gmx order` was designed for united-atom chains and works
# on the carbon skeleton, so it gives the closely-related carbon order parameter
# rather than the exact explicit-H S_CD. Use it as a cross-check.
#
# It needs an index file listing the chain carbons **in order** (one group per
# chain). Build it per lipid with `gmx select`, then run `gmx order`:
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc
#
# # sn-1 carbons of DOPC, in order (C32..C318):
# gmx select -s $TPR -on sn1_DOPC.ndx \
#     -select 'resname DOPC and name C32 C33 C34 C35 C36 C37 C38 C39 C310 \
#              C311 C312 C313 C314 C315 C316 C317 C318'
#
# gmx order -s $TPR -f $XTC -n sn1_DOPC.ndx -d z \
#     -od deuter_DOPC_sn1_65.xvg -os order_DOPC_sn1_65.xvg
# ```
#
# Repeat with `name C22..C218` for sn-2, and for DLPG (12:0, so C32..C312 for sn-1 / C22..C212 for sn-2). `-od` writes the
# deuterium order parameter vs carbon. For error bands, run on time blocks
# (`-b/-e`) and take mean/std across the .xvg files, or use the Python blocks.

# %%  (Optional) plot straight from gmx order -od .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # carbon, -SCD

# gmx_files = {("DOPC","C3"): {"65/35": "deuter_DOPC_sn1_65.xvg"}}
# for (lip, prefix), fdict in gmx_files.items():
#     plt.figure(figsize=(5, 3.6))
#     for L in labels_sorted:
#         p = fdict.get(L)
#         if p and os.path.exists(p):
#             x, y = read_xvg_xy(p); plt.plot(x, y, color=colors[L], marker="o", ms=3, label=L)
#     plt.xlabel("Carbon number"); plt.ylabel(r"$-S_{CD}$"); plt.legend(); plt.title(f"{lip} {prefix}")
#     plt.show()

In [ ]:
# %% [markdown]
# # Lipid lateral diffusion coefficient - DOPC/DLPG composition series
#
# Lateral mean-squared displacement of each lipid's centre of mass, with
# per-leaflet COM drift removed, fitted as  MSD_xy(t) = 4 D t  ->  D = slope/4.
# Reported per lipid (DOPC, DLPG) for each composition.
#
# Plots (overlay style, one distinct colour per composition):
#   panel 1: DOPC MSD vs lag      panel 2: DLPG MSD vs lag
#   panel 3: fitted D vs DLPG %
# Error bands / error bars come from block averaging (NBLOCKS sub-trajectories).
#
# *** Trajectory to use: the NO-JUMP trajectory ***  (this is the ONLY property
# that uses nojump). Make it first, stripping water to keep it small:
#
#   gmx trjconv -s Prod_65.tpr -f <full_traj>.xtc -n mem.ndx \
#       -pbc nojump -o Prod_nojump_65.xtc            # choose the Membrane group
#
# Do NOT use the centered/whole xtc here - individual lipids still wrap around the
# box in those and the MSD will be wrong.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DLPG"]
HEAD     = "P"
NBLOCKS  = 5           # sub-trajectories for the error estimate
STEP     = 1           # frame stride
FITMIN   = 0.2         # fit window (fraction of each block's lag range)
FITMAX   = 0.6

# label -> (topology, NO-JUMP trajectory, DLPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_nojump_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_nojump_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_nojump_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_nojump_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_nojump_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#1b9e77",   # pink
    "72/28": "#d95f02",   # cyan
    "80/20": "#7570b3",   # olive
    "87/13": "#e7298a",   # gray
    "95/5" : "#66a61e",   # brown
}
os.makedirs("results", exist_ok=True)

# %%  Diffusion core (lateral MSD, per-leaflet drift removed, block-averaged)
def _msd_curve(seg):
    # seg: (Ls, n, 2) lateral COM positions -> MSD over lags 0..Ls-1
    Ls = seg.shape[0]
    msd = np.zeros(Ls)
    for lag in range(1, Ls):
        disp = seg[lag:] - seg[:-lag]
        msd[lag] = (disp ** 2).sum(axis=2).mean()
    return msd

def diffusion_blocks(u, resname, head=HEAD, nblocks=NBLOCKS, step=STEP,
                     fitmin=FITMIN, fitmax=FITMAX):
    residues = u.select_atoms(f"resname {resname}").residues
    if len(residues) == 0:
        return None
    P = u.select_atoms(f"name {head} and resname {resname}")
    # leaflet assignment from frame 0
    u.trajectory[0]
    zc0 = P.positions[:, 2].mean()
    pz = {a.resid: a.position[2] for a in P}
    leaf = np.array([0 if pz.get(r.resid, zc0) >= zc0 else 1 for r in residues])

    # gather lateral COM per frame with per-leaflet drift removed
    times, coms = [], []
    for ts in u.trajectory[::step]:
        times.append(ts.time)
        com = np.array([r.atoms.center_of_mass()[:2] for r in residues])
        for lf in (0, 1):
            m = leaf == lf
            if m.any():
                com[m] -= com[m].mean(axis=0)
        coms.append(com)
    times = np.array(times)
    coms = np.array(coms)                      # (T, n, 2)
    T = len(times)
    dt = np.mean(np.diff(times)) if T > 1 else 1.0

    # split into contiguous blocks; MSD per block on a common lag range
    segs = [s for s in np.array_split(np.arange(T), max(1, min(nblocks, T))) if len(s) > 2]
    Lmin = min(len(s) for s in segs)
    lags = np.arange(Lmin) * dt
    msds, Ds = [], []
    i0, i1 = max(1, int(fitmin * Lmin)), int(fitmax * Lmin)
    if i1 - i0 < 2:
        i0, i1 = 1, Lmin
    for s in segs:
        m = _msd_curve(coms[s][:Lmin])
        msds.append(m)
        Ds.append(np.polyfit(lags[i0:i1], m[i0:i1], 1)[0] / 4.0 * 1e-4)  # cm^2/s
    M = np.array(msds)
    Ds = np.array(Ds)
    return {"lags": lags, "msd": M.mean(0), "msd_std": M.std(0) if len(M) > 1 else np.zeros(Lmin),
            "D": Ds.mean(), "D_err": Ds.std() if len(Ds) > 1 else 0.0}

# %%  Compute for every composition and lipid
results = {lip: {} for lip in LIPIDS}
Dtab = {}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for lip in LIPIDS:
        out = diffusion_blocks(u, lip)
        if out is None:
            continue
        results[lip][label] = out
        Dtab[(lip, label)] = (out["D"], out["D_err"])
        np.savetxt(f"results/msd_{lip}_{label.replace('/', '-')}.dat",
                   np.column_stack([out["lags"], out["msd"], out["msd_std"]]),
                   header="lag_ps  msd_A2  std")

print(f"\n{'comp':<8}" + "".join(f"{lip+' D(1e-7)':>18}" for lip in LIPIDS))
for label in SYSTEMS:
    row = ""
    for lip in LIPIDS:
        if (lip, label) in Dtab:
            D, e = Dtab[(lip, label)]
            row += f"{D*1e7:11.3f}+/-{e*1e7:5.3f}"
        else:
            row += f"{'-':>18}"
    print(f"{label:<8}{row}")
print("\n(D in cm^2/s; column shows D x 1e7)")

# %%  Plot - MSD overlays + D vs composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))
# panels 1 & 2: MSD per lipid
for ax, lip in zip(axes[:2], LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        o = results[lip][L]
        lag_k = o["lags"] / 1000.0          # ps -> thousands of ps
        ax.fill_between(lag_k, o["msd"] - o["msd_std"], o["msd"] + o["msd_std"],
                        color=colors[L], alpha=0.18, lw=0)
        ax.plot(lag_k, o["msd"], color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DLPG)")
    ax.set_xlabel(r"lag time ($\times10^{3}$ ps)")
    ax.set_ylabel(r"MSD$_{xy}$ ($\AA^2$)")
    ax.text(0.05, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="left", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)

# panel 3: D vs DLPG %
ax = axes[2]
markers = {LIPIDS[0]: "o", LIPIDS[-1]: "s"}
for lip in LIPIDS:
    xs = [SYSTEMS[L][2] for L in labels_sorted if L in results[lip]]
    ys = [Dtab[(lip, L)][0] * 1e7 for L in labels_sorted if L in results[lip]]
    es = [Dtab[(lip, L)][1] * 1e7 for L in labels_sorted if L in results[lip]]
    cs = [colors[L] for L in labels_sorted if L in results[lip]]
    ax.plot(xs, ys, color="0.6", lw=1.0, zorder=1)
    ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor="0.5", capsize=3, zorder=2)
    ax.scatter(xs, ys, c=cs, marker=markers.get(lip, "o"), s=55,
               edgecolor="k", linewidth=0.5, zorder=3, label=lip)
ax.set_xlabel("DLPG %")
ax.set_ylabel(r"$D$ (10$^{-7}$ cm$^2$/s)")
ax.text(0.05, 0.92, "D vs composition", transform=ax.transAxes,
        ha="left", va="top", fontsize=11)
ax.tick_params(direction="in", top=True, right=True)
ax.legend(title="lipid (marker)", fontsize=9)

handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("DOPC/DLPG lateral diffusion vs composition", y=1.15)
plt.tight_layout()
plt.savefig("results/diffusion_DOPC_DLPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route - `gmx msd`
#
# `gmx msd -lateral z` gives the in-plane MSD and prints D. Run on the NO-JUMP
# trajectory. Fit only the linear (diffusive) regime with `-beginfit/-endfit`.
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_nojump_65.xtc ; NDX=index.ndx
#
# # DOPC (choose the DOPC group at the prompt):
# gmx msd -s $TPR -f $XTC -n $NDX -lateral z -rmcomm \
#     -beginfit 20000 -endfit 80000 -o msd_DOPC_65.xvg
#
# # DLPG:
# gmx msd -s $TPR -f $XTC -n $NDX -lateral z -rmcomm \
#     -beginfit 20000 -endfit 80000 -o msd_DLPG_65.xvg
# ```
#
# `-lateral z` = in-plane MSD, D = slope/4 (gmx prints it). `-rmcomm` removes the
# overall COM motion; the Python version additionally removes per-leaflet drift,
# which is the more careful choice. Report the finite-size (Yeh-Hummer) caveat:
# absolute D depends on box size - compare compositions at matched box sizes.

# %%  (Optional) read D straight from gmx msd terminal output / .xvg
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # time(ps), MSD

# gmx prints "D[ DOPC] = ... (1e-5 cm^2/s)" to the terminal; you can also refit the
# .xvg yourself:
# t, msd = read_xvg_xy("msd_DOPC_65.xvg")
# i0, i1 = int(0.2*len(t)), int(0.6*len(t))
# slope = np.polyfit(t[i0:i1], msd[i0:i1], 1)[0]
# print("D =", slope/4*1e-4, "cm^2/s")

In [ ]:
# %% [markdown]
# # Voronoi analysis (area per lipid) - DOPC/DLPG composition series
#
# Periodic 2D Voronoi tessellation of the phosphate (P) atoms, done per leaflet.
# From it, per lipid type: area per lipid (APL), mean number of Voronoi
# neighbours, and the like-neighbour fraction (fraction of a lipid's neighbours
# that are the SAME type - a mixing / clustering signal).
#
# Plots (overlay style, one distinct colour per composition):
#   panel 1: DOPC APL distribution P(APL)   panel 2: DLPG APL distribution
#   panel 3: mean APL vs DLPG %
# Bands / error bars from block averaging (NBLOCKS sub-trajectories).
#
# Trajectory to use: a **whole + centered** xtc. Voronoi only needs the P atoms,
# so the small P-only `heads` trajectory is ideal (or the full one). Not nojump.
#
# Requires scipy:  pip install scipy

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy scipy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
from scipy.spatial import Voronoi
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DLPG"]
HEAD     = "P"
APL_MIN  = 30.0        # APL histogram range (A^2)
APL_MAX  = 110.0
APL_BINW = 1.5
NBLOCKS  = 5           # time blocks for the error estimate
STEP     = 1           # frame stride (raise to subsample heavy trajectories)

# label -> (topology, whole+centered trajectory, DLPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#1b9e77",   # pink
    "72/28": "#d95f02",   # cyan
    "80/20": "#7570b3",   # olive
    "87/13": "#e7298a",   # gray
    "95/5" : "#66a61e",   # brown
}
os.makedirs("results", exist_ok=True)

# %%  Voronoi core (periodic, per leaflet)
def _poly_area(p):
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def _tile(pts, Lx, Ly):
    sh = [(i, j) for i in (-1, 0, 1) for j in (-1, 0, 1)]
    tp = np.vstack([pts + np.array([a * Lx, b * Ly]) for a, b in sh])
    tmap = np.concatenate([np.arange(len(pts)) for _ in sh])
    return tp, tmap

def _leaflet_voronoi(pts, rtypes, Lx, Ly):
    # returns per-point: area, neighbour count, like-neighbour fraction
    N = len(pts)
    tp, tmap = _tile(pts, Lx, Ly)
    vor = Voronoi(tp)
    central = np.arange(4 * N, 5 * N)          # the (0,0) image block
    areas = np.full(N, np.nan)
    for k, pi in enumerate(central):
        reg = vor.regions[vor.point_region[pi]]
        if reg and -1 not in reg:
            areas[k] = _poly_area(vor.vertices[reg])
    neigh = [set() for _ in range(N)]
    for i, j in vor.ridge_points:
        if 4 * N <= i < 5 * N:
            neigh[i - 4 * N].add(tmap[j])
        if 4 * N <= j < 5 * N:
            neigh[j - 4 * N].add(tmap[i])
    for k in range(N):
        neigh[k].discard(k)
    nn = np.array([len(neigh[k]) for k in range(N)], float)
    like = np.array([np.mean([rtypes[m] == rtypes[k] for m in neigh[k]])
                     if neigh[k] else np.nan for k in range(N)])
    return areas, nn, like

def voronoi_blocks(u, lipids=LIPIDS, head=HEAD, apl_min=APL_MIN, apl_max=APL_MAX,
                   apl_binw=APL_BINW, nblocks=NBLOCKS, step=STEP):
    P = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    rt_all = np.array([a.resname for a in P])
    edges = np.arange(apl_min, apl_max + apl_binw, apl_binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]

    hist = {lip: [] for lip in lipids}     # per-block density histogram
    aplm = {lip: [] for lip in lipids}     # per-block mean APL
    nnm = {lip: [] for lip in lipids}
    likem = {lip: [] for lip in lipids}
    for blk in blocks:
        bucket = {lip: {"apl": [], "nn": [], "like": []} for lip in lipids}
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            pos = P.positions.copy()
            pos[:, 0] %= Lx
            pos[:, 1] %= Ly
            zmid = pos[:, 2].mean()
            for mask in (pos[:, 2] >= zmid, pos[:, 2] < zmid):
                sub = pos[mask][:, :2]
                rs = rt_all[mask]
                if len(sub) < 4:
                    continue
                ar, nn, lk = _leaflet_voronoi(sub, rs, Lx, Ly)
                for lip in lipids:
                    m = rs == lip
                    bucket[lip]["apl"] += list(ar[m][~np.isnan(ar[m])])
                    bucket[lip]["nn"] += list(nn[m])
                    bucket[lip]["like"] += list(lk[m][~np.isnan(lk[m])])
        for lip in lipids:
            a = np.array(bucket[lip]["apl"])
            if a.size:
                hist[lip].append(np.histogram(a, bins=edges, density=True)[0])
                aplm[lip].append(a.mean())
                nnm[lip].append(np.mean(bucket[lip]["nn"]))
                likem[lip].append(np.nanmean(bucket[lip]["like"]))

    out = {}
    for lip in lipids:
        if not hist[lip]:
            continue
        H = np.array(hist[lip])
        out[lip] = {
            "apl": ctr, "P": H.mean(0), "Pstd": H.std(0) if len(H) > 1 else np.zeros(len(ctr)),
            "mean_apl": np.mean(aplm[lip]), "apl_err": np.std(aplm[lip]) if len(aplm[lip]) > 1 else 0.0,
            "nn": np.mean(nnm[lip]), "like": np.nanmean(likem[lip]),
        }
    return out

# %%  Compute for every composition
results = {lip: {} for lip in LIPIDS}
summary = {}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    out = voronoi_blocks(u)
    for lip in LIPIDS:
        if lip in out:
            results[lip][label] = out[lip]
            summary[(lip, label)] = out[lip]
            np.savetxt(f"results/voronoi_{lip}_{label.replace('/', '-')}.dat",
                       np.column_stack([out[lip]["apl"], out[lip]["P"], out[lip]["Pstd"]]),
                       header="APL_A2  P  std")

print(f"\n{'comp':<8}{'lipid':<7}{'APL(A^2)':>12}{'err':>7}{'<neigh>':>9}{'like':>7}")
for label in SYSTEMS:
    for lip in LIPIDS:
        if (lip, label) in summary:
            s = summary[(lip, label)]
            print(f"{label:<8}{lip:<7}{s['mean_apl']:12.2f}{s['apl_err']:7.2f}"
                  f"{s['nn']:9.2f}{s['like']:7.3f}")

# %%  Plot - APL distributions + mean APL vs composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))
for ax, lip in zip(axes[:2], LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        o = results[lip][L]
        ax.fill_between(o["apl"], o["P"] - o["Pstd"], o["P"] + o["Pstd"],
                        color=colors[L], alpha=0.18, lw=0)
        ax.plot(o["apl"], o["P"], color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DLPG)")
    ax.set_xlabel(r"Area per lipid ($\AA^2$)")
    ax.set_ylabel("P(APL)")
    ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="right", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)

ax = axes[2]
markers = {LIPIDS[0]: "o", LIPIDS[-1]: "s"}
for lip in LIPIDS:
    xs = [SYSTEMS[L][2] for L in labels_sorted if L in results[lip]]
    ys = [summary[(lip, L)]["mean_apl"] for L in labels_sorted if L in results[lip]]
    es = [summary[(lip, L)]["apl_err"] for L in labels_sorted if L in results[lip]]
    cs = [colors[L] for L in labels_sorted if L in results[lip]]
    ax.plot(xs, ys, color="0.6", lw=1.0, zorder=1)
    ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor="0.5", capsize=3, zorder=2)
    ax.scatter(xs, ys, c=cs, marker=markers.get(lip, "o"), s=55,
               edgecolor="k", linewidth=0.5, zorder=3, label=lip)
ax.set_xlabel("DLPG %")
ax.set_ylabel(r"mean APL ($\AA^2$)")
ax.text(0.05, 0.92, "APL vs composition", transform=ax.transAxes,
        ha="left", va="top", fontsize=11)
ax.tick_params(direction="in", top=True, right=True)
ax.legend(title="lipid (marker)", fontsize=9)

handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("DOPC/DLPG area per lipid vs composition", y=1.15)
plt.tight_layout()
plt.savefig("results/voronoi_DOPC_DLPG_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS cross-check - overall mean APL from the box
#
# GROMACS has no built-in per-lipid Voronoi tool, but the **overall** mean APL is
# just the box cross-section divided by lipids per leaflet:
#     APL_mean = 2 * Box-X * Box-Y / N_lipids
# Get the box vectors from the energy file:
#
# ```bash
# gmx energy -f Prod_65.edr -o box_65.xvg     # choose Box-X and Box-Y
# ```
#
# Then in Python:  APL = 2 * <Box-X * Box-Y> / N_lipids  (should match the
# mixture-averaged mean of the two Voronoi APLs). For per-lipid-type APL you need
# the Voronoi tessellation above, or a dedicated tool (FATSLiM, APL@Voro,
# GridMAT-MD).

# %%  (Optional) box-based mean APL from a gmx energy .xvg
def read_xvg(path):
    leg, rows = [], []
    for line in open(path):
        if line.startswith("@") and " legend " in line:
            leg.append(line.split('"')[1])
        elif not line.startswith(("#", "@")):
            rows.append([float(x) for x in line.split()])
    return leg, np.array(rows)

# leg, d = read_xvg("box_65.xvg")   # columns: time, Box-X, Box-Y (nm)
# N_lipids = 200                     # lipids per bilayer for that system
# bx, by = d[:, 1], d[:, 2]
# apl_nm2 = 2 * (bx * by).mean() / N_lipids
# print("overall mean APL = %.2f A^2" % (apl_nm2 * 100))   # nm^2 -> A^2

In [ ]:
# %% [markdown]
# # Lipid spatial distribution - DOPC/DLPG composition series
#
# Time-averaged 2D lateral density of each lipid's phosphate in the membrane
# plane, per leaflet, shown as an ENRICHMENT map: each map is divided by its own
# mean, so 1.0 = uniform, >1 = locally enriched (clustering), <1 = depleted.
#
# Unlike the other properties this one is a 2D map, so it is shown as a grid of
# heatmaps (rows = composition, columns = DOPC / DLPG) rather than overlaid
# curves. The radial (curve) version of the same information is the cross-RDF
# (DOPC-DLPG), done in the RDF script.
#
# Trajectory to use: a **whole + centered** xtc. Only the P atoms are needed, so
# the small P-only `heads` trajectory works well (or the full one). Not nojump.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DLPG"]
HEAD     = "P"
BINS     = 40           # grid resolution (BINS x BINS)
LEAFLET  = "upper"      # "upper", "lower", or "both"
SMOOTH   = 1.0          # Gaussian-ish smoothing passes (0 = none); simple box blur
STEP     = 1
VMIN, VMAX = 0.5, 1.5   # enrichment colour range (1.0 = uniform)

# label -> (topology, whole+centered trajectory, DLPG_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}
os.makedirs("results", exist_ok=True)

# %%  Density-map core (per leaflet, enrichment = map / mean)
def _leaf_mask(z, zmid, leaflet):
    if leaflet == "upper":
        return z >= zmid
    if leaflet == "lower":
        return z < zmid
    return np.ones(len(z), dtype=bool)          # both

def _box_blur(M, passes):
    for _ in range(int(passes)):
        M = (M
             + np.roll(M, 1, 0) + np.roll(M, -1, 0)
             + np.roll(M, 1, 1) + np.roll(M, -1, 1)) / 5.0
    return M

def density_maps(u, lipids=LIPIDS, head=HEAD, bins=BINS, leaflet=LEAFLET,
                 smooth=SMOOTH, step=STEP):
    allP = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    sel = {lip: u.select_atoms(f"name {head} and resname {lip}") for lip in lipids}
    H = {lip: np.zeros((bins, bins)) for lip in lipids}
    nframes = 0
    for ts in u.trajectory[::step]:
        Lx, Ly = ts.dimensions[0], ts.dimensions[1]
        zmid = allP.positions[:, 2].mean()
        ex = np.linspace(0, Lx, bins + 1)
        ey = np.linspace(0, Ly, bins + 1)
        for lip in lipids:
            p = sel[lip].positions
            m = _leaf_mask(p[:, 2], zmid, leaflet)
            q = p[m]
            if len(q) == 0:
                continue
            H[lip] += np.histogram2d(np.mod(q[:, 0], Lx), np.mod(q[:, 1], Ly),
                                     bins=[ex, ey])[0]
        nframes += 1
    maps = {}
    for lip in lipids:
        M = H[lip] / max(nframes, 1)
        if smooth:
            M = _box_blur(M, smooth)
        mean = M.mean()
        maps[lip] = M / mean if mean > 0 else M       # enrichment (uniform = 1)
    return maps

# %%  Compute for every composition
results = {}     # label -> {lip: enrichment_map}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    maps = density_maps(u)
    results[label] = maps
    for lip in LIPIDS:
        np.savetxt(f"results/spatial_{lip}_{label.replace('/', '-')}.dat", maps[lip])
print("done")

# %%  Plot - grid of enrichment maps (rows = composition, cols = lipid)
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DLPG -> low
present = [L for L in labels_sorted if L in results]
nrow, ncol = len(present), len(LIPIDS)
fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.0 * nrow), squeeze=False)
norm = TwoSlopeNorm(vmin=VMIN, vcenter=1.0, vmax=VMAX)
im = None
for r, L in enumerate(present):
    for c, lip in enumerate(LIPIDS):
        ax = axes[r, c]
        im = ax.imshow(results[L][lip].T, origin="lower", cmap="RdBu_r",
                       norm=norm, interpolation="bilinear", aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(lip, fontsize=12, style="italic", fontweight="bold")
        if c == 0:
            ax.set_ylabel(f"{L}\n({SYSTEMS[L][2]}% DLPG)", fontsize=10)
fig.suptitle(f"DOPC/DLPG lateral enrichment ({LEAFLET} leaflet)  -  "
             r"P density / mean", y=1.005, fontsize=12)
fig.subplots_adjust(right=0.88)
cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cb = fig.colorbar(im, cax=cax)
cb.set_label("enrichment (density / mean)")
plt.savefig("results/spatial_distribution_DOPC_DLPG_composition.png",
            dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## Notes
#
# * For a well-mixed FLUID bilayer the time-averaged map is close to uniform
#   (~1 everywhere) because lipids diffuse; strong red patches that persist =
#   real clustering/domains. If maps look flat, that is a physical result (good
#   mixing), and the cross-RDF quantifies any short-range preference.
# * `LEAFLET` switches between the two monolayers; run both to check symmetry.
# * `VMIN/VMAX` set the colour range around 1.0 (uniform). Tighten them
#   (e.g. 0.8-1.2) to make weak enrichment visible.

# %%  (Optional) representative snapshot - top view of P atoms coloured by lipid
def snapshot(u, lipids=LIPIDS, head=HEAD, leaflet=LEAFLET, frame=-1):
    u.trajectory[frame]
    allP = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    zmid = allP.positions[:, 2].mean()
    cols = {"DOPC": "#1f77b4", "DLPG": "#d62728"}
    plt.figure(figsize=(4.2, 4.0))
    for lip in lipids:
        p = u.select_atoms(f"name {head} and resname {lip}").positions
        m = _leaf_mask(p[:, 2], zmid, leaflet)
        plt.scatter(p[m, 0], p[m, 1], s=25, color=cols.get(lip, "k"),
                    edgecolor="k", linewidth=0.3, label=lip)
    plt.gca().set_aspect("equal"); plt.xlabel("x (Å)"); plt.ylabel("y (Å)")
    plt.legend(); plt.title(f"snapshot ({leaflet} leaflet)"); plt.tight_layout(); plt.show()

# example:
# u = mda.Universe(*SYSTEMS["65/35"][:2]); snapshot(u)

# %% [markdown]
# ## GROMACS route - `gmx densmap`
#
# `gmx densmap` writes the time-averaged 2D density of a group in the xy plane.
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc ; NDX=index.ndx
#
# # DOPC phosphate 2D map (choose the "DOPC_P" group, or make one with make_ndx):
# gmx densmap -s $TPR -f $XTC -n $NDX -bin 0.1 -od densmap_DOPC_65.dat
# # repeat with the DLPG P group -> densmap_DLPG_65.dat
# ```
#
# `-bin 0.1` = 0.1 nm grid. The .dat is a plain matrix you can load with
# `np.loadtxt` and imshow exactly like the maps above (divide by its mean for the
# enrichment view). `gmx densmap` averages over both leaflets and all z by
# default; for a single leaflet, restrict the input group to that leaflet's P
# atoms (leaflet assignment is dynamic, so the Python route above is easier).

# %%  (Optional) load a gmx densmap .dat and show it
# M = np.loadtxt("densmap_DOPC_65.dat")
# plt.imshow((M/M.mean()).T, origin="lower", cmap="RdBu_r",
#            norm=TwoSlopeNorm(0.5, 1.0, 1.5)); plt.colorbar(); plt.show()

In [ ]:
# %% [markdown]
# # RDF - DOPC/DLPA composition series  (VS Code interactive script)
#
# Run cell-by-cell with the "Run Cell" links above each `# %%` (VS Code's Python
# Interactive window), or run the whole file. Same analysis as the notebook.
#
# Trajectory to use: a **whole + centered** xtc. Do NOT use the nojump xtc here -
# nojump is only for diffusion.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the VS Code terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
HEAD    = "P"
RMAX_NM = 5.0          # x-axis range (nm)
BIN_NM  = 0.02         # bin width (nm)
NBLOCKS = 5            # time blocks for the error band
STEP    = 1            # frame stride

# label -> (topology, trajectory[whole+centered], DLPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#e6194B",   # red
    "72/28": "#ff8c00",   # orange
    "80/20": "#00a878",   # teal
    "87/13": "#2564eb",   # blue
    "95/5" : "#e6009e",   # magenta
}

PAIRS = [("DOPC", "DOPC"), ("DLPA", "DLPA"), ("DOPC", "DLPA")]
YLIM  = {("DOPC", "DOPC"): (0.5, 1.5),
         ("DLPA", "DLPA"): (0.5, 1.5),
         ("DOPC", "DLPA"): (0.75, 1.25)}
os.makedirs("results", exist_ok=True)

# %%  RDF core (per-leaflet, block-averaged)
def _pdist2d(a_xy, b_xy, Lx, Ly, same):
    """Minimum-image lateral distances (A) between two point sets."""
    if len(a_xy) == 0 or len(b_xy) == 0:
        return np.empty(0)
    dx = a_xy[:, None, 0] - b_xy[None, :, 0]
    dy = a_xy[:, None, 1] - b_xy[None, :, 1]
    dx -= Lx * np.round(dx / Lx)
    dy -= Ly * np.round(dy / Ly)
    r = np.sqrt(dx * dx + dy * dy)
    return r[np.triu_indices(r.shape[0], 1)] if same else r.ravel()


def rdf2d_blocks(u, resA, resB, head=HEAD, rmax_nm=RMAX_NM, bin_nm=BIN_NM,
                 nblocks=NBLOCKS, step=STEP):
    same = (resA == resB)
    rmax = rmax_nm * 10.0
    nbins = int(round(rmax / (bin_nm * 10.0)))
    edges = np.linspace(0, rmax, nbins + 1)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    shell = np.pi * (edges[1:] ** 2 - edges[:-1] ** 2)
    selA = u.select_atoms(f"name {head} and resname {resA}")
    selB = u.select_atoms(f"name {head} and resname {resB}")
    allP = u.select_atoms(f"name {head}")
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    gblocks = []
    for blk in blocks:
        hist = np.zeros(nbins)
        norm = 0.0
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            area = Lx * Ly
            zmid = allP.positions[:, 2].mean()
            Apos, Bpos = selA.positions, selB.positions
            for sign in (True, False):                 # upper then lower leaflet
                a = Apos[(Apos[:, 2] >= zmid) == sign][:, :2]
                b = Bpos[(Bpos[:, 2] >= zmid) == sign][:, :2]
                hist += np.histogram(_pdist2d(a, b, Lx, Ly, same), bins=edges)[0]
                Na, Nb = len(a), len(b)
                norm += 0.5 * Na * ((Nb - 1) / area) if same else Na * (Nb / area)
        if norm > 0:
            gblocks.append(hist / (norm * shell))
    G = np.array(gblocks)
    mean = G.mean(axis=0)
    std = G.std(axis=0) if len(G) > 1 else np.zeros_like(mean)
    return ctr / 10.0, mean, std                        # r in nm

# %%  Compute for every composition and pair
results = {(a, b): {} for a, b in PAIRS}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for a, b in PAIRS:
        r, m, s = rdf2d_blocks(u, a, b)
        results[(a, b)][label] = (r, m, s)
        np.savetxt(f"results/rdf_{a}-{b}_{label.replace('/', '-')}.dat",
                   np.column_stack([r, m, s]), header="r_nm  g(r)  std")
print("done")

# %%  Plot - one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DLPA -> low
# fall back to a qualitative colormap for any label missing from COLORS
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, len(PAIRS), figsize=(4.2 * len(PAIRS), 3.6))
if len(PAIRS) == 1:
    axes = [axes]
for ax, (a, b) in zip(axes, PAIRS):
    for L in labels_sorted:
        if L not in results[(a, b)]:
            continue
        r, m, s = results[(a, b)][L]
        ax.fill_between(r, m - s, m + s, color=colors[L], alpha=0.18, lw=0)
        ax.plot(r, m, color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DLPA)")
    ax.axhline(1.0, color="0.6", lw=0.6, ls="--")
    ax.set_xlim(0, RMAX_NM)
    ax.set_ylim(*YLIM.get((a, b), (0.5, 1.5)))
    ax.set_xlabel("Distance (nm)")
    ax.text(0.95, 0.06, f"{a}-{b}", transform=ax.transAxes, style="italic",
            ha="right", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("RDF")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=n if (n := len(labels_sorted)) else 1,
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.12))
fig.suptitle("DOPC/DLPA lateral RDF (P-P) vs composition", y=1.18)
plt.tight_layout()
plt.savefig("results/rdf_DOPC_DLPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## Optional: plot straight from `gmx rdf` .xvg files

# %%  read gmx xvg (optional)
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # r(nm), g(r)

# gmx_files = {"65/35": "rdf_DOPC-DLPA_65.xvg"}
# plt.figure(figsize=(4.5, 3.6))
# for L in labels_sorted:
#     if L in gmx_files:
#         r, g = read_xvg_xy(gmx_files[L]); plt.plot(r, g, color=colors[L], label=L)
# plt.axhline(1, ls="--", color="0.6", lw=.6); plt.xlim(0, RMAX_NM); plt.ylim(0.75, 1.25)
# plt.xlabel("Distance (nm)"); plt.ylabel("RDF"); plt.legend(); plt.show()

In [ ]:
# %% [markdown]
# # Lipid tail splay - DOPC/DLPA composition series  (VS Code interactive script)
#
# Intra-lipid splay = angle between a lipid's two acyl-chain vectors
# (sn-1: first->last C3x carbon; sn-2: first->last C2x carbon). We build the
# probability distribution P(theta) per composition, per lipid type, with an
# error band from block averaging - same style as the RDF figure.
#
# Trajectory to use: the **full-atom, whole + centered** xtc (e.g.
# `Prod_merged_center_*.xtc`). NOT the P-only `heads` trajectory (splay needs the
# chain carbons) and NOT the nojump one (that's diffusion only).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the VS Code terminal
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DLPA"]    # the two lipids -> one panel each
BINW_DEG = 2.0                 # splay-angle bin width (degrees)
ANG_MAX  = 180.0               # angle axis range
NBLOCKS  = 5                   # time blocks for the error band
STEP     = 1                   # frame stride

# label -> (topology, full-atom trajectory[whole+centered], DLPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#e6194B",   # red
    "72/28": "#ff8c00",   # orange
    "80/20": "#00a878",   # teal
    "87/13": "#2564eb",   # blue
    "95/5" : "#e6009e",   # magenta
}
os.makedirs("results", exist_ok=True)

# %%  Splay core (block-averaged distribution)
def _carbons(res, prefix):
    """Acyl carbons of a chain ('C3'=sn-1, 'C2'=sn-2), skipping carbonyl idx 1."""
    pat = re.compile(r"^" + prefix + r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in res.atoms
            if (m := pat.match(at.name)) and int(m.group(1)) >= 2]
    return sorted(hits, key=lambda t: t[0])


def _splay_indices(u, resname):
    """(N,4) atom indices: sn1_first, sn1_last, sn2_first, sn2_last per lipid."""
    idx = []
    for res in u.select_atoms(f"resname {resname}").residues:
        c3 = _carbons(res, "C3")      # sn-1
        c2 = _carbons(res, "C2")      # sn-2
        if len(c3) >= 2 and len(c2) >= 2:
            idx.append((c3[0][1].index, c3[-1][1].index,
                        c2[0][1].index, c2[-1][1].index))
    return np.array(idx, dtype=int)


def splay_blocks(u, resname, binw=BINW_DEG, angmax=ANG_MAX,
                 nblocks=NBLOCKS, step=STEP):
    idx = _splay_indices(u, resname)
    edges = np.arange(0.0, angmax + binw, binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    if len(idx) == 0:
        return ctr, np.zeros_like(ctr), np.zeros_like(ctr), np.nan
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    Hs, all_ang = [], []
    for blk in blocks:
        ang = []
        for fi in blk:
            u.trajectory[int(fi)]
            pos = u.atoms.positions
            v1 = pos[idx[:, 1]] - pos[idx[:, 0]]      # sn-1 vector
            v2 = pos[idx[:, 3]] - pos[idx[:, 2]]      # sn-2 vector
            n1 = np.linalg.norm(v1, axis=1)
            n2 = np.linalg.norm(v2, axis=1)
            good = (n1 > 0) & (n2 > 0)
            cos = np.einsum("ij,ij->i", v1, v2) / (n1 * n2)
            ang.append(np.degrees(np.arccos(np.clip(cos[good], -1, 1))))
        ang = np.concatenate(ang)
        all_ang.append(ang)
        Hs.append(np.histogram(ang, bins=edges, density=True)[0])
    H = np.array(Hs)
    mean = H.mean(axis=0)
    std = H.std(axis=0) if len(H) > 1 else np.zeros_like(mean)
    mean_angle = np.concatenate(all_ang).mean()
    return ctr, mean, std, mean_angle

# %%  Compute for every composition and lipid
results = {lip: {} for lip in LIPIDS}   # lip -> {label: (theta, P, Pstd)}
means = {}                              # (lip,label) -> mean splay angle
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for lip in LIPIDS:
        theta, P, Pstd, mang = splay_blocks(u, lip)
        results[lip][label] = (theta, P, Pstd)
        means[(lip, label)] = mang
        np.savetxt(f"results/splay_{lip}_{label.replace('/', '-')}.dat",
                   np.column_stack([theta, P, Pstd]), header="angle_deg  P  std")

print("\nMean splay angle (deg):")
print(f"{'comp':<8}" + "".join(f"{lip:>10}" for lip in LIPIDS))
for label in SYSTEMS:
    row = "".join(f"{means.get((lip,label), float('nan')):10.1f}" for lip in LIPIDS)
    print(f"{label:<8}{row}")

# %%  Plot - one panel per lipid, one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DLPA -> low
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, len(LIPIDS), figsize=(4.6 * len(LIPIDS), 3.8),
                         sharex=True)
if len(LIPIDS) == 1:
    axes = [axes]
for ax, lip in zip(axes, LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        theta, P, Pstd = results[lip][L]
        ax.fill_between(theta, P - Pstd, P + Pstd, color=colors[L], alpha=0.18, lw=0)
        ax.plot(theta, P, color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DLPA)")
    ax.set_xlim(0, ANG_MAX)
    ax.set_xlabel("Tail splay angle (deg)")
    ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="right", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)
axes[0].set_ylabel("P(θ)")
handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.10))
fig.suptitle("DOPC/DLPA intra-lipid tail splay vs composition", y=1.16)
plt.tight_layout()
plt.savefig("results/splay_DOPC_DLPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route (equivalent) - `gmx gangle`
#
# The intra-lipid splay is the angle between the sn-1 and sn-2 chain vectors.
# `gmx gangle -g1 vector -g2 vector` pairs vector *i* of group1 with vector *i*
# of group2. Each vector is a pair of atoms (chain first & last carbon), so the
# selections below give exactly one vector per lipid.
#
# DOPC is 18:1 (C318 sn-1, C218 sn-2); DLPA is 12:0 (C312 sn-1, C212 sn-2).
# (DOPC keeps C318/C218; DLPA uses C312/C212.)
#
# ```bash
# # one composition (repeat for 65,72,80,87,95), one lipid at a time:
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc
#
# # DOPC splay
# gmx gangle -s $TPR -f $XTC -g1 vector -g2 vector \
#     -group1 'name C32 C318 and resname DOPC' \
#     -group2 'name C22 C218 and resname DOPC' \
#     -oh splay_DOPC_65.xvg -binw 2
#
# # DLPA splay
# gmx gangle -s $TPR -f $XTC -g1 vector -g2 vector \
#     -group1 'name C32 C312 and resname DLPA' \
#     -group2 'name C22 C212 and resname DLPA' \
#     -oh splay_DLPA_65.xvg -binw 2
# ```
#
# `-oh` writes the normalized angle histogram P(theta). For the error band, run on
# time blocks (`-b/-e`) and take mean/std across the .xvg files, or use the Python
# block averaging above.

# %%  (Optional) plot straight from gmx gangle -oh .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # angle(deg), P

# gmx_files = {"DOPC": {"65/35": "splay_DOPC_65.xvg"}, "DLPA": {"65/35": "splay_DLPA_65.xvg"}}
# fig, axes = plt.subplots(1, len(LIPIDS), figsize=(4.6*len(LIPIDS), 3.8))
# for ax, lip in zip(np.atleast_1d(axes), LIPIDS):
#     for L in labels_sorted:
#         p = gmx_files.get(lip, {}).get(L)
#         if p and os.path.exists(p):
#             x, y = read_xvg_xy(p); ax.plot(x, y, color=colors[L], lw=1.8, label=L)
#     ax.set_xlabel("Tail splay angle (deg)"); ax.set_xlim(0, ANG_MAX)
#     ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic", ha="right", va="top")
# plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# %% [markdown]
# # Water depth / penetration - DOPC/DLPA composition series
#
# Symmetrised water density profile across the bilayer normal: water density
# (relative to bulk) vs z from the membrane centre, overlaid for the five
# compositions with a block-averaged error band. Deeper penetration = the curve
# stays high closer to z = 0.
#
# Also prints a table per composition: half-density depth (|z| where water falls
# to 0.5 of bulk), mean phosphate position (headgroup plane), and mean # waters in
# the hydrophobic core (|z| < core).
#
# Trajectory to use: the full **whole + centered** xtc that still contains water
# (`Prod_merged_center_*.xtc`). Not the water-stripped mem/heads trajectories, and
# not nojump (diffusion only). Centering matters here (profile is from the centre).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPID_SEL = "resname DOPC DLPA"          # membrane (defines the centre)
WATER_SEL = "resname TIP3 SOL and (name OH2 or name OW or name O)"  # water oxygen
HEAD      = "P"
ZMAX_NM   = 4.0     # profile half-range from centre (nm)
BINW_NM   = 0.05    # bin width (nm)
CORE_NM   = 1.0     # |z| < CORE counts as hydrophobic-core water
NBLOCKS   = 5       # time blocks for the error band
STEP      = 1

# label -> (topology, full trajectory[whole+centered, WITH water], DLPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#e6194B",   # red
    "72/28": "#ff8c00",   # orange
    "80/20": "#00a878",   # teal
    "87/13": "#2564eb",   # blue
    "95/5" : "#e6009e",   # magenta
}
os.makedirs("results", exist_ok=True)

# %%  Water-profile core (block-averaged, symmetrised)
def water_profile_blocks(u, lipid_sel=LIPID_SEL, water_sel=WATER_SEL, head=HEAD,
                         zmax_nm=ZMAX_NM, binw_nm=BINW_NM, core_nm=CORE_NM,
                         nblocks=NBLOCKS, step=STEP):
    lip = u.select_atoms(lipid_sel)
    wat = u.select_atoms(water_sel)
    P = u.select_atoms(f"name {head}")
    zmax, binw, core = zmax_nm * 10, binw_nm * 10, core_nm * 10
    edges = np.arange(-zmax, zmax + binw, binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    nb_bins = len(ctr)
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]

    prof_blocks = []
    pzs, core_counts = [], []
    for blk in blocks:
        hw = np.zeros(nb_bins)
        areas = []
        nf = 0
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            zc = lip.center_of_mass()[2]
            zw = wat.positions[:, 2] - zc
            hw += np.histogram(zw, bins=edges)[0]
            areas.append(Lx * Ly)
            nf += 1
            pzs.append(np.abs(P.positions[:, 2] - zc).mean())
            core_counts.append(int(np.sum(np.abs(zw) < core)))
        vol = np.mean(areas) * binw                 # A^3 per bin
        dens = hw / nf / vol                          # atoms / A^3
        dens = 0.5 * (dens + dens[::-1])              # symmetrise about centre
        bulk = dens[np.abs(ctr) > (zmax - 10)].mean() # outer 1 nm = bulk
        prof_blocks.append(dens / bulk if bulk > 0 else dens)

    G = np.array(prof_blocks)
    mean = G.mean(axis=0)
    std = G.std(axis=0) if len(G) > 1 else np.zeros_like(mean)

    # half-density depth: |z| where the mean profile drops to 0.5 (going inward)
    zpos = ctr[ctr >= 0]
    mpos = mean[ctr >= 0]
    order = np.argsort(zpos)
    zpos, mpos = zpos[order], mpos[order]
    z_half = np.nan
    for i in range(len(zpos) - 1, 0, -1):
        if mpos[i] < 0.5 <= mpos[i - 1]:
            z_half = np.interp(0.5, [mpos[i], mpos[i - 1]], [zpos[i], zpos[i - 1]])
            break
    return (ctr / 10.0, mean, std, np.mean(pzs) / 10.0,
            (z_half / 10.0 if z_half == z_half else np.nan), np.mean(core_counts))

# %%  Compute for every composition
results = {}   # label -> (z, dens, std)
summary = {}   # label -> (phosphate_nm, half_depth_nm, core_waters)
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    z, d, s, pz, zhalf, ncore = water_profile_blocks(u)
    results[label] = (z, d, s)
    summary[label] = (pz, zhalf, ncore)
    np.savetxt(f"results/water_{label.replace('/', '-')}.dat",
               np.column_stack([z, d, s]), header="z_nm  water_rel_density  std")

print(f"\n{'comp':<8}{'P plane(nm)':>13}{'half-depth(nm)':>16}{'core waters':>13}")
for label in SYSTEMS:
    if label in summary:
        pz, zh, nc = summary[label]
        print(f"{label:<8}{pz:13.2f}{zh:16.2f}{nc:13.1f}")

# %%  Plot - water penetration, one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, ax = plt.subplots(figsize=(6.2, 4.2))
for L in labels_sorted:
    if L not in results:
        continue
    z, d, s = results[L]
    ax.fill_between(z, d - s, d + s, color=colors[L], alpha=0.18, lw=0)
    ax.plot(z, d, color=colors[L], lw=1.8, label=f"{L}  ({SYSTEMS[L][2]}% DLPA)")
# mark the mean phosphate plane
if summary:
    pmean = np.nanmean([summary[L][0] for L in summary])
    for xline in (pmean, -pmean):
        ax.axvline(xline, color="0.5", lw=0.8, ls=":")
    ax.text(pmean, 1.02, "P", color="0.4", ha="center", va="bottom", fontsize=10)
ax.axhline(0.5, color="0.8", lw=0.6, ls="--")
ax.set_xlim(-ZMAX_NM, ZMAX_NM)
ax.set_ylim(0, 1.15)
ax.set_xlabel("z from bilayer centre (nm)")
ax.set_ylabel("Water density (rel. to bulk)")
ax.text(0.03, 0.92, "Water", transform=ax.transAxes, style="italic",
        ha="left", va="top", fontsize=13, fontweight="bold")
ax.tick_params(direction="in", top=True, right=True)
ax.legend(loc="upper center", ncol=3, frameon=True, fontsize=8,
          bbox_to_anchor=(0.5, 1.18))
fig.suptitle("DOPC/DLPA water penetration vs composition", y=1.06)
plt.tight_layout()
plt.savefig("results/water_depth_DOPC_DLPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route (equivalent) - `gmx density`
#
# ```bash
# # one composition (repeat for 65,72,80,87,95):
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc ; NDX=index.ndx
#
# # centering group = Membrane , output group = Water
# printf "Membrane\nWater\n" | gmx density -s $TPR -f $XTC -n $NDX \
#     -d Z -sl 400 -center -symm -dens number -o water_density_65.xvg
#
# # phosphate plane for reference (output group = the P atoms):
# printf "Membrane\nPhosphate\n" | gmx density -s $TPR -f $XTC -n $NDX \
#     -d Z -sl 400 -center -symm -dens number -o phos_density_65.xvg
# ```
#
# `-center -symm` centres on the membrane and symmetrises; `-dens number` gives a
# number density. For error bands, run on time blocks (`-b/-e`) and take mean/std
# across the .xvg files, or use the Python block averaging above.

# %%  (Optional) plot straight from gmx density .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # z(nm), density

# gmx_files = {"65/35": "water_density_65.xvg"}
# plt.figure(figsize=(6.2, 4.2))
# for L in labels_sorted:
#     if L in gmx_files and os.path.exists(gmx_files[L]):
#         z, dw = read_xvg_xy(gmx_files[L])
#         plt.plot(z, dw / np.nanmax(dw), color=colors[L], lw=1.8, label=L)
# plt.xlabel("z (nm)"); plt.ylabel("water density (norm.)"); plt.legend(); plt.show()

In [ ]:
# %% [markdown]
# # Lipid tail order parameter (-S_CD) - DOPC/DLPA composition series
#
# Deuterium order parameter per acyl-chain carbon:
#   S_CD = < (3 cos^2(theta_CH) - 1) / 2 >    (theta_CH = C-H vs bilayer normal z)
# reported as -S_CD (the usual positive quantity). For each CH2 the two C-H
# values are averaged; C-H bonds are found automatically from the topology, so
# the cis double bond at C9-C10 in the oleoyl chains shows its dip on its own.
#
# One panel per (lipid x chain): DOPC/DLPA x sn-1/sn-2, each overlaying the five
# compositions with a block-averaged error band - same style as the RDF/splay
# figures.
#
# Trajectory to use: the full-atom **whole (+centered)** xtc WITH hydrogens
# (`Prod_merged_center_*.xtc`), plus the `.tpr` (needed for C-H bonds). Not the
# P-only `heads` trajectory, and not nojump (that is diffusion only).

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import re
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS  = ["DOPC", "DLPA"]              # one row of panels each
CHAINS  = [("C3", "sn-1"), ("C2", "sn-2")]   # CHARMM36: sn-1=C3x, sn-2=C2x
NBLOCKS = 5                            # time blocks for the error band
STEP    = 1                            # frame stride

# label -> (topology[.tpr], full-atom trajectory[whole+centered], DLPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#e6194B",   # red
    "72/28": "#ff8c00",   # orange
    "80/20": "#00a878",   # teal
    "87/13": "#2564eb",   # blue
    "95/5" : "#e6009e",   # magenta
}
os.makedirs("results", exist_ok=True)

# %%  Order-parameter core (block-averaged, auto C-H detection)
def _carbons(res, prefix):
    # acyl carbons of a chain ('C3'=sn-1, 'C2'=sn-2), skipping carbonyl idx 1
    pat = re.compile(r"^" + prefix + r"(\d+)$")
    hits = [(int(m.group(1)), at) for at in res.atoms
            if (m := pat.match(at.name)) and int(m.group(1)) >= 2]
    return sorted(hits, key=lambda t: t[0])

def _build_pairs(u, resname, prefix, has_bonds):
    # returns (sorted carbon numbers, {k: (C_index_array, H_index_array)})
    C, H = defaultdict(list), defaultdict(list)
    for res in u.select_atoms(f"resname {resname}").residues:
        for k, catom in _carbons(res, prefix):
            if has_bonds:
                hs = [a for a in catom.bonded_atoms if a.name.startswith("H")]
            else:
                hs = list(u.select_atoms("name H* and around 1.3 group c",
                                         c=u.atoms[[catom.index]]))
            for h in hs:
                C[k].append(catom.index); H[k].append(h.index)
    ks = sorted(C)
    return ks, {k: (np.array(C[k]), np.array(H[k])) for k in ks}

def scd_blocks(u, resname, prefix, has_bonds, nblocks=NBLOCKS, step=STEP):
    ks, pairs = _build_pairs(u, resname, prefix, has_bonds)
    if not ks:
        return np.array([]), np.array([]), np.array([])
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]
    block_means = []
    for blk in blocks:
        acc = {k: [] for k in ks}
        for fi in blk:
            u.trajectory[int(fi)]
            pos = u.atoms.positions
            for k in ks:
                Ci, Hi = pairs[k]
                v = pos[Hi] - pos[Ci]
                n = np.linalg.norm(v, axis=1)
                cosz = v[:, 2] / n
                acc[k].append(np.mean(0.5 * (3 * cosz ** 2 - 1)))
        block_means.append([-np.mean(acc[k]) for k in ks])   # -S_CD
    B = np.array(block_means)
    return np.array(ks), B.mean(axis=0), (B.std(axis=0) if len(B) > 1 else np.zeros(len(ks)))

# %%  Compute for every composition, lipid and chain
results = {(lip, prefix): {} for lip in LIPIDS for prefix, _ in CHAINS}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    has_bonds = len(getattr(u, "bonds", [])) > 0
    print(f"{label}: {len(u.trajectory)} frames, bonds={has_bonds}")
    for lip in LIPIDS:
        for prefix, tag in CHAINS:
            ks, scd, err = scd_blocks(u, lip, prefix, has_bonds)
            if ks.size == 0:
                continue
            results[(lip, prefix)][label] = (ks, scd, err)
            np.savetxt(f"results/scd_{lip}_{tag}_{label.replace('/', '-')}.dat",
                       np.column_stack([ks, scd, err]), header="carbon  -SCD  std")
print("done")

# %%  Plot - 2x2 grid (lipid x chain), one distinct colour per composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(len(LIPIDS), len(CHAINS),
                         figsize=(4.4 * len(CHAINS), 3.4 * len(LIPIDS)),
                         sharex=True, sharey=True)
axes = np.atleast_2d(axes)
for r, lip in enumerate(LIPIDS):
    for c, (prefix, tag) in enumerate(CHAINS):
        ax = axes[r, c]
        for L in labels_sorted:
            if L not in results[(lip, prefix)]:
                continue
            ks, scd, err = results[(lip, prefix)][L]
            ax.fill_between(ks, scd - err, scd + err, color=colors[L], alpha=0.18, lw=0)
            ax.plot(ks, scd, color=colors[L], lw=1.8, marker="o", ms=3,
                    label=f"{L}  ({SYSTEMS[L][2]}% DLPA)")
        ax.text(0.95, 0.92, f"{lip} {tag}", transform=ax.transAxes, style="italic",
                ha="right", va="top", fontsize=12, fontweight="bold")
        #ax.grid(alpha=0.3)
        ax.tick_params(direction="in", top=True, right=True)
        if r == len(LIPIDS) - 1:
            ax.set_xlabel("Carbon number")
        if c == 0:
            ax.set_ylabel(r"$-S_{CD}$")
handles, labs = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.05))
fig.suptitle("DOPC/DLPA acyl-chain order parameters vs composition", y=1.10)
plt.tight_layout()
plt.savefig("results/order_parameter_DOPC_DLPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route - `gmx order`
#
# For all-atom CHARMM36 the rigorous S_CD comes from the explicit C-H vectors
# (the Python above). `gmx order` was designed for united-atom chains and works
# on the carbon skeleton, so it gives the closely-related carbon order parameter
# rather than the exact explicit-H S_CD. Use it as a cross-check.
#
# It needs an index file listing the chain carbons **in order** (one group per
# chain). Build it per lipid with `gmx select`, then run `gmx order`:
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc
#
# # sn-1 carbons of DOPC, in order (C32..C318):
# gmx select -s $TPR -on sn1_DOPC.ndx \
#     -select 'resname DOPC and name C32 C33 C34 C35 C36 C37 C38 C39 C310 \
#              C311 C312 C313 C314 C315 C316 C317 C318'
#
# gmx order -s $TPR -f $XTC -n sn1_DOPC.ndx -d z \
#     -od deuter_DOPC_sn1_65.xvg -os order_DOPC_sn1_65.xvg
# ```
#
# Repeat with `name C22..C218` for sn-2, and for DLPA (12:0, so C32..C312 for sn-1 / C22..C212 for sn-2). `-od` writes the
# deuterium order parameter vs carbon. For error bands, run on time blocks
# (`-b/-e`) and take mean/std across the .xvg files, or use the Python blocks.

# %%  (Optional) plot straight from gmx order -od .xvg files
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # carbon, -SCD

# gmx_files = {("DOPC","C3"): {"65/35": "deuter_DOPC_sn1_65.xvg"}}
# for (lip, prefix), fdict in gmx_files.items():
#     plt.figure(figsize=(5, 3.6))
#     for L in labels_sorted:
#         p = fdict.get(L)
#         if p and os.path.exists(p):
#             x, y = read_xvg_xy(p); plt.plot(x, y, color=colors[L], marker="o", ms=3, label=L)
#     plt.xlabel("Carbon number"); plt.ylabel(r"$-S_{CD}$"); plt.legend(); plt.title(f"{lip} {prefix}")
#     plt.show()

In [ ]:
# %% [markdown]
# # Lipid lateral diffusion coefficient - DOPC/DLPA composition series
#
# Lateral mean-squared displacement of each lipid's centre of mass, with
# per-leaflet COM drift removed, fitted as  MSD_xy(t) = 4 D t  ->  D = slope/4.
# Reported per lipid (DOPC, DLPA) for each composition.
#
# Plots (overlay style, one distinct colour per composition):
#   panel 1: DOPC MSD vs lag      panel 2: DLPA MSD vs lag
#   panel 3: fitted D vs DLPA %
# Error bands / error bars come from block averaging (NBLOCKS sub-trajectories).
#
# *** Trajectory to use: the NO-JUMP trajectory ***  (this is the ONLY property
# that uses nojump). Make it first, stripping water to keep it small:
#
#   gmx trjconv -s Prod_65.tpr -f <full_traj>.xtc -n mem.ndx \
#       -pbc nojump -o Prod_nojump_65.xtc            # choose the Membrane group
#
# Do NOT use the centered/whole xtc here - individual lipids still wrap around the
# box in those and the MSD will be wrong.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DLPA"]
HEAD     = "P"
NBLOCKS  = 5           # sub-trajectories for the error estimate
STEP     = 1           # frame stride
FITMIN   = 0.2         # fit window (fraction of each block's lag range)
FITMAX   = 0.6

# label -> (topology, NO-JUMP trajectory, DLPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_nojump_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_nojump_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_nojump_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_nojump_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_nojump_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#e6194B",   # red
    "72/28": "#ff8c00",   # orange
    "80/20": "#00a878",   # teal
    "87/13": "#2564eb",   # blue
    "95/5" : "#e6009e",   # magenta
}
os.makedirs("results", exist_ok=True)

# %%  Diffusion core (lateral MSD, per-leaflet drift removed, block-averaged)
def _msd_curve(seg):
    # seg: (Ls, n, 2) lateral COM positions -> MSD over lags 0..Ls-1
    Ls = seg.shape[0]
    msd = np.zeros(Ls)
    for lag in range(1, Ls):
        disp = seg[lag:] - seg[:-lag]
        msd[lag] = (disp ** 2).sum(axis=2).mean()
    return msd

def diffusion_blocks(u, resname, head=HEAD, nblocks=NBLOCKS, step=STEP,
                     fitmin=FITMIN, fitmax=FITMAX):
    residues = u.select_atoms(f"resname {resname}").residues
    if len(residues) == 0:
        return None
    P = u.select_atoms(f"name {head} and resname {resname}")
    # leaflet assignment from frame 0
    u.trajectory[0]
    zc0 = P.positions[:, 2].mean()
    pz = {a.resid: a.position[2] for a in P}
    leaf = np.array([0 if pz.get(r.resid, zc0) >= zc0 else 1 for r in residues])

    # gather lateral COM per frame with per-leaflet drift removed
    times, coms = [], []
    for ts in u.trajectory[::step]:
        times.append(ts.time)
        com = np.array([r.atoms.center_of_mass()[:2] for r in residues])
        for lf in (0, 1):
            m = leaf == lf
            if m.any():
                com[m] -= com[m].mean(axis=0)
        coms.append(com)
    times = np.array(times)
    coms = np.array(coms)                      # (T, n, 2)
    T = len(times)
    dt = np.mean(np.diff(times)) if T > 1 else 1.0

    # split into contiguous blocks; MSD per block on a common lag range
    segs = [s for s in np.array_split(np.arange(T), max(1, min(nblocks, T))) if len(s) > 2]
    Lmin = min(len(s) for s in segs)
    lags = np.arange(Lmin) * dt
    msds, Ds = [], []
    i0, i1 = max(1, int(fitmin * Lmin)), int(fitmax * Lmin)
    if i1 - i0 < 2:
        i0, i1 = 1, Lmin
    for s in segs:
        m = _msd_curve(coms[s][:Lmin])
        msds.append(m)
        Ds.append(np.polyfit(lags[i0:i1], m[i0:i1], 1)[0] / 4.0 * 1e-4)  # cm^2/s
    M = np.array(msds)
    Ds = np.array(Ds)
    return {"lags": lags, "msd": M.mean(0), "msd_std": M.std(0) if len(M) > 1 else np.zeros(Lmin),
            "D": Ds.mean(), "D_err": Ds.std() if len(Ds) > 1 else 0.0}

# %%  Compute for every composition and lipid
results = {lip: {} for lip in LIPIDS}
Dtab = {}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    for lip in LIPIDS:
        out = diffusion_blocks(u, lip)
        if out is None:
            continue
        results[lip][label] = out
        Dtab[(lip, label)] = (out["D"], out["D_err"])
        np.savetxt(f"results/msd_{lip}_{label.replace('/', '-')}.dat",
                   np.column_stack([out["lags"], out["msd"], out["msd_std"]]),
                   header="lag_ps  msd_A2  std")

print(f"\n{'comp':<8}" + "".join(f"{lip+' D(1e-7)':>18}" for lip in LIPIDS))
for label in SYSTEMS:
    row = ""
    for lip in LIPIDS:
        if (lip, label) in Dtab:
            D, e = Dtab[(lip, label)]
            row += f"{D*1e7:11.3f}+/-{e*1e7:5.3f}"
        else:
            row += f"{'-':>18}"
    print(f"{label:<8}{row}")
print("\n(D in cm^2/s; column shows D x 1e7)")

# %%  Plot - MSD overlays + D vs composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))
# panels 1 & 2: MSD per lipid
for ax, lip in zip(axes[:2], LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        o = results[lip][L]
        lag_k = o["lags"] / 1000.0          # ps -> thousands of ps
        ax.fill_between(lag_k, o["msd"] - o["msd_std"], o["msd"] + o["msd_std"],
                        color=colors[L], alpha=0.18, lw=0)
        ax.plot(lag_k, o["msd"], color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DLPA)")
    ax.set_xlabel(r"lag time ($\times10^{3}$ ps)")
    ax.set_ylabel(r"MSD$_{xy}$ ($\AA^2$)")
    ax.text(0.05, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="left", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)

# panel 3: D vs DLPA %
ax = axes[2]
markers = {LIPIDS[0]: "o", LIPIDS[-1]: "s"}
for lip in LIPIDS:
    xs = [SYSTEMS[L][2] for L in labels_sorted if L in results[lip]]
    ys = [Dtab[(lip, L)][0] * 1e7 for L in labels_sorted if L in results[lip]]
    es = [Dtab[(lip, L)][1] * 1e7 for L in labels_sorted if L in results[lip]]
    cs = [colors[L] for L in labels_sorted if L in results[lip]]
    ax.plot(xs, ys, color="0.6", lw=1.0, zorder=1)
    ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor="0.5", capsize=3, zorder=2)
    ax.scatter(xs, ys, c=cs, marker=markers.get(lip, "o"), s=55,
               edgecolor="k", linewidth=0.5, zorder=3, label=lip)
ax.set_xlabel("DLPA %")
ax.set_ylabel(r"$D$ (10$^{-7}$ cm$^2$/s)")
ax.text(0.05, 0.92, "D vs composition", transform=ax.transAxes,
        ha="left", va="top", fontsize=11)
ax.tick_params(direction="in", top=True, right=True)
ax.legend(title="lipid (marker)", fontsize=9)

handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("DOPC/DLPA lateral diffusion vs composition", y=1.15)
plt.tight_layout()
plt.savefig("results/diffusion_DOPC_DLPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS route - `gmx msd`
#
# `gmx msd -lateral z` gives the in-plane MSD and prints D. Run on the NO-JUMP
# trajectory. Fit only the linear (diffusive) regime with `-beginfit/-endfit`.
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_nojump_65.xtc ; NDX=index.ndx
#
# # DOPC (choose the DOPC group at the prompt):
# gmx msd -s $TPR -f $XTC -n $NDX -lateral z -rmcomm \
#     -beginfit 20000 -endfit 80000 -o msd_DOPC_65.xvg
#
# # DLPA:
# gmx msd -s $TPR -f $XTC -n $NDX -lateral z -rmcomm \
#     -beginfit 20000 -endfit 80000 -o msd_DLPA_65.xvg
# ```
#
# `-lateral z` = in-plane MSD, D = slope/4 (gmx prints it). `-rmcomm` removes the
# overall COM motion; the Python version additionally removes per-leaflet drift,
# which is the more careful choice. Report the finite-size (Yeh-Hummer) caveat:
# absolute D depends on box size - compare compositions at matched box sizes.

# %%  (Optional) read D straight from gmx msd terminal output / .xvg
def read_xvg_xy(path):
    rows = [[float(x) for x in l.split()]
            for l in open(path) if not l.startswith(("#", "@"))]
    d = np.array(rows)
    return d[:, 0], d[:, 1]     # time(ps), MSD

# gmx prints "D[ DOPC] = ... (1e-5 cm^2/s)" to the terminal; you can also refit the
# .xvg yourself:
# t, msd = read_xvg_xy("msd_DOPC_65.xvg")
# i0, i1 = int(0.2*len(t)), int(0.6*len(t))
# slope = np.polyfit(t[i0:i1], msd[i0:i1], 1)[0]
# print("D =", slope/4*1e-4, "cm^2/s")

In [ ]:
# %% [markdown]
# # Voronoi analysis (area per lipid) - DOPC/DLPA composition series
#
# Periodic 2D Voronoi tessellation of the phosphate (P) atoms, done per leaflet.
# From it, per lipid type: area per lipid (APL), mean number of Voronoi
# neighbours, and the like-neighbour fraction (fraction of a lipid's neighbours
# that are the SAME type - a mixing / clustering signal).
#
# Plots (overlay style, one distinct colour per composition):
#   panel 1: DOPC APL distribution P(APL)   panel 2: DLPA APL distribution
#   panel 3: mean APL vs DLPA %
# Bands / error bars from block averaging (NBLOCKS sub-trajectories).
#
# Trajectory to use: a **whole + centered** xtc. Voronoi only needs the P atoms,
# so the small P-only `heads` trajectory is ideal (or the full one). Not nojump.
#
# Requires scipy:  pip install scipy

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy scipy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
import MDAnalysis as mda
from scipy.spatial import Voronoi
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DLPA"]
HEAD     = "P"
APL_MIN  = 30.0        # APL histogram range (A^2)
APL_MAX  = 110.0
APL_BINW = 1.5
NBLOCKS  = 5           # time blocks for the error estimate
STEP     = 1           # frame stride (raise to subsample heavy trajectories)

# label -> (topology, whole+centered trajectory, DLPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}

# One distinct colour per composition (edit freely).
COLORS = {
    "65/35": "#e6194B",   # red
    "72/28": "#ff8c00",   # orange
    "80/20": "#00a878",   # teal
    "87/13": "#2564eb",   # blue
    "95/5" : "#e6009e",   # magenta
}
os.makedirs("results", exist_ok=True)

# %%  Voronoi core (periodic, per leaflet)
def _poly_area(p):
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def _tile(pts, Lx, Ly):
    sh = [(i, j) for i in (-1, 0, 1) for j in (-1, 0, 1)]
    tp = np.vstack([pts + np.array([a * Lx, b * Ly]) for a, b in sh])
    tmap = np.concatenate([np.arange(len(pts)) for _ in sh])
    return tp, tmap

def _leaflet_voronoi(pts, rtypes, Lx, Ly):
    # returns per-point: area, neighbour count, like-neighbour fraction
    N = len(pts)
    tp, tmap = _tile(pts, Lx, Ly)
    vor = Voronoi(tp)
    central = np.arange(4 * N, 5 * N)          # the (0,0) image block
    areas = np.full(N, np.nan)
    for k, pi in enumerate(central):
        reg = vor.regions[vor.point_region[pi]]
        if reg and -1 not in reg:
            areas[k] = _poly_area(vor.vertices[reg])
    neigh = [set() for _ in range(N)]
    for i, j in vor.ridge_points:
        if 4 * N <= i < 5 * N:
            neigh[i - 4 * N].add(tmap[j])
        if 4 * N <= j < 5 * N:
            neigh[j - 4 * N].add(tmap[i])
    for k in range(N):
        neigh[k].discard(k)
    nn = np.array([len(neigh[k]) for k in range(N)], float)
    like = np.array([np.mean([rtypes[m] == rtypes[k] for m in neigh[k]])
                     if neigh[k] else np.nan for k in range(N)])
    return areas, nn, like

def voronoi_blocks(u, lipids=LIPIDS, head=HEAD, apl_min=APL_MIN, apl_max=APL_MAX,
                   apl_binw=APL_BINW, nblocks=NBLOCKS, step=STEP):
    P = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    rt_all = np.array([a.resname for a in P])
    edges = np.arange(apl_min, apl_max + apl_binw, apl_binw)
    ctr = 0.5 * (edges[1:] + edges[:-1])
    frames = list(range(0, len(u.trajectory), step))
    nb = max(1, min(nblocks, len(frames)))
    blocks = [b for b in np.array_split(frames, nb) if len(b)]

    hist = {lip: [] for lip in lipids}     # per-block density histogram
    aplm = {lip: [] for lip in lipids}     # per-block mean APL
    nnm = {lip: [] for lip in lipids}
    likem = {lip: [] for lip in lipids}
    for blk in blocks:
        bucket = {lip: {"apl": [], "nn": [], "like": []} for lip in lipids}
        for fi in blk:
            ts = u.trajectory[int(fi)]
            Lx, Ly = ts.dimensions[0], ts.dimensions[1]
            pos = P.positions.copy()
            pos[:, 0] %= Lx
            pos[:, 1] %= Ly
            zmid = pos[:, 2].mean()
            for mask in (pos[:, 2] >= zmid, pos[:, 2] < zmid):
                sub = pos[mask][:, :2]
                rs = rt_all[mask]
                if len(sub) < 4:
                    continue
                ar, nn, lk = _leaflet_voronoi(sub, rs, Lx, Ly)
                for lip in lipids:
                    m = rs == lip
                    bucket[lip]["apl"] += list(ar[m][~np.isnan(ar[m])])
                    bucket[lip]["nn"] += list(nn[m])
                    bucket[lip]["like"] += list(lk[m][~np.isnan(lk[m])])
        for lip in lipids:
            a = np.array(bucket[lip]["apl"])
            if a.size:
                hist[lip].append(np.histogram(a, bins=edges, density=True)[0])
                aplm[lip].append(a.mean())
                nnm[lip].append(np.mean(bucket[lip]["nn"]))
                likem[lip].append(np.nanmean(bucket[lip]["like"]))

    out = {}
    for lip in lipids:
        if not hist[lip]:
            continue
        H = np.array(hist[lip])
        out[lip] = {
            "apl": ctr, "P": H.mean(0), "Pstd": H.std(0) if len(H) > 1 else np.zeros(len(ctr)),
            "mean_apl": np.mean(aplm[lip]), "apl_err": np.std(aplm[lip]) if len(aplm[lip]) > 1 else 0.0,
            "nn": np.mean(nnm[lip]), "like": np.nanmean(likem[lip]),
        }
    return out

# %%  Compute for every composition
results = {lip: {} for lip in LIPIDS}
summary = {}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    out = voronoi_blocks(u)
    for lip in LIPIDS:
        if lip in out:
            results[lip][label] = out[lip]
            summary[(lip, label)] = out[lip]
            np.savetxt(f"results/voronoi_{lip}_{label.replace('/', '-')}.dat",
                       np.column_stack([out[lip]["apl"], out[lip]["P"], out[lip]["Pstd"]]),
                       header="APL_A2  P  std")

print(f"\n{'comp':<8}{'lipid':<7}{'APL(A^2)':>12}{'err':>7}{'<neigh>':>9}{'like':>7}")
for label in SYSTEMS:
    for lip in LIPIDS:
        if (lip, label) in summary:
            s = summary[(lip, label)]
            print(f"{label:<8}{lip:<7}{s['mean_apl']:12.2f}{s['apl_err']:7.2f}"
                  f"{s['nn']:9.2f}{s['like']:7.3f}")

# %%  Plot - APL distributions + mean APL vs composition
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])
_tab = plt.get_cmap("tab10")
colors = {L: COLORS.get(L, _tab(i % 10)) for i, L in enumerate(labels_sorted)}

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.0))
for ax, lip in zip(axes[:2], LIPIDS):
    for L in labels_sorted:
        if L not in results[lip]:
            continue
        o = results[lip][L]
        ax.fill_between(o["apl"], o["P"] - o["Pstd"], o["P"] + o["Pstd"],
                        color=colors[L], alpha=0.18, lw=0)
        ax.plot(o["apl"], o["P"], color=colors[L], lw=1.8,
                label=f"{L}  ({SYSTEMS[L][2]}% DLPA)")
    ax.set_xlabel(r"Area per lipid ($\AA^2$)")
    ax.set_ylabel("P(APL)")
    ax.text(0.95, 0.92, lip, transform=ax.transAxes, style="italic",
            ha="right", va="top", fontsize=13, fontweight="bold")
    ax.tick_params(direction="in", top=True, right=True)

ax = axes[2]
markers = {LIPIDS[0]: "o", LIPIDS[-1]: "s"}
for lip in LIPIDS:
    xs = [SYSTEMS[L][2] for L in labels_sorted if L in results[lip]]
    ys = [summary[(lip, L)]["mean_apl"] for L in labels_sorted if L in results[lip]]
    es = [summary[(lip, L)]["apl_err"] for L in labels_sorted if L in results[lip]]
    cs = [colors[L] for L in labels_sorted if L in results[lip]]
    ax.plot(xs, ys, color="0.6", lw=1.0, zorder=1)
    ax.errorbar(xs, ys, yerr=es, fmt="none", ecolor="0.5", capsize=3, zorder=2)
    ax.scatter(xs, ys, c=cs, marker=markers.get(lip, "o"), s=55,
               edgecolor="k", linewidth=0.5, zorder=3, label=lip)
ax.set_xlabel("DLPA %")
ax.set_ylabel(r"mean APL ($\AA^2$)")
ax.text(0.05, 0.92, "APL vs composition", transform=ax.transAxes,
        ha="left", va="top", fontsize=11)
ax.tick_params(direction="in", top=True, right=True)
ax.legend(title="lipid (marker)", fontsize=9)

handles, labs = axes[0].get_legend_handles_labels()
fig.legend(handles, labs, loc="upper center", ncol=len(labels_sorted),
           frameon=True, fontsize=9, bbox_to_anchor=(0.5, 1.08))
fig.suptitle("DOPC/DLPA area per lipid vs composition", y=1.15)
plt.tight_layout()
plt.savefig("results/voronoi_DOPC_DLPA_composition.png", dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## GROMACS cross-check - overall mean APL from the box
#
# GROMACS has no built-in per-lipid Voronoi tool, but the **overall** mean APL is
# just the box cross-section divided by lipids per leaflet:
#     APL_mean = 2 * Box-X * Box-Y / N_lipids
# Get the box vectors from the energy file:
#
# ```bash
# gmx energy -f Prod_65.edr -o box_65.xvg     # choose Box-X and Box-Y
# ```
#
# Then in Python:  APL = 2 * <Box-X * Box-Y> / N_lipids  (should match the
# mixture-averaged mean of the two Voronoi APLs). For per-lipid-type APL you need
# the Voronoi tessellation above, or a dedicated tool (FATSLiM, APL@Voro,
# GridMAT-MD).

# %%  (Optional) box-based mean APL from a gmx energy .xvg
def read_xvg(path):
    leg, rows = [], []
    for line in open(path):
        if line.startswith("@") and " legend " in line:
            leg.append(line.split('"')[1])
        elif not line.startswith(("#", "@")):
            rows.append([float(x) for x in line.split()])
    return leg, np.array(rows)

# leg, d = read_xvg("box_65.xvg")   # columns: time, Box-X, Box-Y (nm)
# N_lipids = 200                     # lipids per bilayer for that system
# bx, by = d[:, 1], d[:, 2]
# apl_nm2 = 2 * (bx * by).mean() / N_lipids
# print("overall mean APL = %.2f A^2" % (apl_nm2 * 100))   # nm^2 -> A^2

In [ ]:
# %% [markdown]
# # Lipid spatial distribution - DOPC/DLPA composition series
#
# Time-averaged 2D lateral density of each lipid's phosphate in the membrane
# plane, per leaflet, shown as an ENRICHMENT map: each map is divided by its own
# mean, so 1.0 = uniform, >1 = locally enriched (clustering), <1 = depleted.
#
# Unlike the other properties this one is a 2D map, so it is shown as a grid of
# heatmaps (rows = composition, columns = DOPC / DLPA) rather than overlaid
# curves. The radial (curve) version of the same information is the cross-RDF
# (DOPC-DLPA), done in the RDF script.
#
# Trajectory to use: a **whole + centered** xtc. Only the P atoms are needed, so
# the small P-only `heads` trajectory works well (or the full one). Not nojump.

# %%  Setup
# pip install "MDAnalysis>=2.4" numpy matplotlib     # run once in the terminal
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import MDAnalysis as mda
print("MDAnalysis", mda.__version__)

# %%  Config  -- EDIT THE PATHS
LIPIDS   = ["DOPC", "DLPA"]
HEAD     = "P"
BINS     = 40           # grid resolution (BINS x BINS)
LEAFLET  = "upper"      # "upper", "lower", or "both"
SMOOTH   = 1.0          # Gaussian-ish smoothing passes (0 = none); simple box blur
STEP     = 1
VMIN, VMAX = 0.5, 1.5   # enrichment colour range (1.0 = uniform)

# label -> (topology, whole+centered trajectory, DLPA_percent)
SYSTEMS = {
    "65/35": ("Prod_65.tpr", "Prod_merged_center_65.xtc", 35),
    "72/28": ("Prod_72.tpr", "Prod_merged_center_72.xtc", 28),
    "80/20": ("Prod_80.tpr", "Prod_merged_center_80.xtc", 20),
    "87/13": ("Prod_87.tpr", "Prod_merged_center_87.xtc", 13),
    "95/5" : ("Prod_95.tpr", "Prod_merged_center_95.xtc", 5),
}
os.makedirs("results", exist_ok=True)

# %%  Density-map core (per leaflet, enrichment = map / mean)
def _leaf_mask(z, zmid, leaflet):
    if leaflet == "upper":
        return z >= zmid
    if leaflet == "lower":
        return z < zmid
    return np.ones(len(z), dtype=bool)          # both

def _box_blur(M, passes):
    for _ in range(int(passes)):
        M = (M
             + np.roll(M, 1, 0) + np.roll(M, -1, 0)
             + np.roll(M, 1, 1) + np.roll(M, -1, 1)) / 5.0
    return M

def density_maps(u, lipids=LIPIDS, head=HEAD, bins=BINS, leaflet=LEAFLET,
                 smooth=SMOOTH, step=STEP):
    allP = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    sel = {lip: u.select_atoms(f"name {head} and resname {lip}") for lip in lipids}
    H = {lip: np.zeros((bins, bins)) for lip in lipids}
    nframes = 0
    for ts in u.trajectory[::step]:
        Lx, Ly = ts.dimensions[0], ts.dimensions[1]
        zmid = allP.positions[:, 2].mean()
        ex = np.linspace(0, Lx, bins + 1)
        ey = np.linspace(0, Ly, bins + 1)
        for lip in lipids:
            p = sel[lip].positions
            m = _leaf_mask(p[:, 2], zmid, leaflet)
            q = p[m]
            if len(q) == 0:
                continue
            H[lip] += np.histogram2d(np.mod(q[:, 0], Lx), np.mod(q[:, 1], Ly),
                                     bins=[ex, ey])[0]
        nframes += 1
    maps = {}
    for lip in lipids:
        M = H[lip] / max(nframes, 1)
        if smooth:
            M = _box_blur(M, smooth)
        mean = M.mean()
        maps[lip] = M / mean if mean > 0 else M       # enrichment (uniform = 1)
    return maps

# %%  Compute for every composition
results = {}     # label -> {lip: enrichment_map}
for label, (top, traj, pct) in SYSTEMS.items():
    if not (os.path.exists(top) and os.path.exists(traj)):
        print(f"  skip {label}: file(s) missing ({top} / {traj})")
        continue
    u = mda.Universe(top, traj)
    print(f"{label}: {len(u.trajectory)} frames")
    maps = density_maps(u)
    results[label] = maps
    for lip in LIPIDS:
        np.savetxt(f"results/spatial_{lip}_{label.replace('/', '-')}.dat", maps[lip])
print("done")

# %%  Plot - grid of enrichment maps (rows = composition, cols = lipid)
labels_sorted = sorted(SYSTEMS, key=lambda L: -SYSTEMS[L][2])   # high DLPA -> low
present = [L for L in labels_sorted if L in results]
nrow, ncol = len(present), len(LIPIDS)
fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.0 * nrow), squeeze=False)
norm = TwoSlopeNorm(vmin=VMIN, vcenter=1.0, vmax=VMAX)
im = None
for r, L in enumerate(present):
    for c, lip in enumerate(LIPIDS):
        ax = axes[r, c]
        im = ax.imshow(results[L][lip].T, origin="lower", cmap="RdBu_r",
                       norm=norm, interpolation="bilinear", aspect="equal")
        ax.set_xticks([]); ax.set_yticks([])
        if r == 0:
            ax.set_title(lip, fontsize=12, style="italic", fontweight="bold")
        if c == 0:
            ax.set_ylabel(f"{L}\n({SYSTEMS[L][2]}% DLPA)", fontsize=10)
fig.suptitle(f"DOPC/DLPA lateral enrichment ({LEAFLET} leaflet)  -  "
             r"P density / mean", y=1.005, fontsize=12)
fig.subplots_adjust(right=0.88)
cax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cb = fig.colorbar(im, cax=cax)
cb.set_label("enrichment (density / mean)")
plt.savefig("results/spatial_distribution_DOPC_DLPA_composition.png",
            dpi=600, bbox_inches="tight")
plt.show()

# %% [markdown]
# ## Notes
#
# * For a well-mixed FLUID bilayer the time-averaged map is close to uniform
#   (~1 everywhere) because lipids diffuse; strong red patches that persist =
#   real clustering/domains. If maps look flat, that is a physical result (good
#   mixing), and the cross-RDF quantifies any short-range preference.
# * `LEAFLET` switches between the two monolayers; run both to check symmetry.
# * `VMIN/VMAX` set the colour range around 1.0 (uniform). Tighten them
#   (e.g. 0.8-1.2) to make weak enrichment visible.

# %%  (Optional) representative snapshot - top view of P atoms coloured by lipid
def snapshot(u, lipids=LIPIDS, head=HEAD, leaflet=LEAFLET, frame=-1):
    u.trajectory[frame]
    allP = u.select_atoms(f"name {head} and resname " + " ".join(lipids))
    zmid = allP.positions[:, 2].mean()
    cols = {"DOPC": "#1f77b4", "DLPA": "#d62728"}
    plt.figure(figsize=(4.2, 4.0))
    for lip in lipids:
        p = u.select_atoms(f"name {head} and resname {lip}").positions
        m = _leaf_mask(p[:, 2], zmid, leaflet)
        plt.scatter(p[m, 0], p[m, 1], s=25, color=cols.get(lip, "k"),
                    edgecolor="k", linewidth=0.3, label=lip)
    plt.gca().set_aspect("equal"); plt.xlabel("x (Å)"); plt.ylabel("y (Å)")
    plt.legend(); plt.title(f"snapshot ({leaflet} leaflet)"); plt.tight_layout(); plt.show()

# example:
# u = mda.Universe(*SYSTEMS["65/35"][:2]); snapshot(u)

# %% [markdown]
# ## GROMACS route - `gmx densmap`
#
# `gmx densmap` writes the time-averaged 2D density of a group in the xy plane.
#
# ```bash
# TPR=Prod_65.tpr ; XTC=Prod_merged_center_65.xtc ; NDX=index.ndx
#
# # DOPC phosphate 2D map (choose the "DOPC_P" group, or make one with make_ndx):
# gmx densmap -s $TPR -f $XTC -n $NDX -bin 0.1 -od densmap_DOPC_65.dat
# # repeat with the DLPA P group -> densmap_DLPA_65.dat
# ```
#
# `-bin 0.1` = 0.1 nm grid. The .dat is a plain matrix you can load with
# `np.loadtxt` and imshow exactly like the maps above (divide by its mean for the
# enrichment view). `gmx densmap` averages over both leaflets and all z by
# default; for a single leaflet, restrict the input group to that leaflet's P
# atoms (leaflet assignment is dynamic, so the Python route above is easier).

# %%  (Optional) load a gmx densmap .dat and show it
# M = np.loadtxt("densmap_DOPC_65.dat")
# plt.imshow((M/M.mean()).T, origin="lower", cmap="RdBu_r",
#            norm=TwoSlopeNorm(0.5, 1.0, 1.5)); plt.colorbar(); plt.show()

In [ ]:
# %% [markdown]
# # Area Compressibility Modulus (K_A) - grouped bar chart
#
# Grouped bars of K_A (mN/m) per composition for each DOPC/secondary-lipid
# system, with error bars, and PURE DOPC as a dashed benchmark line with a
# shaded band.  Legend for the systems on top.
#
# VALUES ARE HARDCODED, NOT READ FROM EXCEL.
# "New Paper_Updated II.xlsx" (sheet "New Systems") still carries the DOPA row
# in its original order, in which the 80/20, 87/13 and 95/5 entries are rotated
# by one position.  Reading from it reproduces the error.  The values below are
# the corrected ones, each reconfirmed by rerunning K_A = kT<A>/Var(A) on the
# corresponding box_DOPA_*.xvg trajectory:
#
#        composition     spreadsheet      corrected
#        80/20             228.9            261.8
#        87/13             261.8            247.1
#        95/5              247.1            228.9
#
# DOPG, DLPG and DLPA were unaffected.

# %%  Setup
import numpy as np
import matplotlib.pyplot as plt

# %%  Data
#
# K_A from area fluctuations, K_A = k_B T <A> / Var(A), on the total box area
# at T = 298.15 K, over the full 400 ns production trajectory.
#
# KA_SEM   standard error over five consecutive 80 ns blocks  (what is plotted)
# KA_SD    block-to-block standard deviation = SEM * sqrt(5)
#
# Set ERRBAR = "sd" to show the block scatter instead of the standard error.
# The SEM is the uncertainty on the mean and is the correct bar for comparing
# systems; the SD describes how much a single 80 ns window varies and is the
# more conservative thing to show if a referee asks for spread.

COMPS = [65, 72, 80, 87, 95]

KA = {
    "DOPG": [236.7, 223.3, 217.3, 217.0, 260.0],
    "DOPA": [246.4, 250.5, 261.8, 247.1, 228.9],   # corrected
    "DLPG": [216.2, 209.3, 274.1, 246.0, 287.3],
    "DLPA": [224.9, 269.7, 221.0, 264.8, 262.0],
}
KA_SEM = {
    "DOPG": [35.9, 12.1, 14.9, 13.5, 20.5],
    "DOPA": [13.0, 38.5, 12.1, 17.2, 14.4],        # corrected, moved with K_A
    "DLPG": [23.6, 14.4, 16.5, 12.4, 30.3],
    "DLPA": [20.7, 12.8, 16.7, 16.0, 14.7],
}
KA_SD = {
    "DOPG": [80.2, 27.0, 33.4, 30.2, 45.8],
    "DOPA": [29.1, 86.1, 27.1, 38.5, 32.1],
    "DLPG": [52.9, 32.3, 36.8, 27.7, 67.7],
    "DLPA": [46.3, 28.6, 37.4, 35.8, 32.8],
}

DOPC_BENCH = (245.81, 9.85)      # pure DOPC, prior study
ERRBAR = "sem"                   # "sem" or "sd"

DATA = {lip: {c: (KA[lip][j], (KA_SEM if ERRBAR == "sem" else KA_SD)[lip][j])
              for j, c in enumerate(COMPS)}
        for lip in KA}

print("Pure DOPC benchmark:", DOPC_BENCH)
for k in DATA:
    print(k, {c: DATA[k][c] for c in sorted(DATA[k])})

# %%  Style
LIPID_ORDER = ["DOPG", "DOPA", "DLPG", "DLPA"]
COLORS = {
    "DOPG": "#E8000B",   # red
    "DOPA": "#1F4E79",   # dark blue
    "DLPG": "#A9D18E",   # light green
    "DLPA": "#FFD966",   # gold/yellow
}
YMIN = 170          # baseline; top is set automatically to the tallest error bar

# %%  Plot
comps = sorted({c for s in DATA.values() for c in s})   # 65,72,80,87,95
lipids = [l for l in LIPID_ORDER if l in DATA]
n = len(lipids)
x = np.arange(len(comps))
width = 0.8 / n

fig, ax = plt.subplots(figsize=(7.2, 5.4))

# pure-DOPC benchmark band + dashed mean (behind bars)
m, sd = DOPC_BENCH
ax.axhspan(m - sd, m + sd, color="#CFE8F7", alpha=0.7, zorder=0)   # band
ax.axhline(m, color="k", ls="--", lw=1.2, zorder=1)               # mean

for i, lip in enumerate(lipids):
    xs = x + (i - (n - 1) / 2) * width
    ys = [DATA[lip].get(c, (np.nan, 0))[0] for c in comps]
    es = [DATA[lip].get(c, (np.nan, 0))[1] for c in comps]
    ax.bar(xs, ys, width, yerr=es, capsize=3, label=lip,
           color=COLORS.get(lip), edgecolor="black", linewidth=0.8,
           error_kw=dict(ecolor="0.25", lw=1.0), zorder=3)

ax.set_xticks(x)
ax.set_xticklabels([f"{c}%" for c in comps])
# top of y-axis = top of the tallest error bar (value + error), + small margin
ytop = max(DATA[l][c][0] + DATA[l][c][1] for l in lipids for c in DATA[l])
ax.set_ylim(YMIN, ytop + 2)
ax.set_ylabel("Area Compressibility Modulus (" + r"$\mathit{mN/m}$)", fontsize=12)
ax.tick_params(direction="out")
ax.legend(ncol=n, loc="upper center", bbox_to_anchor=(0.5, 1.08),
          frameon=True, fontsize=10, handlelength=1.2, columnspacing=1.2)
plt.tight_layout()
plt.savefig("KA_barplot.png", dpi=600, bbox_inches="tight")
plt.savefig("KA_barplot.pdf", bbox_inches="tight")
plt.show()